# 세탁 유형 8종 분류기 — 원본 거래 파일에서 바로 (4모델 최종 후보)

## 쓰는 법 (3단계)

1. 바로 아래 **설정** 셀에 원본 파일 3개의 경로를 적습니다. 팀 서버에서는 이미 적혀 있습니다. 분류할 날은 `TARGET_DAYS` 에 적습니다(기본 `2022-10-06` · val 17일 전부는 `'val'`).
2. 메뉴에서 **Run All(모두 실행)** 을 누릅니다.
3. 끝나면 `OUT_DIR` 폴더에 날마다 **`분류결과_<날짜>.csv`** 가 생깁니다. 화면에는 요약, 점수, 단계별 시간이 나옵니다.

## 이것이 하는 일

- **2단계 분류기**입니다. 세탁 패턴 거래가 8유형 중 어느 것인지 고릅니다. 8유형은 FAN-OUT · FAN-IN · CYCLE · SCATTER-GATHER · GATHER-SCATTER · BIPARTITE · STACK · RANDOM 입니다.
- 거래가 세탁인지 아닌지는 가리지 않습니다. 그것은 1단계 탐지 모델의 일이고, 이 파일에는 없습니다.
- **분류할 행**
  - 기본(`TARGETS = 'patterns'`): 그날 `Patterns.txt` 에 적힌 거래 전부. 원래 val 결과와 비교할 수 있습니다.
  - `TARGETS` 에 edge_id 목록을 줄 수도 있습니다. edge_id 는 거래 파일에서 머리줄을 뺀 0부터의 행 번호입니다(파일 줄 번호 = edge_id + 2). 이때 값은 유효합니다. 다만 그래프 모델 결과가 함께 묶인 행에 따라 달라져 원래 결과와 그대로 비교할 수는 없습니다.
- **모델**: 네 모델을 섞은 최종 후보입니다.
  - 네 모델: LightGBM · SALT-GNN C · TransformerConv B · TGT B0. GNN 셋은 시드 3개를 평균합니다.
  - 섞기: 가중 4 · 10 · 4 · 12 로 로그 평균을 낸 뒤 BIP × e^0.375 · STACK × e^−0.375 로 보정합니다.
  - 학습된 가중치는 모두 이 파일 안에 들어 있습니다(숨긴 셀).
  - 팀이 채택하기 전의 후보입니다(2026-10-16 묶음 `모델묶음_유형8_최종후보_20261016`).
- **입력은 원본 3파일뿐입니다.** 노트북이 원본을 한 번 훑어 압축 원장(`SCRATCH/ledger`)을 짓습니다. 그 원장에서 날마다 네 모델의 입력을 원래 전처리와 같은 규칙으로 다시 만듭니다(val 17일에서 원래 캐시와 비트 단위로 같음 확인).

## 필요한 것

- **Python 3.11 이상**과 numpy · pandas · torch(2.0 이상 · 2.10 에서 시험). 없으면 무엇을 깔지 알리고 멈춥니다.
- pyarrow 가 있으면 첫 실행이 약 2배 빠릅니다. 없으면 pandas 로 읽는다고 알리고 그대로 돕니다.
- lightgbm · torch_geometric · DuckDB 는 **필요 없습니다**.
- **Linux**(팀 JupyterLab 서버)에서 돕니다. Windows 에서는 돌지 않습니다.
- **디스크**: `SCRATCH` 에 빈 공간이 **6 GB 이상** 있어야 합니다. 원장 3.9 GB 가 남고, 처음 지을 때만 1.7 GB 를 잠깐 더 씁니다. 모자라면 원장을 짓기 전에 멈춥니다.
  - 기본 `SCRATCH = '유형8_작업폴더'` 는 노트북 커널의 작업 폴더(보통 이 노트북이 있는 폴더) 아래에 생깁니다. 팀 서버에서는 `Final_preprocess/HI-Large_EDA/유형8_작업폴더` 입니다.
  - 다 쓴 뒤 폴더째 지워도 됩니다. 다음 실행 때 다시 짓습니다.
- **메모리**: 약 4.5 GB(프로세스 꼭대기 4.4 GiB · 그중 익명 메모리 3.1 GiB).
- **시간**(팀 서버 · CPU 2스레드 실측)
  - 첫 실행 원장 짓기: 약 5분 반(pyarrow · 실측 313~316초). pyarrow 가 없으면 약 12~13분(원장 단계만 따로 잰 값 746초).
  - 원장은 `TARGET_DAYS` 와 상관없이 val 끝날(2022-10-14)까지 짓습니다. 그래서 다음 실행부터는 val 어느 날이든 원장을 다시 씁니다. 원본 파일이나 숨긴 원장 코드가 바뀌었으면 까닭을 알리고 다시 짓습니다.
  - 그 뒤 하루당 입력 만들기: 약 19초(val 17일 실측 17.9~22.5초).
  - 모델 4개: 하루치 약 17초(2022-10-06 · 1,415행) · val 17일 약 5분(24,280행 · LightGBM 63초 · SALT C 26초 · TC B 17초 · TGT B0 186초).
  - GPU 가 있으면(`DEVICE = 'auto'`) TGT B0 를 원래처럼 fp16 으로 돌립니다. 묶음 aml8.py 와 같은 코드지만, 이 노트북의 GPU 경로는 시험하지 않았습니다.

## test 봉인

- `2022-10-15` 이후 날짜는 test 기간이라 **거부**합니다. 팀 결정 전입니다.
- 원본을 훑을 때 그 시각 이후 행은 시각 칸만 해석하고, 다른 칸은 해석 · 저장하지 않고 버립니다(개수만 셈). 다만 CSV 읽기 도구(pyarrow · pandas)는 줄 전체를 글자 칸으로 나눈 뒤 넘겨 줍니다.
- `Patterns.txt` 도 시각 칸만 먼저 봅니다. 봉인 뒤 줄만 있는 시도는 유형 글자도 읽지 않습니다.
- 원본이 같은지 보는 지문은 파일 크기 · 수정 시각 · 앞뒤 1 MB 의 sha256 입니다(원래 전처리 daily_pipeline.py 와 같은 방식). 파일 끝 1 MB(봉인 뒤 행일 수 있음)는 해시에만 들어가고 해석하지 않습니다.
- 거래 파일 크기가 원본과 다르면 원장을 짓기 전에, 행 수가 다르면 끝까지 훑은 뒤 멈춥니다. edge_id(원본 행 번호)가 결과에 쓰이기 때문입니다.

## 결과 csv 열

| 열 | 뜻 |
|---|---|
| `edge_id` · `date` · `시각` | 원본 거래 파일의 행 번호(머리줄을 뺀 0부터 · 파일 줄 번호 = edge_id + 2) · 날 · 거래 시각 |
| `보낸_은행` · `보낸_계좌` · `받은_은행` · `받은_계좌` | 거래 양 끝 계좌. 은행은 숫자로 바꾼 은행 번호라 원본 거래 파일의 앞자리 0 이 빠집니다(예: `0214872` → `214872` · 계좌표 `Bank ID` 와 같은 꼴 · daily_pipeline.py:175). 원본 줄을 찾을 때는 edge_id 를 쓰세요 |
| `예측` · `예측확률` | 최종 후보가 고른 유형과 그 확률 |
| `p_<유형>` × 8 | 유형별 최종 확률(합 1) |
| `LGB_예측` · `SALT_C_예측` · `TC_B_예측` · `TGT_B0_예측` | 모델 하나씩만 썼을 때 고른 유형 |
| `정답` · `맞음` | Patterns.txt 의 유형(패턴 행일 때만)과 최종 예측이 맞았는지 |

- 파일은 UTF-8(BOM 있음) csv 입니다. Excel 로 바로 열면 `810017E50` 같은 계좌 번호가 지수 숫자로 바뀝니다. Excel 에서는 **데이터 → 텍스트/CSV 가져오기**로 열고 계좌 · 은행 열을 '텍스트' 로 지정하세요. pandas 는 `pd.read_csv(파일, dtype=str)` 로 읽으세요.

In [ ]:
# ═══ 설정 — 여기만 고치세요 ═══
# 원본 파일 3개(받은 그대로 · 이 노트북은 읽기만 합니다)
TRANS_CSV    = '/workspace/IBM_AML_dataset/HI-Large_Trans.csv'      # 거래 17.05 GB · 179,702,229행 · 행 순서 그대로
ACCOUNTS_CSV = '/workspace/IBM_AML_dataset/HI-Large_accounts.csv'   # 계좌표 2,126,855행
PATTERNS_TXT = '/workspace/IBM_AML_dataset/HI-Large_Patterns.txt'   # 세탁 패턴 목록(분류할 행 · 정답)

# 분류할 날(여러 날 가능). val = '2022-09-28' ~ '2022-10-14'. 2022-10-15 이후는 test 봉인이라 거부합니다.
#   예: ['2022-10-06'] · 'val'(val 17일 전부 · 원래 점수와 비교) · '2022-09-28~2022-10-05'(양 끝 포함) · ['2022-09-28', '2022-10-14']
TARGET_DAYS = ['2022-10-06']

# 분류할 행: 'patterns' = 그날 Patterns.txt 에 나온 거래 전부(기본 · 원래 결과와 비교 가능)
#            또는 edge_id 목록, 예: [5467000, 5467001]  ·  또는 edge_id 열이 있는 csv 경로
#            edge_id = 거래 파일에서 머리줄을 뺀 0부터의 행 번호(파일 줄 번호 − 2)
TARGETS = 'patterns'

# 원장(약 3.9 GB)을 두는 폴더. 상대 경로는 노트북 커널의 작업 폴더(보통 이 노트북이 있는 폴더) 기준입니다
#   → 팀 서버에서는 Final_preprocess/HI-Large_EDA/유형8_작업폴더. 다음 실행 때 그대로 다시 씁니다. 다 쓴 뒤 폴더째 지워도 됩니다(다시 지음).
SCRATCH = '유형8_작업폴더'
OUT_DIR = '.'               # 결과 분류결과_<날짜>.csv 를 쓸 폴더(상대 경로 기준은 SCRATCH 와 같음)
DEVICE  = 'auto'            # 'auto'(GPU 가 있으면 GPU) · 'cpu' · 'cuda'
THREADS = 2                 # CPU 스레드 수

In [ ]:
# @title 상수 — 숨김 · 고치지 마세요
# 스레드 수 · 장치는 numpy · torch 를 부르기 전에 정해야 해서 맨 먼저 정함(aml8.py:34-48 _early_option 과 같은 뜻).
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_v] = str(int(THREADS))
if str(DEVICE).lower() == 'cpu':
    os.environ['CUDA_VISIBLE_DEVICES'] = ''          # cpu 를 고르면 GPU 를 아예 보지 않음(aml8.py:47-48)

# 유형 순서: tree9_lib.py:57 CLASS_NAMES 앞 8개 = aml8.py:57 = aml8_모델.pt class_names
CLASS_NAMES8 = ['FAN-OUT', 'FAN-IN', 'CYCLE', 'SCATTER-GATHER', 'GATHER-SCATTER', 'BIPARTITE', 'STACK', 'RANDOM']
# LightGBM 55열 이름 · 순서(aml8_모델.pt features_55 = val_pattern55.npz features): 보낸 계좌 19 · 받은 계좌 19 · 거래 17
FEATURES_55 = [
    'src_log_out_count', 'src_log_in_count', 'src_log_out_neighbors', 'src_log_in_neighbors',
    'src_log_self_count', 'src_log_reciprocal_neighbors', 'src_mean_paid_currency_z', 'src_paid_outlier_fraction',
    'src_log_currency_count', 'src_log_active_days', 'src_type_Corporation', 'src_type_Partnership',
    'src_type_Sole_Proprietorship', 'src_type_Individual', 'src_type_Country', 'src_type_Direct',
    'src_log_entity_observed_accounts', 'src_log_entity_out_diversity', 'src_log_entity_in_diversity', 'dst_log_out_count',
    'dst_log_in_count', 'dst_log_out_neighbors', 'dst_log_in_neighbors', 'dst_log_self_count',
    'dst_log_reciprocal_neighbors', 'dst_mean_paid_currency_z', 'dst_paid_outlier_fraction', 'dst_log_currency_count',
    'dst_log_active_days', 'dst_type_Corporation', 'dst_type_Partnership', 'dst_type_Sole_Proprietorship',
    'dst_type_Individual', 'dst_type_Country', 'dst_type_Direct', 'dst_log_entity_observed_accounts',
    'dst_log_entity_out_diversity', 'dst_log_entity_in_diversity', 'pair_prior_count', 'pair_gap_minutes',
    'pair_count_1h', 'sender_gap_minutes', 'sender_count_1h', 'receiver_gap_minutes',
    'receiver_count_1h', 'paid_currency_z', 'pair_history_count', 'reverse_history_count',
    'pair_age_at_day_start', 'same_entity', 'same_known_country', 'both_countries_known',
    'same_currency', 'is_self_transfer', 'amt_z_pay_ccy']

# 섞기(final_candidate.py:24-41 = aml8.py:1200-1215): 로그 평균 가중 4 · 10 · 4 · 12(합 30) → BIP × e^(δ/2) · STACK × e^(−δ/2)
SEEDS = (42, 43, 44)                                   # GNN 3개는 시드 3개 평균 · LightGBM 은 시드 42 하나
COMP = {'LGB기준': (['lgb_s42'], 4), 'SALT C': ([f'salt_c_s{s}' for s in SEEDS], 10),
        'TC B': ([f'tc_b_s{s}' for s in SEEDS], 4), 'TGT B0': ([f'tgt_b0_s{s}' for s in SEEDS], 12)}
DELTA = 0.75
MODEL_KEYS = ('lgb', 'salt_c', 'tc_b', 'tgt_b0')
COMP_NAME = {'lgb': 'LGB기준', 'salt_c': 'SALT C', 'tc_b': 'TC B', 'tgt_b0': 'TGT B0'}          # aml8.py:62
MODEL_LABEL = {'lgb': 'LightGBM', 'salt_c': 'SALT-GNN C', 'tc_b': 'TransformerConv B', 'tgt_b0': 'TGT B0'}
COL_PREFIX = {'lgb': 'LGB', 'salt_c': 'SALT_C', 'tc_b': 'TC_B', 'tgt_b0': 'TGT_B0'}

# 날짜
TEST_START = '2022-10-15'                              # test 봉인 시작(tree9_lib.py:54, 70-80)
TRAIN_FIRST, TRAIN_LAST = '2022-08-08', '2022-09-27'   # 모델이 배운 51일(tree9_lib.py:54). 이 날을 넣으면 점수가 부풀려짐
VAL_FIRST, VAL_LAST = '2022-09-28', '2022-10-14'       # val 17일(common.py:39)
BACK9_FIRST = '2022-10-06'                             # 뒤 9일 = val day 번호 ≥ 8(lib8.py FIT_DAYS = 8 · aml8.py:1326-1331)

# 원본 거래 파일 확인값(받은 원본 HI-Large_Trans.csv 를 잰 값: 크기 = stat · 행 = 머리줄 뺀 줄 수). 다르면 edge_id(원본 행 번호)가 달라져 멈춤.
SOURCE_ROWS, SOURCE_BYTES = 179_702_229, 17_052_760_651

# 원래 결과(확인용_기대값.npz final_back_macro_precision · recall = README.md:149 의 0.5089 · 0.5026, 반올림 전)
REF_BACK9 = dict(precision=0.5089, recall=0.5026)

# 통화 상수 ①(paid_currency_z: scale · med · iqr)와 ②(amt_z: μ · σ)는 아래 '원장' 코드 셀 안 LG.CURRENCY_STATS · LG.AMTZ_STATS 에 있음
# (daily_pipeline.py:279-283 · build_nb_A.py:116-126 값). 원장을 지을 때 보정 기간(08-01~08-07) 행으로 다시 계산해 차 0 인지 meta 에 적음.

# 아래 코드 셀들이 채우는 자리
_MODULE_SRC = {}                                       # 모듈 이름 → 원본 코드 글자(숨긴 셀 ①~⑦)
# 만들 때 잰 sha256. 숨긴 셀이 바뀌었으면 멈춤(일부러 코드를 고쳤으면 설정 셀에 ALLOW_MODIFIED_CODE = True — 결과가 원래와 달라질 수 있음).
_MODULE_SHA = {
    'ledger': 'c84457dd12d1629df4c66731d2a4c7cd2af57bb5d892e009fcb92c935c39efaf',
    'x17': '1f6817cbfcf4063cc54a61cde33639ee731e1fb4981c5f49a068a943ff40249e',
    'node19': '971d5bd21ee69b0b7cf2ccc5e786610f34d61a9b3dba2e8ef0b340c2246cb646',
    'salt_inputs': '25e68322828b11bbfa405286f4f91fb3925ee56506861b1ea749aaa045f2d09d',
    'tgt_inputs': 'ecaab9c7e3b485fe2df6da14cebfce631c536f11ffba309a607f9b3da84c8516',
    'salt_models': '213e6b455d1b7c4109bcec24f894eefd0e07aee7a90a87c7786e48414a99899f',
    'tgt_model': '7e5f9d9bc6d3105120d85885b948bd65abac4b8defc65f43ad52a4607b5af8fc'}
# 숨긴 자료(LightGBM 나무 배열 · GNN/TGT 가중치)를 zlib 으로 푼 바이트의 sha256. 다르면 멈춤.
_PAYLOAD_SHA = {'LGB': '2d5a8b79847ede0cce152245341eb1506ee092ede3cbdd38531e96e6bd955350', 'NN': 'e4ce4e208b20cad02a9a28ad4dc30af09023a1f633ff0513ef700762e3b56773'}

In [ ]:
# @title 학습된 LightGBM 나무 6,880개(860 round × 8유형) — 숨김 · 고치지 마세요
# aml8_모델.pt models.lgb(L31D6_m20_r0_lam0_s42 · LightGBM 4.7.0) → 배열 npz(feat · thr · left · right · leaf · nl) → zlib → base64. 풀 때 sha256 을 상수 셀 _PAYLOAD_SHA["LGB"] 와 비교
LGB_B64 = [
'...',
]

In [ ]:
# @title 학습된 GNN · TGT 가중치 9개(SALT C · TC B · TGT B0 × 시드 42 · 43 · 44) + SALT z-표준화 · δ + TGT 설정 — 숨김 · 고치지 마세요
# aml8_모델.pt(sha256 e08c49e3cf508303)에서 LightGBM 문자열만 뺀 dict → torch.save → zlib → base64. 풀 때 sha256 을 _PAYLOAD_SHA["NN"] 와 비교한 뒤 torch.load(weights_only=True) 로 읽음
NN_B64 = [
'...',
]

In [ ]:
# @title 코드 ① 원장(ledger.py): 원본 3파일 → 압축 원장 · 중복 · 계좌 번호 · 통화 상수 · 패턴 행 · test 봉인 — 숨김 · 고치지 마세요
_MODULE_SRC['ledger'] = r'''"""원장(ledger) 단계 — HI-Large 원본 3파일을 한 번 훑어 4모델 입력에 필요한 원본 열만 압축해 디스크에 둔다.

입력: HI-Large_Trans.csv(시간순 아님, edge_id = 0부터 세는 원본 행 번호) · HI-Large_accounts.csv · HI-Large_Patterns.txt.
출력: out_dir/ 아래 .npy(memmap 으로 읽음) + meta.json. 형식은 LEDGER_FORMAT.md.

원래 코드(읽기 전용)와의 대응 — 각 규칙 옆에 file:line 을 적음(경로는 팀 작업 공간의 Final_preprocess/ 부터).
  DP  = Final_preprocess/태훈님_전처리/feature_code_40_20260921/CODE/daily_pipeline.py
  T9  = Final_preprocess/HI-Large_EDA/tree9_lib.py
  NBA = Final_preprocess/HI-Large_EDA/tgnn36_20260928/_builders/build_nb_A.py
  TGD = Final_preprocess/HI-Large_EDA/Transformer모델/기법실행_20261008/code/tgt_data.py

지키는 규칙
  - 원본은 읽기만 함.
  - test 봉인: d_max ≥ 2022-10-15 는 환경 변수 AML_TEST_UNSEAL=1 없이는 거부(T9:65-87 과 같은 뜻).
  - CSV 를 훑을 때 Timestamp 열만 먼저 해석하고, 자르는 시각(d_max 다음 날 00:00) 이후 행은 다른 열을 해석 · 해시 · 저장하지 않고
    버림(개수만 셈). 단 CSV 읽기 도구(pyarrow · pandas)는 줄 전체를 11개 글자 칸으로 나눈 뒤 넘겨 줌(pyarrow 는 UTF-8 확인도 함).
    Patterns.txt 도 같은 방식: 시각 칸만 먼저 보고 버림. 그 시도의 BEGIN 줄 유형 글자도 자르는 시각 전 줄이 나올 때만 해석함.
  - 원본 확인 지문(fingerprint)은 DP:73-78 처럼 크기 · 수정 시각 · 앞뒤 1 MB 의 sha256. 파일 끝 1 MB(봉인 뒤 행일 수 있음)는
    해시에만 넣고 해석하지 않음.
  - 기존 원장은 원본 지문 · d_max · 형식 · 이 코드의 sha256 이 모두 맞을 때만 다시 씀(DP:97-103 require_stage 와 같은 뜻).
  - 쓰는 라이브러리: numpy · pandas(C 엔진) · (있으면) pyarrow 빠른 길. DuckDB · lightgbm · torch_geometric 없음.
"""
from __future__ import annotations

import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '2')

import csv
import datetime as _dt
import hashlib
import json
import re
import resource
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:                                    # 빠른 길(선택). 없으면 pandas C 엔진만 씀.
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.csv as pacsv
    HAVE_ARROW = True
except Exception:                       # pragma: no cover
    pa = pc = pacsv = None
    HAVE_ARROW = False

FORMAT_VERSION = 1
ARROW_BLOCK = 16 << 20                 # pyarrow 읽기 블록(16 MB — 64 MB 는 RSS 4 GB 까지 올라감, scan_raw 실측). 시험에서만 줄임.

# ───────────────────────── 고정 상수(원래 코드에서 옮김) ─────────────────────────
# 원본 열 이름은 위치로 붙임 — 헤더에 'Account' 가 두 번 나옴(DP:20-24, 133-144).
RAW_COLUMNS = ['timestamp_raw', 'from_bank', 'from_account', 'to_bank', 'to_account',
               'amount_received_raw', 'receiving_currency', 'amount_paid_raw',
               'payment_currency', 'payment_format', 'label_raw']
HEADERS = ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account', 'Amount Received',
           'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']
ACCOUNT_HEADERS = ['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']      # DP:25
ACCOUNT_COLUMNS = ['bank_name', 'bank_id', 'account_number', 'entity_id', 'entity_name']      # DP:26
ENTITY_TYPES = ['Corporation', 'Partnership', 'Sole Proprietorship', 'Individual', 'Country', 'Direct']  # DP:27 (원핫 순서, DP:387)
CLASS_NAMES8 = ['FAN-OUT', 'FAN-IN', 'CYCLE', 'SCATTER-GATHER', 'GATHER-SCATTER', 'BIPARTITE', 'STACK', 'RANDOM']  # T9:57 앞 8개

BASE_DATE = '2022-08-01'               # 분 단위 시각의 0점(TGD:33 BASE, make_p1.py ts_min)
CAL_START, CAL_END = '2022-08-01', '2022-08-08'   # 통화 보정 구간 [08-01, 08-08) (DP:45-46, 279-281; amtz_meta.json)
TRAIN_FIRST, TRAIN_LAST = '2022-08-08', '2022-09-27'  # 'seen' 판정용 train 창, 양 끝 포함(T9:54)
TEST_START = '2022-10-15'              # test 봉인 시작(T9:54, 70)
PATTERN_BEGIN_RE = re.compile(r'BEGIN LAUNDERING ATTEMPT - ([^:]+)(?::.*)?')  # DP:216
ENTITY_TYPE_RE = r'^([A-Za-z ]+?)\s*#'                                       # DP:195
COUNTRY_RE = re.compile(r'^(.*?) Bank #')                                    # DP:116

# 원본 파일 기대값(확인용) — 받은 원본 HI-Large_Trans.csv 를 잰 값(크기 = stat · 행 = 머리줄 뺀 줄 수).
EXPECTED = {'HI-Large_Trans.csv': dict(bytes=17_052_760_651, rows=179_702_229)}

# 통화 상수 ① paid_currency_z: 통화 → (scale, med, iqr). DP:279-283 이 [08-01, 08-08) 로 맞춘 값.
# 출처: Final_preprocess/태훈님_전처리/feature_code_40_20260921/currency_stats.parquet (차 0). 원장을 지을 때 다시 계산해 meta 에 차를 적음.
CURRENCY_STATS = {
    'Australian Dollar': (1338.9, 0.6931471805599453, 1.8091916721490713),
    'Bitcoin': (0.074037, 0.6931471805599453, 1.879415750992096),
    'Brazil Real': (5420.13, 0.6931471805599453, 1.8266080556985667),
    'Canadian Dollar': (1209.17, 0.6931471805599453, 1.8014374822437402),
    'Euro': (801.57, 0.6931471805599453, 1.8117647968792057),
    'Mexican Peso': (20934.25, 0.6931471805599453, 1.8355007307128108),
    'Ruble': (75154.88, 0.6931471805572342, 1.8833509462315643),
    'Rupee': (70654.61, 0.6931471805599453, 1.876669789903581),
    'Saudi Riyal': (3633.22, 0.6931471805599453, 1.8113942672126997),
    'Shekel': (3164.95, 0.6931471805599453, 1.8283007986757869),
    'Swiss Franc': (858.65, 0.6931471805599453, 1.8021027989727405),
    'UK Pound': (725.575, 0.6931471805540095, 1.8162719595710928),
    'US Dollar': (934.29, 0.6931471805599453, 1.8175503807996234),
    'Yen': (98868.11, 0.6931471805599453, 1.8951548063412347),
    'Yuan': (6399.91, 0.6931471805599453, 1.8232475273038067),
}
# 통화 상수 ② amt_z_pay_ccy: 통화 → (mu, sigma ddof 0) of ln(1+paid), 보정 구간 송금액만, edge_id 순서 numpy float64(NBA:116-126).
# 출처: tgnn36_20260928/results/A1_amtz_통화별_보정통계.csv (float_precision='round_trip' 로 읽은 값).
AMTZ_STATS = {
    'Australian Dollar': (7.331291137574385, 2.9032033210182355),
    'Bitcoin': (0.523493571584354, 1.1157704126150796),
    'Brazil Real': (8.700227164981321, 2.9445194197732194),
    'Canadian Dollar': (7.237778502239755, 2.885985207174699),
    'Euro': (6.833833647581592, 2.883955024575965),
    'Mexican Peso': (10.001216574727529, 3.028092639337961),
    'Ruble': (11.273959157389443, 3.1261861683128522),
    'Rupee': (11.19894831648783, 3.154481926818126),
    'Saudi Riyal': (8.300421928971414, 2.9336544559327753),
    'Shekel': (8.182167217421698, 2.941717307258179),
    'Swiss Franc': (6.912347131202526, 2.8747617066578792),
    'UK Pound': (6.745107679449901, 2.878223503635805),
    'US Dollar': (6.996561734516492, 2.9064962480388057),
    'Yen': (11.526063352721104, 3.184398426642972),
    'Yuan': (8.82494771047599, 2.9816647970695365),
}
CURRENCIES = sorted(CURRENCY_STATS)    # 통화 코드 0..14(이름 정렬 순). 처음 보는 통화는 15, 16, … 로 뒤에 붙고 pz = az = 0.
PAYMENT_FORMATS = ['ACH', 'Bitcoin', 'Cash', 'Cheque', 'Credit Card', 'Reinvestment', 'Wire']  # 결제 방식 코드 0..6
NULL_TOKENS = {'null', 'none', 'nan', 'na', 'n/a', '?'}   # DP:150 의 결측 표기

# 압축 원장 행 파일: (이름, dtype). 행 = 남긴 거래, 원본 행 순서(= edge_id 오름차순).
ROW_FIELDS = [('eid', '<i4'), ('ts', '<i4'), ('src', '<i4'), ('dst', '<i4'),
              ('pz', '<f4'), ('az', '<f4'), ('pcur', '|i1'), ('rcur', '|i1'), ('fmt', '|i1')]


# ───────────────────────── 작은 도구 ─────────────────────────
def _days_from_civil(y, m, d):
    """그레고리력 (y, m, d) → 1970-01-01 기준 일수(정수 배열). H. Hinnant days_from_civil."""
    y = y - (m <= 2)
    era = np.floor_divide(y, 400)
    yoe = y - era * 400
    mp = (m + 9) % 12
    doy = (153 * mp + 2) // 5 + d - 1
    doe = yoe * 365 + yoe // 4 - yoe // 100 + doy
    return era * 146097 + doe - 719468


_BASE_DAYS = int(_days_from_civil(np.int64(2022), np.int64(8), np.int64(1)))
assert _BASE_DAYS == (np.datetime64(BASE_DATE) - np.datetime64('1970-01-01')).astype(int)


def minutes_of(date_str: str) -> int:
    """'YYYY-MM-DD' 의 00:00 → 2022-08-01 00:00 기준 분."""
    return int((np.datetime64(date_str) - np.datetime64(BASE_DATE)).astype('timedelta64[m]').astype(np.int64))


def _ts_minutes_py(s: str) -> int:
    """시각 문자열 하나 → 분. DuckDB strptime('%Y/%m/%d %H:%M')(DP:201)과 같은 형식. 실패하면 ValueError."""
    t = _dt.datetime.strptime(s, '%Y/%m/%d %H:%M')
    return int((t - _dt.datetime(2022, 8, 1)).total_seconds() // 60)


def _h64(s: str) -> int:
    """문자열 → 안정적인 64비트 해시(실행마다 같음)."""
    return int.from_bytes(hashlib.blake2b(s.encode('utf-8'), digest_size=8).digest(), 'little')


_M1 = np.uint64(0xBF58476D1CE4E5B9)
_M2 = np.uint64(0x94D049BB133111EB)
_GOLD = np.uint64(0x9E3779B97F4A7C15)


def _smix(z):
    z = z ^ (z >> np.uint64(30)); z = z * _M1
    z = z ^ (z >> np.uint64(27)); z = z * _M2
    return z ^ (z >> np.uint64(31))


def _row_hash(parts):
    """uint64 배열 여러 개 → 행 해시. 같은 11열 문자열이면 반드시 같은 값(역은 문자열로 다시 확인)."""
    with np.errstate(over='ignore'):
        h = np.full(len(parts[0]), np.uint64(0x243F6A8885A308D3))
        for p in parts:
            h = _smix(h ^ (p.astype(np.uint64, copy=False) + _GOLD))
    return h


def fingerprint(path) -> dict:
    """DP:73-78 file_fingerprint 와 같은 값(크기 · mtime · 앞뒤 1 MB sha256)."""
    p = Path(path); s = p.stat()
    with p.open('rb') as f:
        head = f.read(1 << 20); f.seek(max(0, s.st_size - (1 << 20))); tail = f.read()
    return {'path': str(p.resolve()), 'bytes': s.st_size, 'mtime_ns': s.st_mtime_ns,
            'head_tail_sha256': hashlib.sha256(head + tail).hexdigest()}


def _same_source(a: dict, b: dict) -> bool:
    """원본이 같은지: 크기 · 수정 시각 · 앞뒤 1 MB sha256 이 모두 같아야 함(DP:100 은 지문 dict 전체를 == 로 비교).
    경로는 비교하지 않음(링크 · 마운트 위치만 다른 같은 파일은 같은 원본으로 봄)."""
    return (a['bytes'] == b['bytes'] and a.get('mtime_ns') == b.get('mtime_ns')
            and a['head_tail_sha256'] == b['head_tail_sha256'])


def source_fingerprints(trans_csv, accounts_csv, patterns_txt) -> dict:
    return {'transactions': fingerprint(trans_csv), 'accounts': fingerprint(accounts_csv), 'patterns': fingerprint(patterns_txt)}


def code_sha256() -> str:
    """이 파일(ledger.py) 글자의 sha256. 원장 meta 에 적고, 다시 쓸 때 비교함(DP:101 require_stage 의 code_sha256 과 같은 뜻)."""
    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()


# 원장 폴더 표시 파일 · 이 코드가 쓰는 파일 이름(지우기 전에 '이 노트북이 지은 원장 폴더' 인지 확인하는 데 씀)
LEDGER_MARK = '.ledger_by_aml8_notebook'
_OWN_NAMES = ({f'{n}.npy' for n, _ in ROW_FIELDS}
              | {f'pat_{k}.npy' for k in ('eid', 'day', 'ts', 'y', 'attempt', 'seen', 'row')}
              | {f'acct_{k}.npy' for k in ('bank', 'num', 'ent', 'etype', 'country', 'active')}
              | {'ent_id.npy', 'dup_pairs.npy', 'meta.json', '_dedup_buckets', LEDGER_MARK})
BUILD_NEED_BYTES = 6_000_000_000       # 원장 짓기에 필요한 빈 공간: 원장 3.84 GB + 짓는 동안 중복 확인 통 1.68 GB(실측 · d_max 10-14) + 여유


def _is_own_dir(p: Path) -> bool:
    """지워도 되는 '이 코드가 지은 원장 폴더' 인지: 비었거나 · 표시 파일이 있거나 · 이 코드가 쓰는 이름만 들어 있음."""
    if p.is_symlink() or not p.is_dir():
        return False
    names = {c.name for c in p.iterdir()}
    return (not names) or (LEDGER_MARK in names) or names <= _OWN_NAMES


def _dir_bytes(p: Path) -> int:
    if not p.exists() or p.is_symlink():
        return 0
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file() and not f.is_symlink())


def _remove_own_dir(p: Path, what: str, log):
    """원장 폴더 지우기 — 이 코드가 지은 폴더일 때만(아니면 멈춤 · 아무것도 지우지 않음)."""
    if p.is_symlink():
        raise RuntimeError(f'{p} 는 링크입니다. 링크가 가리키는 자료를 지우지 않으려고 멈춥니다. '
                           'SCRATCH 를 다른 폴더로 바꾸거나 링크를 직접 정리해 주세요.')
    if not p.exists():
        return
    if not _is_own_dir(p):
        raise RuntimeError(f'{p} 에 이 노트북이 짓지 않은 파일이 있어 지우지 않고 멈춥니다({what}). '
                           'SCRATCH 를 비어 있는 새 폴더로 바꿔 주세요.')
    log(f'[ledger] {what} 지움: {p}')
    shutil.rmtree(p)


def reuse_status(out_dir, fps: dict, d_max: str):
    """(다시 쓸 Ledger 또는 None, 까닭). 원본 지문 · d_max · 형식 · 코드 sha256 이 모두 맞아야 다시 씀."""
    out_dir = Path(out_dir)
    if not (out_dir / 'meta.json').exists():
        return None, '원장이 아직 없음'
    try:
        L = load_ledger(out_dir)
    except Exception as e:                           # 깨졌거나 다른 형식
        return None, f'있는 원장을 읽지 못함({e})'
    m = L.meta
    for k in fps:
        if k not in m.get('sources', {}) or not _same_source(m['sources'][k], fps[k]):
            return None, f'원본 {k} 파일이 원장을 지을 때와 다름(크기 · 수정 시각 · 앞뒤 1 MB 해시 중 하나)'
    if m.get('format_version') != FORMAT_VERSION:
        return None, f'원장 형식 {m.get("format_version")} ≠ {FORMAT_VERSION}'
    if m.get('code_sha256') != code_sha256():
        return None, '원장 코드(ledger.py)가 원장을 지을 때와 다름'
    if m['d_max'] < d_max:
        return None, f'있는 원장은 {m["d_max"]} 까지라 {d_max} 까지 새로 지어야 함'
    return L, f'원본 · 코드가 같고 {m["d_max"]} 까지 있음'


def cgroup_anon_gib() -> float:
    try:
        for l in open('/sys/fs/cgroup/memory.stat'):
            if l.startswith('anon '):
                return int(l.split()[1]) / 2 ** 30
    except OSError:
        pass
    return float('nan')


def wait_for_memory(limit_gib: float = 8.0, poll: float = 30.0, log=print):
    """컨테이너 anon 이 limit_gib 아래로 내려올 때까지 기다림(다른 작업과 함께 돌기 때문)."""
    while True:
        a = cgroup_anon_gib()
        if not (a >= limit_gib):
            return a
        log(f'[wait] cgroup anon {a:.2f} GiB ≥ {limit_gib} GiB — {poll:.0f}초 기다림')
        time.sleep(poll)


def _rss_gib() -> float:
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20


def check_seal(date_str: str, what: str):
    """test 봉인(T9:74-80 과 같은 뜻): 2022-10-15 이후 날짜는 AML_TEST_UNSEAL=1 이 있어야 함."""
    if str(date_str)[:10] >= TEST_START and os.environ.get('AML_TEST_UNSEAL') != '1':
        raise PermissionError(f'[test 봉인] {what}({str(date_str)[:10]}): {TEST_START} 이후 자료는 봉인되어 있음. '
                              '팀 결정 뒤 AML_TEST_UNSEAL=1 로만 엶.')


class _NpyAppender:
    """행 수를 모르는 채로 .npy 에 이어 씀. 앞 128 B 를 비워 두고 끝에 실제 shape 로 헤더를 씀."""
    HEADER = 128

    def __init__(self, path, dtype, tail=()):
        self.path = Path(path); self.dtype = np.dtype(dtype); self.tail = tuple(tail); self.n = 0
        self.f = open(self.path, 'wb'); self.f.write(b'\0' * self.HEADER)

    def append(self, a):
        a = np.ascontiguousarray(a, dtype=self.dtype)
        assert a.shape[1:] == self.tail, (self.path, a.shape)
        if a.size:
            self.f.write(memoryview(a).cast('B'))
        self.n += a.shape[0]

    def close(self):
        d = {'descr': np.lib.format.dtype_to_descr(self.dtype), 'fortran_order': False, 'shape': (self.n,) + self.tail}
        h = repr(d).encode('latin1')
        pre = b'\x93NUMPY\x01\x00'
        pad = self.HEADER - len(pre) - 2 - len(h) - 1
        assert pad >= 0, 'npy header too long'
        self.f.flush(); self.f.seek(0)
        self.f.write(pre + np.uint16(len(h) + pad + 1).tobytes() + h + b' ' * pad + b'\n')
        self.f.close()


# ───────────────────────── 열 접근(두 엔진 공통) ─────────────────────────
# 묶음(batch) = {열 이름: 열}. 열은 numpy object 배열(pandas C 엔진) 또는 pyarrow StringArray(빠른 길).
# 아래 함수는 두 꼴 모두 같은 값을 냄 — 두 엔진의 원장이 비트 단위로 같아야 함(시험으로 확인).
def _is_arrow(col) -> bool:
    return HAVE_ARROW and isinstance(col, (pa.Array, pa.ChunkedArray))


def _arrow_flat(col):
    return col.combine_chunks() if isinstance(col, pa.ChunkedArray) else col


def _len(col) -> int:
    return len(col)


def _filter(col, mask):
    if _is_arrow(col):
        return _arrow_flat(col).filter(pa.array(mask))
    return col[mask]


def _str_buffers(col):
    """문자열 열 → (utf-8 바이트 uint8 배열, 시작 위치 int64[n+1])."""
    if _is_arrow(col):
        a = _arrow_flat(col)
        if a.null_count:
            raise ValueError('빈 값(null)이 있음')
        bufs = a.buffers()
        wide = pa.types.is_large_string(a.type)
        off = np.frombuffer(bufs[1], dtype=np.int64 if wide else np.int32, count=len(a) + 1,
                            offset=a.offset * (8 if wide else 4)).astype(np.int64)
        data = np.frombuffer(bufs[2], np.uint8) if bufs[2] is not None else np.zeros(0, np.uint8)
        data = data[off[0]:off[-1]]
        return data, off - off[0]
    n = len(col)
    s = ''.join(col)                                   # 문자열이 아닌 값(NaN 등)이면 TypeError
    b = s.encode('utf-8')
    lens = np.fromiter(map(len, col), np.int64, n)
    if len(b) != int(lens.sum()):                      # ASCII 아닌 글자 — 바이트 길이로 다시 셈(드묾)
        enc = [x.encode('utf-8') for x in col]
        lens = np.fromiter(map(len, enc), np.int64, n); b = b''.join(enc)
    off = np.zeros(n + 1, np.int64); np.cumsum(lens, out=off[1:])
    return np.frombuffer(b, np.uint8), off


def _fixed_rows(data, off, w):
    """길이가 정확히 w 인 행 마스크와 그 행들의 (k, w) 바이트."""
    lens = np.diff(off)
    m = lens == w
    if m.all():
        return m, data.reshape(-1, w) if len(data) else np.zeros((0, w), np.uint8)
    st = off[:-1][m]
    return m, data[st[:, None] + np.arange(w)]


def _strings(col, idx):
    """idx 행의 문자열 목록."""
    idx = np.asarray(idx, np.int64)
    if _is_arrow(col):
        return _arrow_flat(col).take(pa.array(idx)).to_pylist()
    return [col[i] for i in idx]


def _rows_strings(cols, idx):
    """idx 행의 11열 문자열 튜플 목록(중복 · 패턴 대조용)."""
    per = [_strings(cols[c], idx) for c in RAW_COLUMNS]
    return list(zip(*per))


def _factorize(col):
    """→ (코드 int64[n], 고유 문자열 목록)."""
    if _is_arrow(col):
        d = _arrow_flat(col).dictionary_encode()
        return d.indices.to_numpy().astype(np.int64), d.dictionary.to_pylist()
    codes, uniq = pd.factorize(col)
    return codes.astype(np.int64, copy=False), list(uniq)


def _to_float(col):
    """금액 문자열 → float64, 올바른 반올림. Python float() = pyarrow cast = DuckDB DECIMAL→DOUBLE(1.5.5 에서 20만 개 확인, DP:198, 202)."""
    if _is_arrow(col):
        return pc.cast(_arrow_flat(col), pa.float64()).to_numpy()
    return col.astype(np.float64)                      # object → 원소마다 Python float()


_HEX_LUT = np.full(256, 255, np.uint8)
for _i, _ch in enumerate('0123456789ABCDEF'):
    _HEX_LUT[ord(_ch)] = _i
_HEX_POW = (np.uint64(16) ** np.arange(8, -1, -1, dtype=np.uint64)).astype(np.uint64)


# ───────────────────────── 계좌 표(accounts.csv) ─────────────────────────
def _country_of(name) -> str:
    """DP:115-119 그대로."""
    m = COUNTRY_RE.match(str(name))
    value = m.group(1).strip() if m else 'UNKNOWN'
    if value.lower() in {'crypto', 'crytpo'}:
        return 'UNKNOWN'
    return value or 'UNKNOWN'


def _check_header(path, expected, what):
    """DP:128-130: 헤더 줄이 기대한 열 순서인지."""
    with open(path, newline='') as f:
        actual = next(csv.reader(f))
    if actual != expected:
        raise ValueError(f'{what}: unexpected column order/header: {actual}')


class _Accounts:
    """계좌 키 (str(int(bank)), account) → 계좌 번호 acct_id.

    acct_id = (은행 문자열, 계좌 문자열) 의 바이트 사전순 순위(DP:176 ORDER BY bank, account 와 같은 순서).
    원래 node_id 는 거래에 나온 계좌만 셌지만 순서가 같으므로, 정렬 · 동점 · 이웃 순서가 모두 원래와 같음.
    """

    def __init__(self, accounts_csv, log=print):
        t0 = time.time()
        _check_header(accounts_csv, ACCOUNT_HEADERS, 'accounts')
        df = pd.read_csv(accounts_csv, header=None, skiprows=1, names=ACCOUNT_COLUMNS, dtype=object,
                         na_filter=False, engine='c')
        n = len(df)
        cols = {c: df[c].to_numpy(dtype=object) for c in ACCOUNT_COLUMNS}
        for c in ACCOUNT_COLUMNS:                      # DP:182-184 검사: 빈 값 · 앞뒤 공백 · 은행 숫자 · 계좌 16진
            data, off = _str_buffers(cols[c])
            lens = np.diff(off)
            if (lens == 0).any():
                raise ValueError(f'Invalid account registry: empty {c}')
            ws = np.zeros(256, bool); ws[[9, 10, 11, 12, 13, 32]] = True
            if ws[data[off[:-1]]].any() or ws[data[off[1:] - 1]].any():
                raise ValueError(f'Invalid account registry: untrimmed {c}')
            if c == 'bank_id' and not ((data >= 48) & (data <= 57)).all():
                raise ValueError('Invalid account registry: bank_id not digits')
            if c == 'account_number':
                hexok = np.zeros(256, bool); hexok[[ord(x) for x in '0123456789ABCDEFabcdef']] = True
                if not hexok[data].all():
                    raise ValueError('Invalid account registry: account_number not hex')
        bank_int = np.array([int(x) for x in cols['bank_id']], dtype=object)
        bank_norm = np.array([str(v) for v in bank_int], dtype=object)               # DP:175 cast(cast(x AS BIGINT) AS VARCHAR)
        acct = cols['account_number']
        # 순서: (은행 문자열, 계좌 문자열) 바이트 사전순(DP:176). 고정폭 바이트(NUL 채움)로 lexsort — NUL < 모든 글자라 접두가 먼저 옴.
        order = np.lexsort((acct.astype('S'), bank_norm.astype('S'))).astype(np.int64)
        self.bank = bank_norm[order]; self.acct = acct[order]
        dup = (self.bank[1:] == self.bank[:-1]) & (self.acct[1:] == self.acct[:-1])
        if dup.any():                                  # DP:186-187
            raise ValueError(f'{int(dup.sum())} duplicate registry keys: resolve explicitly, never choose first silently')
        ent = cols['entity_id'][order]
        ent_codes, ent_uniq = pd.factorize(ent, sort=True)
        self.ent = ent_codes.astype(np.int32); self.ent_ids = list(ent_uniq)
        et = pd.Series(cols['entity_name'][order]).str.extract(ENTITY_TYPE_RE)[0].fillna('UNKNOWN').to_numpy(dtype=object)  # DP:195
        vocab = list(ENTITY_TYPES) + sorted(set(et) - set(ENTITY_TYPES) - {'UNKNOWN'})
        vmap = {v: i for i, v in enumerate(vocab)}; vmap['UNKNOWN'] = -1
        ec, eu = pd.factorize(et)
        self.etype = np.array([vmap[x] for x in eu], np.int8)[ec]
        self.etype_vocab = vocab
        bc, bu = pd.factorize(cols['bank_name'][order])
        cmap = [_country_of(u) for u in bu]                                              # DP:194
        cvocab = sorted(set(cmap) - {'UNKNOWN'})
        cidx = {v: i for i, v in enumerate(cvocab)}; cidx['UNKNOWN'] = -1
        self.country = np.array([cidx[x] for x in cmap], np.int16)[bc]
        self.country_vocab = cvocab
        self.n_registry = n
        # 빠른 찾기: '정규형'(대문자 16진 9자, 은행 < 2^27) 계좌는 64비트 정수 키로, 나머지는 문자열 사전으로 찾음.
        data, off = _str_buffers(self.acct)
        m9, M = _fixed_rows(data, off, 9)
        canon = np.zeros(n, bool); val = np.zeros(n, np.int64)
        if len(M):
            d = _HEX_LUT[M]
            v = d[:, 0].astype(np.int64)
            for i in range(1, 9):
                v = v * 16 + d[:, i]
            r9 = np.flatnonzero(m9)
            canon[r9] = d.max(1) < 16; val[r9] = v
        bsmall = np.array([b < (1 << 27) for b in bank_int[order]])
        canon &= bsmall
        ci = np.flatnonzero(canon)
        bint = np.zeros(n, np.int64); bint[bsmall] = np.array([int(b) for b in self.bank[bsmall]], np.int64)
        k64 = (bint[ci] << 36) | val[ci]
        self.k64_index = pd.Index(k64)
        assert self.k64_index.is_unique
        self.k64_pos = ci.astype(np.int64)
        self.noncanon = {(self.bank[i], self.acct[i]): int(i) for i in np.flatnonzero(~canon)}
        self.unknown = {}                              # 계좌표에 없는 거래 계좌 → 임시 번호(n_registry + k)
        self.unknown_keys = []
        log(f'[accounts] {n:,}행 · entity {len(ent_uniq):,} · 비정규형 {len(self.noncanon)} · {time.time() - t0:.1f}s')

    def lookup(self, bank_codes, bank_norm_uniq, bank_int_rows, acct_col):
        """행마다 계좌 번호(int64). bank_codes/bank_norm_uniq: 은행 고유값 코드와 고유값별 정규화 문자열, bank_int_rows: 행별 int."""
        data, off = _str_buffers(acct_col)
        n = len(off) - 1
        out = np.full(n, -1, np.int64)
        m9, M = _fixed_rows(data, off, 9)
        if len(M):
            d = _HEX_LUT[M]
            okhex = d.max(1) < 16
            val = d[:, 0].astype(np.int64)
            for i in range(1, 9):
                val = val * 16 + d[:, i]
            if m9.all():
                rows9 = None; bint = bank_int_rows
            else:
                rows9 = np.flatnonzero(m9); bint = bank_int_rows[rows9]
            ok = okhex & (bint < (1 << 27))
            k = (bint << 36) | val
            gi = self.k64_index.get_indexer(k)
            hit = ok & (gi >= 0)
            if rows9 is None:
                out[hit] = self.k64_pos[gi[hit]]
            else:
                out[rows9[hit]] = self.k64_pos[gi[hit]]
        rest = np.flatnonzero(out < 0)                 # 비정규형이거나 계좌표에 없는 계좌(드묾) — 문자열 키로 찾음
        if len(rest):
            astr = _strings(acct_col, rest)
            for i, a in zip(rest, astr):
                if not re.fullmatch(r'[0-9A-Fa-f]+', a):       # DP:152
                    raise ValueError(f'invalid account string {a!r}')
                key = (bank_norm_uniq[bank_codes[i]], a)
                j = self.noncanon.get(key)
                if j is None:
                    j = self.unknown.get(key)
                    if j is None:
                        j = self.n_registry + len(self.unknown_keys)
                        self.unknown[key] = j; self.unknown_keys.append(key)
                out[i] = j
        return out

    @property
    def n_total(self):
        return self.n_registry + len(self.unknown_keys)


# ───────────────────────── 원장 짓기 ─────────────────────────
class _Builder:
    def __init__(self, accounts: _Accounts, cut_min: int, tmp: Path, carry: int, log):
        self.A = accounts; self.cut_min = cut_min; self.tmp = tmp; self.carry_n = carry; self.log = log
        self.bank_cache = {}                           # 원본 은행 문자열 → (int, 정규화 문자열, 해시)
        self.cat_hash = {}                             # 범주 문자열 → 해시
        self.ccy = {c: i for i, c in enumerate(CURRENCIES)}
        self.fmt = {c: i for i, c in enumerate(PAYMENT_FORMATS)}
        self.app = {name: _NpyAppender(tmp / f'{name}.npy', dt) for name, dt in ROW_FIELDS}
        self.bucket_dir = tmp / '_dedup_buckets'; self.bucket_dir.mkdir()
        self.bucket_f = [open(self.bucket_dir / f'{b:03d}.bin', 'wb') for b in range(256)]
        self.carry = None                              # (h, eid, strings 목록) — 앞 묶음 끝 쪽의 남긴 행
        self.dup_pairs = []                            # (지운 edge_id, 남긴 edge_id)
        self.verified_distinct = set()                 # 해시는 같지만 문자열로 다르다고 확인한 (앞 eid, 뒤 eid)
        self.n_total = 0; self.n_after_cut = 0; self.n_kept_pre = 0; self.n_hash_collide_local = 0
        self.ts_min = None; self.ts_max = None; self.n_before_base = 0
        self.day_counts = np.zeros(max(cut_min // 1440, 1) + 1, np.int64)
        self.cal = {'paid': [], 'recv': [], 'pcur': [], 'rcur': []}
        self.cal_end = minutes_of(CAL_END)
        self.acct_seen = np.zeros(accounts.n_registry, bool)
        self.anon_peak = 0.0
        self._stats_arrays()
        self.pat = None

    def _stats_arrays(self):
        nc = 256
        self.SC = np.full(nc, np.nan); self.MED = np.full(nc, np.nan); self.IQR = np.full(nc, np.nan)
        self.MU = np.full(nc, np.nan); self.SD = np.full(nc, np.nan)
        for c, i in self.ccy.items():
            if c in CURRENCY_STATS:
                self.SC[i], self.MED[i], self.IQR[i] = CURRENCY_STATS[c]
            if c in AMTZ_STATS:
                self.MU[i], self.SD[i] = AMTZ_STATS[c]

    # ── 범주 열 ──
    def _cat(self, col, vocab, what, allowed=None):
        codes, uniq = _factorize(col)
        cmap = np.empty(len(uniq), np.int64); hmap = np.empty(len(uniq), np.uint64)
        for j, u in enumerate(uniq):
            if (not u) or u != u.strip() or u.lower() in NULL_TOKENS:      # DP:150
                raise ValueError(f'{what}: invalid value {u!r}')
            if allowed is not None and u not in allowed:
                raise ValueError(f'{what}: invalid value {u!r}')
            if vocab is not None:
                if u not in vocab:
                    if len(vocab) >= 127:
                        raise ValueError(f'{what}: too many distinct values')
                    vocab[u] = len(vocab)
                    self.log(f'[warn] {what}: 처음 보는 값 {u!r} → 코드 {vocab[u]}')
                cmap[j] = vocab[u]
            h = self.cat_hash.get(u)
            if h is None:
                h = self.cat_hash[u] = _h64(what + '\x1f' + u)
            hmap[j] = h
        return cmap[codes], hmap[codes]

    def _banks(self, col):
        """은행 열 → (행별 int, 행별 원문 해시, 행별 고유값 코드, 고유값별 정규화 문자열). 정규화 = str(int(x))(DP:175)."""
        codes, uniq = _factorize(col)
        bi = np.empty(len(uniq), np.int64); bh = np.empty(len(uniq), np.uint64); bn = []
        cache = self.bank_cache
        for j, u in enumerate(uniq):
            r = cache.get(u)
            if r is None:
                if not re.fullmatch(r'[0-9]+', u) or int(u) >= (1 << 63):          # DP:151
                    raise ValueError(f'invalid bank {u!r}')
                v = int(u)
                r = cache[u] = (v, str(v), _h64('bank\x1f' + u))
            bi[j] = r[0]; bh[j] = r[2]; bn.append(r[1])
        return bi[codes], bh[codes], codes, bn

    @staticmethod
    def _check_decimal(col, what):
        """DP:153 정규식 [0-9]+([.][0-9]{1,8})? 와 DECIMAL(38,8) 범위(정수부 ≤ 30자리)를 확인.
        빠른 길: 글자가 모두 숫자/점이고, 점이 없거나 끝에서 셋째 자리에 하나뿐인 행. 나머지 행은 정규식으로 하나씩."""
        data, off = _str_buffers(col)
        n = len(off) - 1
        if n == 0:
            return
        lens = np.diff(off)
        if (lens < 1).any():
            raise ValueError(f'{what}: empty decimal string')
        isdot = data == 46
        if not (((data >= 48) & (data <= 57)) | isdot).all():
            raise ValueError(f'{what}: invalid characters (DP:153 rule)')
        ndot = np.add.reduceat(isdot.astype(np.int32), off[:-1])
        endm3 = off[1:] - 3
        dot3 = np.zeros(n, bool); okp = endm3 > off[:-1]
        dot3[okp] = isdot[endm3[okp]]
        fast = ((ndot == 0) & (lens <= 30)) | ((ndot == 1) & dot3 & (lens - 3 <= 30))
        rest = np.flatnonzero(~fast)
        if len(rest):
            rx = re.compile(r'[0-9]{1,30}([.][0-9]{1,8})?')
            for x in _strings(col, rest):
                if not rx.fullmatch(x):
                    raise ValueError(f'{what}: invalid decimal string {x!r} (DP:153 rule)')

    def _ts(self, col):
        """시각 열 → 분(int64). 대부분 'YYYY/MM/DD HH:MM' 16자 고정폭이라 바이트로 바로 셈. 아니면 strptime(DP:201 과 같은 형식)."""
        data, off = _str_buffers(col)
        n = len(off) - 1
        out = np.full(n, np.iinfo(np.int64).min, np.int64)
        m16, M = _fixed_rows(data, off, 16)
        if len(M):
            sep = (M[:, 4] == 47) & (M[:, 7] == 47) & (M[:, 10] == 32) & (M[:, 13] == 58)
            Dg = M[:, [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 14, 15]] - np.uint8(48)        # 숫자 아니면 9 보다 큼(uint8 넘침)
            dig = Dg.max(1) <= 9
            Dg = Dg.astype(np.int32)
            y = Dg[:, 0] * 1000 + Dg[:, 1] * 100 + Dg[:, 2] * 10 + Dg[:, 3]
            mo = Dg[:, 4] * 10 + Dg[:, 5]; dd = Dg[:, 6] * 10 + Dg[:, 7]
            hh = Dg[:, 8] * 10 + Dg[:, 9]; mi = Dg[:, 10] * 10 + Dg[:, 11]
            leap = ((y % 4 == 0) & (y % 100 != 0)) | (y % 400 == 0)
            dim = np.array([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], np.int32)[np.clip(mo, 0, 12)] + ((mo == 2) & leap)
            ok = sep & dig & (mo >= 1) & (mo <= 12) & (dd >= 1) & (dd <= dim) & (hh <= 23) & (mi <= 59) & (y >= 1)
            v = (_days_from_civil(y.astype(np.int64), mo.astype(np.int64), dd.astype(np.int64)) - _BASE_DAYS) * 1440 + hh * 60 + mi
            if m16.all():
                out[ok] = v[ok]
            else:
                rows = np.flatnonzero(m16); out[rows[ok]] = v[ok]
        rest = np.flatnonzero(out == np.iinfo(np.int64).min)
        if len(rest):                                  # 고정폭이 아닌 시각(드묾): 파이썬 strptime, 실패면 원래처럼 멈춤(DP:154)
            for i, s in zip(rest, _strings(col, rest)):
                out[i] = _ts_minutes_py(s)
        return out

    # ── 11열 → 수치 표현과 행 해시 ──
    def numeric(self, cols, ts):
        A = self.A
        fbi, fbh, fbc, fbu = self._banks(cols['from_bank'])
        tbi, tbh, tbc, tbu = self._banks(cols['to_bank'])
        src = A.lookup(fbc, fbu, fbi, cols['from_account'])
        dst = A.lookup(tbc, tbu, tbi, cols['to_account'])
        self._check_decimal(cols['amount_paid_raw'], 'amount_paid_raw')
        self._check_decimal(cols['amount_received_raw'], 'amount_received_raw')
        paid = _to_float(cols['amount_paid_raw']); recv = _to_float(cols['amount_received_raw'])
        pcur, pch = self._cat(cols['payment_currency'], self.ccy, 'currency')
        rcur, rch = self._cat(cols['receiving_currency'], self.ccy, 'currency')
        fmt, fh = self._cat(cols['payment_format'], self.fmt, 'payment_format')
        _, lh = self._cat(cols['label_raw'], None, 'label_raw', allowed={'0', '1'})   # DP:154 — 라벨은 해시에만 씀
        h = _row_hash([ts.view(np.uint64), src.view(np.uint64), dst.view(np.uint64), fbh, tbh,
                       paid.view(np.uint64), recv.view(np.uint64), pch, rch, fh, lh])
        return dict(src=src, dst=dst, paid=paid, recv=recv, pcur=pcur, rcur=rcur, fmt=fmt, h=h)

    def pz_az(self, paid, pcur):
        """paid_currency_z(DP:299, 309 FLOAT) · amt_z_pay_ccy(NBA:134-138)."""
        sc, med, iqr = self.SC[pcur], self.MED[pcur], self.IQR[pcur]
        with np.errstate(invalid='ignore', divide='ignore'):
            z = np.clip((np.log(1 + paid / sc) - med) / iqr, -8, 8)       # greatest(-8, least(8, (ln(1+paid/scale)-med)/iqr))
        z = np.where(np.isnan(sc), 0.0, z)                                 # 통계 없는 통화 → 0
        pz = z.astype(np.float32)
        mu, sd = self.MU[pcur], self.SD[pcur]
        okc = np.isfinite(mu) & np.isfinite(sd)
        a = np.zeros(len(paid), np.float64)
        with np.errstate(invalid='ignore', divide='ignore', over='ignore'):
            zr = (np.log(1.0 + paid[okc]) - mu[okc]) / (sd[okc] + 1e-6)
        a[okc] = np.clip(zr, -10.0, 10.0)
        a[~np.isfinite(a)] = 0.0
        return pz, a.astype(np.float32)

    # ── 패턴(Patterns.txt) ──
    def load_patterns(self, patterns_txt):
        """DP:209-226 과 같은 블록 규칙으로 읽음. 시각 칸만 먼저 보고 자르는 시각 이후 줄은 버림(개수만 셈).
        BEGIN 줄은 블록 구조(BEGIN · END 짝)만 먼저 보고, 유형 글자는 그 시도에 자르는 시각 전 줄이 처음 나올 때 해석 · 확인함
        → 봉인 뒤 줄만 있는 시도의 유형은 읽지 않음(DP 는 전부 읽지만, 쓰는 행 · 유형은 같음)."""
        t0 = time.time()
        recs = []; n_tx = 0; n_after = 0; attempt = -1; in_block = False; begin = None; ptype = None
        lo, hi = minutes_of(TRAIN_FIRST), minutes_of(TRAIN_LAST) + 1440
        seen = set(); n_att_used = 0
        with open(patterns_txt, newline='') as f:
            for line_no, line in enumerate(f, 1):
                raw = line.rstrip('\r\n'); s = raw.strip()
                if not s:
                    continue
                if s.startswith('BEGIN'):
                    if in_block:
                        raise ValueError('Nested pattern block')                       # DP:215
                    attempt += 1; in_block = True; begin = (s, line_no); ptype = None   # 유형 글자는 아직 해석하지 않음
                elif s.startswith('END'):
                    if not in_block:
                        raise ValueError('Unmatched END')                              # DP:220
                    in_block = False; begin = None; ptype = None
                else:
                    if not in_block:
                        raise ValueError(f'Bad pattern transaction {line_no}')         # DP:224
                    row_id = n_tx; n_tx += 1
                    ts_s = raw.split(',', 1)[0] if not raw.startswith('"') else next(csv.reader([raw]))[0]
                    tm = _ts_minutes_py(ts_s)
                    if lo <= tm < hi:
                        seen.add(attempt)                     # 'seen' = train 창(08-08~09-27)에 줄이 하나라도 있는 시도
                    if tm >= self.cut_min:
                        n_after += 1; continue                # 자르는 시각 이후 줄 — 다른 칸은 보지 않음
                    if ptype is None:                         # 이 시도의 자르는 시각 전 첫 줄 → 이제 BEGIN 줄의 유형을 해석(DP:216-218)
                        m = PATTERN_BEGIN_RE.fullmatch(begin[0])
                        if not m:
                            raise ValueError(f'Bad BEGIN line {begin[1]}')
                        ptype = m.group(1).strip()
                        if ptype not in CLASS_NAMES8:
                            raise ValueError(f'unknown pattern type {ptype!r} (BEGIN line {begin[1]})')
                        n_att_used += 1
                    vals = next(csv.reader([raw]))
                    if len(vals) != 11:
                        raise ValueError(f'Bad pattern transaction {line_no}')
                    recs.append((row_id, attempt, ptype, tm, tuple(vals)))
        if in_block:
            raise ValueError('Unclosed pattern block')                                 # DP:226
        n = len(recs)
        cols = {c: np.array([r[4][j] for r in recs], dtype=object) for j, c in enumerate(RAW_COLUMNS)}
        ts = np.array([r[3] for r in recs], np.int64)
        num = self.numeric(cols, ts) if n else dict(h=np.zeros(0, np.uint64))
        h = num['h']
        hu, first, inv = np.unique(h, return_index=True, return_inverse=True)
        groups = {}
        for li, g in enumerate(inv):
            groups.setdefault(int(g), []).append(li)
        self.pat = dict(h_index=pd.Index(hu), h_groups=[groups[g] for g in range(len(hu))], n=n, row_id=np.array([r[0] for r in recs], np.int64), attempt=np.array([r[1] for r in recs], np.int64),
                        y=np.array([CLASS_NAMES8.index(r[2]) for r in recs], np.int8), ts=ts, vals=[r[4] for r in recs],
                        eid=np.full(n, -1, np.int64), n_tx_lines=n_tx, n_after_cut=n_after,
                        n_attempts=attempt + 1, n_attempts_used=n_att_used, seen_attempts=np.array(sorted(seen), np.int64))
        self.log(f'[patterns] 거래 줄 {n_tx:,} · 시도(BEGIN 블록 수만 셈) {attempt + 1:,} · 그중 자르는 시각 전 줄이 있는 시도 {n_att_used:,} · '
                 f'자르는 시각 이후 버린 줄 {n_after:,} · 대조할 줄 {n:,} · {time.time() - t0:.1f}s')

    def _match_patterns(self, cols, h, eid):
        P = self.pat
        if P is None or P['n'] == 0 or len(h) == 0:
            return
        gi = P['h_index'].get_indexer(h)
        hit = np.flatnonzero(gi >= 0)
        if not len(hit):
            return
        rs = _rows_strings(cols, hit)
        for r, sr in zip(hit, rs):
            for li in P['h_groups'][gi[r]]:
                if P['vals'][li] == sr:                        # 11열 문자열 완전 일치(DP:229-231)
                    if P['eid'][li] != -1:
                        raise ValueError('Ambiguous pattern transaction match')            # DP:233
                    P['eid'][li] = eid[r]

    # ── 묶음 하나 처리 ──
    def process(self, cols, eid0):
        n = _len(cols['timestamp_raw'])
        self.n_total += n
        next_start = eid0 + n
        # 1) 시각만 먼저 해석 → 자르는 시각 이후 행은 버림(다른 열은 해석 · 해시 · 저장하지 않음 · 개수만 셈. 읽기 도구가 나눈 글자 칸은 그대로 버려짐)
        ts_all = self._ts(cols['timestamp_raw'])
        keep = ts_all < self.cut_min
        self.n_after_cut += int(n - keep.sum())
        if not keep.all():
            idx = np.flatnonzero(keep)
            cols = {c: _filter(cols[c], keep) for c in RAW_COLUMNS}
            ts = ts_all[idx]; eid = eid0 + idx
        else:
            ts = ts_all; eid = eid0 + np.arange(n, dtype=np.int64)
        del ts_all
        k = len(ts)
        self.n_kept_pre += k
        if k == 0:
            self._set_carry(None, None, None, next_start)
            return
        # 2) 수치 표현 · 행 해시
        num = self.numeric(cols, ts)
        h = num['h']
        # 3) 완전 중복 제거(DP:161-174): 11열이 모두 같으면 가장 작은 edge_id 만 남김. 앞 묶음 끝 carry 행과 함께 봄.
        drop = self._dedup(cols, h, eid)
        if drop.any():
            kk = ~drop
            ts, eid, h = ts[kk], eid[kk], h[kk]
            num = {key: v[kk] for key, v in num.items()}
            cols = {c: _filter(cols[c], kk) for c in RAW_COLUMNS}
        self._set_carry(h, eid, cols, next_start)
        # 4) 패턴 줄과 대조(남긴 행만 — DP:230-231 은 retained 와 JOIN)
        self._match_patterns(cols, h, eid)
        # 5) 행 파일 쓰기
        pz, az = self.pz_az(num['paid'], num['pcur'])
        src, dst = num['src'], num['dst']
        out = dict(eid=eid, ts=ts, src=src, dst=dst, pz=pz, az=az, pcur=num['pcur'], rcur=num['rcur'], fmt=num['fmt'])
        for name, _ in ROW_FIELDS:
            self.app[name].append(out[name])
        # 6) 보정 구간 행(통화 상수 다시 계산용) · 날짜별 수 · 계좌 쓰임 · 전역 중복 확인 통
        cm = (ts >= 0) & (ts < self.cal_end)
        if cm.any():
            self.cal['paid'].append(num['paid'][cm]); self.cal['recv'].append(num['recv'][cm])
            self.cal['pcur'].append(num['pcur'][cm].astype(np.int16)); self.cal['rcur'].append(num['rcur'][cm].astype(np.int16))
        self.n_before_base += int((ts < 0).sum())
        dmin, dmax = int(ts.min()), int(ts.max())
        self.ts_min = dmin if self.ts_min is None else min(self.ts_min, dmin)
        self.ts_max = dmax if self.ts_max is None else max(self.ts_max, dmax)
        dd = ts[ts >= 0] // 1440
        self.day_counts[:] += np.bincount(dd, minlength=len(self.day_counts))[:len(self.day_counts)]
        reg = self.A.n_registry
        self.acct_seen[src[src < reg]] = True; self.acct_seen[dst[dst < reg]] = True
        b = (h >> np.uint64(56)).astype(np.uint8)
        o = np.argsort(b, kind='stable')
        rec = np.empty(len(h), dtype=[('h', '<u8'), ('e', '<i4')])
        rec['h'] = h[o]; rec['e'] = eid[o]
        cnt = np.bincount(b, minlength=256); st = 0
        for bi in np.flatnonzero(cnt):
            self.bucket_f[bi].write(memoryview(rec[st:st + cnt[bi]]).cast('B')); st += cnt[bi]
        self.anon_peak = max(self.anon_peak, cgroup_anon_gib())

    def _dedup(self, cols, h, eid):
        """완전 중복 표시(DP:161-174). 해시가 같은 앞 행과 11열 문자열을 직접 비교해 확정함(해시 충돌은 남김).
        앞 묶음 끝의 carry 행(원본 행 번호 차 ≤ carry_n)과 이 묶음을 함께 봄 — HI-Large 의 중복 짝은 모두 15행 안(duplicate_audit.csv)."""
        k = len(h)
        if self.carry is not None:
            ch, ce, cs = self.carry
        else:
            ch, ce, cs = np.zeros(0, np.uint64), np.zeros(0, np.int64), []
        m = len(ch)
        H = np.concatenate([ch, h])
        flag = pd.Series(H).duplicated(keep='first').to_numpy().copy()
        flag[:m] = False
        drop = np.zeros(k, bool)
        cand = np.flatnonzero(flag)
        if len(cand):
            cache = {}

            def S(i):                                   # 결합 위치 i 의 11열 문자열
                if i < m:
                    return cs[i]
                if i not in cache:
                    cache[i] = _rows_strings(cols, [i - m])[0]
                return cache[i]
            for r in cand:
                prev = np.flatnonzero(H[:r] == H[r])
                hit = None
                for j in prev:
                    if j >= m and drop[j - m]:
                        continue
                    if S(j) == S(r):
                        hit = j; break
                    ej = ce[j] if j < m else eid[j - m]
                    self.verified_distinct.add((int(ej), int(eid[r - m])))   # 문자열로 확인한 '다른 행' 짝
                if hit is None:
                    self.n_hash_collide_local += 1      # 해시만 같고 문자열은 다름 → 남김
                    continue
                drop[r - m] = True                      # 같은 문자열의 가장 앞 행(가장 작은 edge_id)만 남음
                gid = ce[hit] if hit < m else eid[hit - m]
                self.dup_pairs.append((int(eid[r - m]), int(gid)))
        return drop

    def _set_carry(self, h, eid, cols, next_start):
        """다음 묶음과 비교할 끝 쪽 남긴 행(eid ≥ next_start − carry_n)을 남김. h/eid/cols 는 중복을 뺀 이 묶음 행."""
        lo = next_start - self.carry_n
        if self.carry is not None:
            ch, ce, cs = self.carry
            mc = ce >= lo
            ch, ce, cs = ch[mc], ce[mc], [s for s, t in zip(cs, mc) if t]
        else:
            ch, ce, cs = np.zeros(0, np.uint64), np.zeros(0, np.int64), []
        if h is not None and len(h):
            sel = np.flatnonzero(eid >= lo)
            if len(sel):
                ch = np.concatenate([ch, h[sel]]); ce = np.concatenate([ce, eid[sel]])
                cs = cs + _rows_strings(cols, sel)
        self.carry = (ch, ce, cs) if len(ch) else None

    def close_rows(self):
        for a in self.app.values():
            a.close()
        for f in self.bucket_f:
            f.close()


def _engine_batches(trans_csv, engine, chunk_rows, threads, log):
    """원본 CSV 를 원래 행 순서대로 묶음으로 냄. (cols, eid0)."""
    if engine == 'arrow':
        pa.set_cpu_count(max(1, threads)); pa.set_io_thread_count(max(1, threads))
        ro = pacsv.ReadOptions(use_threads=threads > 1, block_size=ARROW_BLOCK, skip_rows=1, column_names=RAW_COLUMNS)
        co = pacsv.ConvertOptions(column_types={c: pa.string() for c in RAW_COLUMNS}, strings_can_be_null=False,
                                  quoted_strings_can_be_null=False, check_utf8=True)
        rd = pacsv.open_csv(trans_csv, read_options=ro, convert_options=co)
        buf = []; nb = 0; off = 0
        for b in rd:
            buf.append(b); nb += b.num_rows
            if nb >= chunk_rows:
                t = pa.Table.from_batches(buf)
                yield {c: t.column(c).combine_chunks() for c in RAW_COLUMNS}, off
                off += nb; buf = []; nb = 0
                pa.default_memory_pool().release_unused()
        if buf:
            t = pa.Table.from_batches(buf)
            yield {c: t.column(c).combine_chunks() for c in RAW_COLUMNS}, off
    else:
        off = 0
        for df in pd.read_csv(trans_csv, header=None, skiprows=1, names=RAW_COLUMNS, dtype=object, na_filter=False,
                              chunksize=chunk_rows, engine='c'):
            n = len(df)
            yield {c: df[c].to_numpy(dtype=object) for c in RAW_COLUMNS}, off
            off += n


def _write_npy(path, arr):
    np.save(path, np.ascontiguousarray(arr))


def _calibration_rederive(cal, ccy_vocab):
    """보정 구간 행으로 통화 상수를 다시 계산(확인용). DP:279-283 과 NBA:116-119 를 numpy 로 옮김(recompute.py 와 같은 식)."""
    paid = np.concatenate(cal['paid']) if cal['paid'] else np.zeros(0)
    recv = np.concatenate(cal['recv']) if cal['recv'] else np.zeros(0)
    pcur = np.concatenate(cal['pcur']) if cal['pcur'] else np.zeros(0, np.int16)
    rcur = np.concatenate(cal['rcur']) if cal['rcur'] else np.zeros(0, np.int16)
    inv = {v: k for k, v in ccy_vocab.items()}
    A_c = np.concatenate([pcur, rcur]); A_a = np.concatenate([paid, recv])      # paid(지불 통화) ∪ received(받는 통화)
    cs, az = {}, {}
    for code in np.unique(A_c):
        a = A_a[A_c == code]
        scale = float(np.median(a[a > 0]))                                       # median(a) FILTER(WHERE a>0)
        z = np.log(1 + a / scale)                                                # ln(1+a/scale), 모든 a
        q25, q75 = np.quantile(z, [.25, .75])                                    # quantile_cont = 선형 보간
        cs[inv[int(code)]] = (scale, float(np.median(z)), max(float(q75 - q25), .001))
    for code in np.unique(pcur):
        v = np.log(1.0 + paid[pcur == code])                                    # edge_id 순서 그대로
        az[inv[int(code)]] = (float(v.mean()), float(v.std(ddof=0)))
    d_cs = {c: [abs(x - y) for x, y in zip(cs[c], CURRENCY_STATS[c])] if c in CURRENCY_STATS else None for c in cs}
    d_az = {c: [abs(x - y) for x, y in zip(az[c], AMTZ_STATS[c])] if c in AMTZ_STATS else None for c in az}
    mx_cs = max((max(v) for v in d_cs.values() if v is not None), default=float('nan'))
    mx_az = max((max(v) for v in d_az.values() if v is not None), default=float('nan'))
    return dict(rows=int(len(paid)), currency_stats=cs, amtz=az, max_abs_diff_currency_stats=mx_cs, max_abs_diff_amtz=mx_az,
                missing_vs_embedded=sorted(set(CURRENCY_STATS) - set(cs)), extra_vs_embedded=sorted(set(cs) - set(CURRENCY_STATS)))


def build_ledger(trans_csv, accounts_csv, patterns_txt, d_max, out_dir, threads=2, engine='auto',
                 chunk_rows=1_000_000, carry=64, expected_rows='auto', expected_bytes='auto', reuse=True, log=print):
    """원본 3파일 → 압축 원장(out_dir). d_max(포함)까지의 행만 남김.

    engine: 'auto'(pyarrow 있으면 'arrow') · 'arrow' · 'pandas'. 두 엔진의 결과는 같아야 함.
    expected_rows/bytes: 'auto' 면 파일 이름이 HI-Large_Trans.csv 일 때 179,702,229행 · 17,052,760,651 B 를 요구함. None 이면 확인 안 함.
    reuse: out_dir 에 같은 원본(크기 · 수정 시각 · 앞뒤 해시) · 같은 코드 · d_max 이상인 완성 원장이 있으면 그대로 읽어 돌려줌.
           아니면 까닭을 log 에 적고, 이 코드가 지은 옛 원장 폴더만 먼저 지운 뒤(남의 파일이 있으면 멈춤) 빈 공간을 확인하고 새로 지음.
    """
    t_start = time.time()
    d_max = str(pd.Timestamp(d_max).date())
    check_seal(d_max, 'build_ledger')
    if d_max < TRAIN_FIRST:
        raise ValueError(f'd_max {d_max} < {TRAIN_FIRST}: 보정 구간만으로는 예측할 날이 없음(DP:350)')
    trans_csv, accounts_csv, patterns_txt = Path(trans_csv), Path(accounts_csv), Path(patterns_txt)
    out_dir = Path(out_dir)
    if engine == 'auto':
        engine = 'arrow' if HAVE_ARROW else 'pandas'
    if engine == 'arrow' and not HAVE_ARROW:
        raise RuntimeError('pyarrow 가 없음 — engine="pandas" 를 쓰세요')
    exp = EXPECTED.get(trans_csv.name, {})
    if expected_rows == 'auto':
        expected_rows = exp.get('rows')
    if expected_bytes == 'auto':
        expected_bytes = exp.get('bytes')
    fps = source_fingerprints(trans_csv, accounts_csv, patterns_txt)
    if expected_bytes is not None and fps['transactions']['bytes'] != expected_bytes:
        raise ValueError(f"거래 파일 크기 {fps['transactions']['bytes']:,} B ≠ 기대 {expected_bytes:,} B — 원본 그대로가 아님(edge_id 가 달라짐)")
    if reuse:
        L, why = reuse_status(out_dir, fps, d_max)
        if L is not None:
            log(f'[ledger] 기존 원장을 다시 씀: {out_dir} ({why})')
            return L
        log(f'[ledger] 새로 지음: {why}')
    _check_header(trans_csv, HEADERS, 'transactions')
    cut_min = minutes_of(d_max) + 1440               # d_max 다음 날 00:00(이 시각 이후 행은 버림)
    tmp = out_dir.with_name(out_dir.name + '.building')
    for p_, what in ((out_dir, '다시 쓸 수 없는 옛 원장'), (tmp, '지난번에 짓다 만 원장')):
        if p_.is_symlink() or (p_.exists() and not _is_own_dir(p_)):
            _remove_own_dir(p_, what, log)            # 링크이거나 남의 파일이 있으면 여기서 멈춤(아무것도 지우지 않음)
    # 빈 공간 확인(옛 원장은 새로 짓기 전에 지우므로 그 크기만큼은 돌아옴)
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(out_dir.parent).free
    back = _dir_bytes(out_dir) + _dir_bytes(tmp)
    if free + back < BUILD_NEED_BYTES:
        raise RuntimeError(f'[디스크 부족] 원장을 지으려면 {out_dir.parent} 에 빈 공간이 약 {BUILD_NEED_BYTES / 1e9:.0f} GB 필요합니다'
                           f'(원장 3.9 GB + 짓는 동안 임시 1.7 GB). 지금 빈 공간 {free / 1e9:.1f} GB'
                           + (f' + 지울 옛 원장 {back / 1e9:.1f} GB' if back else '') +
                           '. SCRATCH 를 빈 공간이 넉넉한 폴더로 바꾸거나 공간을 비운 뒤 다시 실행해 주세요. 아무것도 지우거나 쓰지 않았습니다.')
    _remove_own_dir(tmp, '지난번에 짓다 만 원장', log)
    _remove_own_dir(out_dir, '다시 쓸 수 없는 옛 원장', log)   # 새 원장을 다 지은 뒤가 아니라 지금 지움 → 디스크가 두 벌 들지 않음
    tmp.mkdir(parents=True)
    (tmp / LEDGER_MARK).write_text('유형8_분류기_4모델 노트북(ledger.py)이 지은 원장 폴더. 지워도 됨(다음 실행 때 다시 지음).\n', encoding='utf-8')
    log(f'[ledger] 시작 engine={engine} d_max={d_max} cut={cut_min} min chunk={chunk_rows:,} carry={carry} → {tmp}')
    T = {}
    t0 = time.time(); A = _Accounts(accounts_csv, log=log); T['accounts_sec'] = round(time.time() - t0, 1)
    B = _Builder(A, cut_min, tmp, carry, log)
    t0 = time.time(); B.load_patterns(patterns_txt); T['patterns_sec'] = round(time.time() - t0, 1)
    t0 = time.time(); nchunk = 0; next_log = time.time() + 60
    for cols, eid0 in _engine_batches(trans_csv, engine, chunk_rows, threads, log):
        n = _len(cols['timestamp_raw'])
        if eid0 + n > np.iinfo(np.int32).max:
            raise ValueError('행 번호가 int32 를 넘음')
        B.process(cols, eid0)
        del cols
        nchunk += 1
        if time.time() > next_log:
            el = time.time() - t0
            log(f'[scan] {B.n_total:,}행 · 남김 {B.app["eid"].n:,} · 버림(시각) {B.n_after_cut:,} · 중복 {len(B.dup_pairs)} · '
                f'{el:.0f}s ({B.n_total / el / 1e6:.2f}M행/s) · maxrss {_rss_gib():.2f} GiB · anon {cgroup_anon_gib():.2f} GiB')
            next_log = time.time() + 60
    B.close_rows()
    T['scan_sec'] = round(time.time() - t0, 1)
    log(f'[scan] 끝 {B.n_total:,}행 {T["scan_sec"]}s · 남김 {B.app["eid"].n:,} · 중복 {len(B.dup_pairs)}')
    if expected_rows is not None and B.n_total != expected_rows:
        raise ValueError(f'원본 행 수 {B.n_total:,} ≠ 기대 {expected_rows:,} — 원본 그대로가 아님. 멈춤.')
    n_rows = B.app['eid'].n
    # 계좌표에 없는 거래 계좌가 있었으면 사전순 번호로 다시 매김(HI-Large 는 0개 — kyc_unmatched_nodes 0)
    t0 = time.time()
    nU = len(A.unknown_keys)
    bank = list(A.bank) + [k[0] for k in A.unknown_keys]
    acct = list(A.acct) + [k[1] for k in A.unknown_keys]
    ent = np.r_[A.ent, np.full(nU, -1, np.int32)]
    etype = np.r_[A.etype, np.full(nU, -1, np.int8)]
    country = np.r_[A.country, np.full(nU, -1, np.int16)]
    seen = np.r_[B.acct_seen, np.zeros(nU, bool)]
    if nU:
        log(f'[warn] 계좌표에 없는 거래 계좌 {nU}개 — 번호를 사전순으로 다시 매김')
        order = np.asarray(sorted(range(len(bank)), key=lambda i: (bank[i], acct[i])), np.int64)
        newid = np.empty(len(order), np.int64); newid[order] = np.arange(len(order))
        for name in ('src', 'dst'):
            mm = np.load(tmp / f'{name}.npy', mmap_mode='r+')
            for s in range(0, len(mm), 10_000_000):
                v = np.asarray(mm[s:s + 10_000_000])
                seen[v[v >= A.n_registry]] = True
                mm[s:s + 10_000_000] = newid[v]
            mm.flush(); del mm
        bank = [bank[i] for i in order]; acct = [acct[i] for i in order]
        ent, etype, country, seen = ent[order], etype[order], country[order], seen[order]
    T['remap_sec'] = round(time.time() - t0, 1)
    # 전역 중복 확인: 같은 해시가 남은 행에 두 번 있으면, 남긴 수치 열까지 같은지 봄(같으면 묶음 사이 중복을 놓친 것 → 멈춤)
    t0 = time.time()
    rows = {name: np.load(tmp / f'{name}.npy', mmap_mode='r') for name, _ in ROW_FIELDS}
    suspects = []; collide = 0
    for b in range(256):
        a = np.fromfile(B.bucket_dir / f'{b:03d}.bin', dtype=[('h', '<u8'), ('e', '<i4')])
        if len(a) < 2:
            continue
        o = np.argsort(a['h'], kind='stable'); hs = a['h'][o]; es = a['e'][o]
        eq = np.flatnonzero(hs[1:] == hs[:-1])
        if not len(eq):
            continue
        st = np.unique(np.r_[eq, eq + 1])               # 같은 해시 묶음에 든 위치
        for g in np.split(st, np.flatnonzero(np.diff(hs[st]) != 0) + 1):
            ge = sorted(int(x) for x in es[g])
            for i in range(len(ge)):
                for j in range(i + 1, len(ge)):
                    e1, e2 = ge[i], ge[j]
                    if (e1, e2) in B.verified_distinct:
                        continue
                    p1, p2 = np.searchsorted(rows['eid'], [e1, e2])
                    same = all(rows[nm][p1] == rows[nm][p2] for nm in ('ts', 'src', 'dst', 'pz', 'az', 'pcur', 'rcur', 'fmt'))
                    if same:
                        suspects.append((e1, e2))
                    else:
                        collide += 1
    shutil.rmtree(B.bucket_dir)
    T['dedup_global_check_sec'] = round(time.time() - t0, 1)
    if suspects:
        raise RuntimeError(f'묶음 경계를 넘는 중복 의심 {len(suspects)}쌍(예: {suspects[:5]}) — carry 를 늘려 다시 지으세요')
    # 통화 상수 다시 계산(확인용) — 노트북은 넣은 상수를 씀
    t0 = time.time()
    cal = _calibration_rederive(B.cal, B.ccy); B.cal = None
    T['calibration_sec'] = round(time.time() - t0, 1)
    # 패턴 결과
    P = B.pat
    unmatched = int((P['eid'] < 0).sum())
    if unmatched:
        raise ValueError(f'Unmatched pattern rows: {unmatched} (DP:232)')
    if len(np.unique(P['eid'])) != P['n']:
        raise ValueError('One edge belongs to multiple pattern records (DP:234)')
    pday = (P['ts'] // 1440).astype(np.int16)
    o = np.lexsort((P['eid'], pday))                    # 날짜 순 → 날짜 안에서 edge_id 오름차순(val_pattern55 순서)
    pat_seen = np.isin(P['attempt'], P['seen_attempts'])
    _write_npy(tmp / 'pat_eid.npy', P['eid'][o].astype(np.int64))
    _write_npy(tmp / 'pat_day.npy', pday[o])
    _write_npy(tmp / 'pat_ts.npy', P['ts'][o].astype(np.int32))
    _write_npy(tmp / 'pat_y.npy', P['y'][o])
    _write_npy(tmp / 'pat_attempt.npy', P['attempt'][o].astype(np.int32))
    _write_npy(tmp / 'pat_seen.npy', pat_seen[o])
    _write_npy(tmp / 'pat_row.npy', P['row_id'][o].astype(np.int32))
    # 계좌 표
    _write_npy(tmp / 'acct_bank.npy', np.array(bank, dtype='S'))
    _write_npy(tmp / 'acct_num.npy', np.array(acct, dtype='S'))
    _write_npy(tmp / 'acct_ent.npy', ent.astype(np.int32))
    _write_npy(tmp / 'acct_etype.npy', etype.astype(np.int8))
    _write_npy(tmp / 'acct_country.npy', country.astype(np.int16))
    _write_npy(tmp / 'acct_active.npy', seen)
    _write_npy(tmp / 'ent_id.npy', np.array(A.ent_ids, dtype='S'))
    dp = np.array(sorted(B.dup_pairs), np.int64).reshape(-1, 2)
    _write_npy(tmp / 'dup_pairs.npy', dp)
    # meta
    day0 = np.datetime64(BASE_DATE)
    pat_days = {str(day0 + int(d)): int(c) for d, c in zip(*np.unique(pday, return_counts=True))}
    meta = dict(
        format_version=FORMAT_VERSION, complete=True, created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        code_sha256=code_sha256(), engine=engine, chunk_rows=chunk_rows, carry=carry,
        threads=threads, d_max=d_max, cut_minute=cut_min, base_date=BASE_DATE, test_start=TEST_START,
        sources=fps,
        rows=dict(source_total=B.n_total, dropped_after_cut=B.n_after_cut, kept_before_dedup=B.n_kept_pre,
                  duplicates_removed=len(B.dup_pairs), ledger=n_rows, before_base_date=B.n_before_base,
                  ts_min=B.ts_min, ts_max=B.ts_max, local_hash_collisions=B.n_hash_collide_local,
                  global_hash_collisions=collide, global_dup_suspects=0, verified_distinct_pairs=len(B.verified_distinct)),
        day_counts={str(day0 + i): int(c) for i, c in enumerate(B.day_counts) if c},
        accounts=dict(registry=A.n_registry, unknown=nU, total=len(bank), active=int(seen.sum()), entities=len(A.ent_ids)),
        vocab=dict(currency=list(B.ccy), payment_format=list(B.fmt), country=A.country_vocab, entity_type=A.etype_vocab,
                   class_names=CLASS_NAMES8),
        embedded=dict(currency_stats=CURRENCY_STATS, amtz=AMTZ_STATS, entity_types=ENTITY_TYPES, calibration=[CAL_START, CAL_END],
                      train_window=[TRAIN_FIRST, TRAIN_LAST]),
        calibration_check=cal,
        patterns=dict(tx_lines=P['n_tx_lines'], attempts=P['n_attempts'], attempts_before_cut=P['n_attempts_used'],
                      dropped_after_cut=P['n_after_cut'], matched=P['n'],
                      seen_attempts=int(len(P['seen_attempts'])), per_day=pat_days,
                      per_type={CLASS_NAMES8[i]: int(c) for i, c in enumerate(np.bincount(P['y'], minlength=8))}),
        timings=T, peak=dict(maxrss_gib=round(_rss_gib(), 3), cgroup_anon_peak_gib=round(B.anon_peak, 3)),
        files={f.name: list(np.load(f, mmap_mode='r').shape) for f in sorted(tmp.glob('*.npy'))},
    )
    meta['timings']['total_sec'] = round(time.time() - t_start, 1)
    (tmp / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=1, default=float))
    _remove_own_dir(out_dir, '같은 자리에 그사이 생긴 원장', log)   # 보통은 위에서 이미 지워 없음
    tmp.rename(out_dir)
    log(f'[ledger] 끝 {out_dir} · {n_rows:,}행 · {meta["timings"]["total_sec"]}s · maxrss {meta["peak"]["maxrss_gib"]} GiB')
    return load_ledger(out_dir)


class Ledger:
    """load_ledger 결과. ledger['ts'] 처럼 이름으로 memmap 을 꺼냄. meta 는 dict."""

    def __init__(self, path, meta, arrays):
        self.path = Path(path); self.meta = meta; self.arrays = arrays

    def __getitem__(self, k):
        return self.arrays[k]

    def __contains__(self, k):
        return k in self.arrays

    @property
    def n(self):
        return int(self.meta['rows']['ledger'])

    def keys(self):
        return self.arrays.keys()

    def __repr__(self):
        return f'Ledger({self.path}, rows={self.n:,}, d_max={self.meta["d_max"]})'


def load_ledger(out_dir, sources=None, mmap=True) -> Ledger:
    """완성 원장 읽기. sources={'transactions': 경로, ...} 를 주면 원본이 같은지(크기 · 수정 시각 · 앞뒤 해시)도 확인함."""
    out_dir = Path(out_dir)
    meta = json.loads((out_dir / 'meta.json').read_text())
    if not meta.get('complete') or meta.get('format_version') != FORMAT_VERSION:
        raise ValueError(f'원장이 완성되지 않았거나 형식이 다름: {out_dir}')
    check_seal(meta['d_max'], 'load_ledger')
    if sources:
        for k, p in sources.items():
            if not _same_source(meta['sources'][k], fingerprint(p)):
                raise ValueError(f'원본 {k} 가 원장을 지을 때와 다름: {p}')
    arrays = {}
    for name, shape in meta['files'].items():
        a = np.load(out_dir / name, mmap_mode='r' if mmap else None)
        if list(a.shape) != shape:
            raise ValueError(f'{name}: shape {a.shape} ≠ meta {shape}')
        arrays[name[:-4]] = a
    return Ledger(out_dir, meta, arrays)


if __name__ == '__main__':                               # 명령줄: python ledger.py TRANS ACC PAT D_MAX OUT [engine] [chunk_rows]
    a = sys.argv[1:]
    eng = a[5] if len(a) > 5 else 'auto'
    ch = int(a[6]) if len(a) > 6 else 1_000_000
    L = build_ledger(a[0], a[1], a[2], a[3], a[4], engine=eng, chunk_rows=ch,
                     log=lambda s: print(time.strftime('%H:%M:%S'), s, flush=True))
    print(json.dumps(L.meta['rows'], indent=1))
'''

In [ ]:
# @title 코드 ② 거래 17열(x17.py): 2022-08-01 부터 누적한 이력으로 아무 거래나 — 숨김 · 고치지 마세요
_MODULE_SRC['x17'] = r'''"""x17 단계 — 원장(ledger)에서 거래 17열(edge 16 + amt_z_pay_ccy)을 아무 거래 집합에나 계산한다.

17열은 LightGBM 55열의 38:55 · SALT/TC 캐시의 ea·tx · TGT 캐시의 x·tx 에 그대로 쓰인다.
값은 그 거래 자신의 시각 기준이고, 이력은 2022-08-01 부터 기간 제한 없이 누적된다(DP:253-263, 293-296).
그래서 '그날 D' 는 값을 바꾸지 않는다. D 를 주면 대상 거래가 모두 D 날인지 확인하는 데만 쓴다.

원래 코드(읽기 전용):
  (경로는 팀 작업 공간의 Final_preprocess/ 부터)
  DP  = Final_preprocess/태훈님_전처리/feature_code_40_20260921/CODE/daily_pipeline.py
  NBA = Final_preprocess/HI-Large_EDA/tgnn36_20260928/_builders/build_nb_A.py
  열 순서 = EDGE_FEATURES[:16] (DP:28-32) + amt_z_pay_ccy (모델묶음_유형8_최종후보_20261016/data/val_pattern55.npz features[38:55] 와 같음).

계산 방법(DuckDB 없음, numpy 만):
  - 필요한 끝점 계좌(보낸 계좌 S · 받은 계좌 R)와 쌍(s→d) · 역쌍(d→s)의 이력만 원장에서 골라
    (무리 번호)·2^20 + 분 꼴의 int64 키로 만들고, 정렬한 뒤 searchsorted 로
    '1분 전까지 개수 · 마지막 시각 · 1시간 개수 · 그날 00:00 전 개수 · 그 마지막 시각' 을 셈.
  - 이력 행이 budget_rows 를 넘으면 계좌 번호 구간으로 나눠(샤드) 여러 번 훑음 → 메모리 상한을 지킴.
  - 원장 파일은 memmap 으로 만지지 않고 os.preadv 로 묶음씩 읽음 → 읽은 쪽이 page cache 에만 남고
    프로세스 RSS(파일 페이지)에 쌓이지 않음.
  - ln 은 정수 인자를 모아 math.log(= libm log, DuckDB ln 이 부르는 std::log 와 같은 함수)로 계산한 뒤 float32 로 자름
    (DP:309 cast(... AS FLOAT)).

쓰는 법:
  X = compute_x17(L, eids)              # eids 순서 그대로 float32 [n, 17]
  X = x17_day(L, '2022-10-06')          # 그날 패턴 행 전부(= val55 그날 행 · 순서)
  tab = x17_table(L, eids); X = x17_take(tab, q)   # 하루치 합집합을 한 번 계산해 두고 여러 소비자가 꺼내 씀
"""
from __future__ import annotations

import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '2')

import math
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import ledger as LG  # noqa: E402

# 열 이름 · 순서: DP:28-32 EDGE_FEATURES 앞 16개 + amt_z_pay_ccy(NBA:134-138). 두 *_currency_unseen 은 빠짐.
X17_NAMES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'sender_gap_minutes',
             'sender_count_1h', 'receiver_gap_minutes', 'receiver_count_1h', 'paid_currency_z',
             'pair_history_count', 'reverse_history_count', 'pair_age_at_day_start',
             'same_entity', 'same_known_country', 'both_countries_known', 'same_currency', 'is_self_transfer',
             'amt_z_pay_ccy']
assert len(X17_NAMES) == 17

SH = 1 << 20              # 키 = 무리 번호·2^20 + 분. 분은 2^20 − 61(약 728일) 미만이어야 함(아래 assert).
CHUNK = 4_000_000         # 원장을 훑는 묶음 크기(행). 묶음당 임시 메모리 약 0.1~0.2 GB.
BUDGET_ROWS = 60_000_000  # 한 번 훑기에 모을 보냄 + 받음 이력 키 상한(int64 → 약 0.5 GB). 넘으면 샤드로 나눔.

_DEG_CACHE: dict = {}     # 원장 경로 → (보낸 수, 받은 수). 같은 프로세스에서 여러 날을 돌릴 때 다시 안 셈.


# ───────────────────────── 파일 읽기(memmap 대신) ─────────────────────────
class _Reader:
    """원장 .npy 를 os.preadv 로 읽음. memmap 이 아니면(원장을 mmap=False 로 읽었을 때) 배열을 그대로 자름."""

    def __init__(self, arr):
        self.arr = arr; self.n = len(arr); self.fd = None
        self.dt = arr.dtype; self.tail = tuple(arr.shape[1:])
        self.row = arr.dtype.itemsize * int(np.prod(self.tail, dtype=np.int64))
        if isinstance(arr, np.memmap) and getattr(arr, 'filename', None) and arr.flags['C_CONTIGUOUS']:
            self.fd = os.open(arr.filename, os.O_RDONLY); self.off = int(arr.offset)

    def close(self):
        if self.fd is not None:
            os.close(self.fd); self.fd = None

    def _pread_into(self, out, c0):
        mv = memoryview(out).cast('B'); got = 0; need = len(mv); off = self.off + c0 * self.row
        while got < need:                                              # 짧게 읽히면 이어 읽음
            k = os.preadv(self.fd, [mv[got:]], off + got)
            if k <= 0:
                raise IOError(f'원장 파일 읽기 실패: {self.arr.filename} @ {off + got}')
            got += k
        return out

    def slice(self, c0, c1):
        c1 = min(c1, self.n)
        if self.fd is None:
            return np.asarray(self.arr[c0:c1])
        return self._pread_into(np.empty((c1 - c0,) + self.tail, self.dt), c0)

    def take(self, pos, block=1 << 20, dense=256):
        """오름차순 위치 pos 의 값. 한 구간(block 행)에 dense 개 이상이면 구간을 통째로, 아니면 한 칸씩 읽음."""
        pos = np.asarray(pos, np.int64)
        if self.fd is None:
            return np.asarray(self.arr[pos])
        out = np.empty((len(pos),) + self.tail, self.dt)
        if len(pos) == 0:
            return out
        assert (np.diff(pos) >= 0).all() and pos[0] >= 0 and pos[-1] < self.n
        b = pos // block
        cuts = np.flatnonzero(np.diff(b)) + 1
        for i0, i1 in zip(np.r_[0, cuts], np.r_[cuts, len(pos)]):
            p = pos[i0:i1]
            if i1 - i0 >= dense:
                c0 = int(b[i0]) * block; out[i0:i1] = self.slice(c0, c0 + block)[p - c0]
            else:
                for k, q in enumerate(p.tolist()):
                    self._pread_into(out[i0 + k:i0 + k + 1], q)
        return out


def _readers(L, names):
    return {k: _Reader(L[k]) for k in names}


# ───────────────────────── 작은 도구 ─────────────────────────
def _ln_int(a) -> np.ndarray:
    """정수 배열 → ln(float(a)) float64. 서로 다른 값만 모아 math.log 로 계산(libm log = DuckDB ln 의 std::log).
    인자는 1+개수(DP:301-302) 또는 2+분(DP:298, 303)이라 서로 다른 값이 적음."""
    a = np.asarray(a, np.int64)
    if a.size == 0:
        return np.zeros(0, np.float64)
    u, inv = np.unique(a, return_inverse=True)
    assert u[0] >= 1, 'ln 인자는 1 이상이어야 함'
    lu = np.fromiter((math.log(float(x)) for x in u), np.float64, len(u))
    return lu[inv.reshape(-1)]


def _degrees(L, chunk=CHUNK):
    """원장 전체(자르기 전)의 계좌별 보낸 수 · 받은 수. 샤드 나누기 · 버퍼 크기의 상한으로만 씀."""
    key = str(Path(L.path).resolve())
    if key not in _DEG_CACHE:
        na = len(L['acct_ent']); R = _readers(L, ('src', 'dst'))
        do = np.zeros(na, np.int64); di = np.zeros(na, np.int64)
        for c0 in range(0, L.n, chunk):
            do += np.bincount(R['src'].slice(c0, c0 + chunk), minlength=na)
            di += np.bincount(R['dst'].slice(c0, c0 + chunk), minlength=na)
        for r in R.values():
            r.close()
        _DEG_CACHE[key] = (do, di)
    return _DEG_CACHE[key]


def _locate(L, ue, chunk=CHUNK):
    """오름차순 고유 edge_id ue → 원장 행 위치(없으면 −1). 원장 eid 를 묶음씩 읽어 그 범위의 질의만 찾음."""
    pos = np.full(len(ue), -1, np.int64); r = _Reader(L['eid'])
    try:
        for c0 in range(0, L.n, chunk):
            e = r.slice(c0, c0 + chunk)
            i0 = np.searchsorted(ue, e[0], 'left'); i1 = np.searchsorted(ue, e[-1], 'right')
            if i1 > i0:
                q = ue[i0:i1]; j = np.searchsorted(e, q)
                hit = e[np.minimum(j, len(e) - 1)] == q
                pos[i0:i1][hit] = c0 + j[hit]
            if i1 >= len(ue):
                break
    finally:
        r.close()
    return pos


def _proc_mem():
    """현재 프로세스 RssAnon · RssFile · VmHWM(GiB)."""
    out = {}
    try:
        for l in open('/proc/self/status'):
            if l.startswith(('RssAnon', 'RssFile', 'VmHWM')):
                k, v = l.split(':'); out[k] = int(v.split()[0]) / 2 ** 20
    except OSError:
        pass
    return out


def _hist(keys, grp, t, d0=None):
    """정렬된 키(무리·SH + 분)에서 대상마다 이력 통계를 셈.
    grp = 대상의 무리 번호(계좌 또는 쌍), t = 대상 시각(분), d0 = 대상 날 00:00(분).
    돌려줌: n_prior = #{ts ≤ t−1} (DP:294 RANGE UNBOUNDED PRECEDING .. 1 minute PRECEDING),
            last = 그중 마지막 ts(없으면 −1) (DP:295 max(ts)),
            n1h = #{t−60 ≤ ts ≤ t−1} (DP:296 RANGE 60 minutes PRECEDING .. 1 minute PRECEDING),
            (d0 를 주면) n_day = #{ts < d0}, last_day = 그중 마지막 ts (DP:253-256 pair_daily_state + DP:262 ASOF t.day > f.day)."""
    n = len(grp)
    if keys.size == 0 or n == 0:
        z = np.zeros(n, np.int64); m1 = np.full(n, -1, np.int64)
        return (z, m1, z) if d0 is None else (z, m1, z, z, m1)
    base = grp.astype(np.int64) * SH
    lo = np.searchsorted(keys, base, 'left')                         # 이 무리의 첫 키
    hi = np.searchsorted(keys, base + t - 1, 'right')                # ts ≤ t−1 끝(t=0 이면 base−1 → lo)
    n_prior = hi - lo
    last = np.where(n_prior > 0, keys[np.maximum(hi - 1, 0)] - base, -1)
    l60 = np.searchsorted(keys, base + np.maximum(t - 60, 0), 'left')  # ts ≥ t−60 시작(음수는 0으로 — ts ≥ 0)
    n1h = hi - l60
    if d0 is None:
        return n_prior, last, n1h
    hd = np.searchsorted(keys, base + d0, 'left')                    # ts < d0 끝
    n_day = hd - lo
    last_day = np.where(n_day > 0, keys[np.maximum(hd - 1, 0)] - base, -1)
    return n_prior, last, n1h, n_day, last_day


def _plan_shards(w, budget):
    """계좌별 무게 w(모을 이력 행 수 상한)를 계좌 번호 순으로 이어 붙여 budget 이하 구간으로 자름 → [(a0, a1), ...].
    계좌 하나가 budget 보다 크면(허브) 그 계좌만으로 한 샤드가 됨."""
    na = len(w); total = int(w.sum())
    if total <= budget:
        return [(0, na)]
    ns = -(-total // budget)
    cw = np.cumsum(w)
    cuts = np.searchsorted(cw, (np.arange(1, ns) * total) // ns, 'right')
    b = np.unique(np.r_[0, cuts, na])
    return [(int(b[i]), int(b[i + 1])) for i in range(len(b) - 1) if b[i + 1] > b[i]]


# ───────────────────────── 본 계산 ─────────────────────────
def compute_x17(L, eids, day=None, budget_rows=BUDGET_ROWS, chunk=CHUNK, stats=None, log=None):
    """원장 L 의 거래 eids(edge_id = 원본 행 번호) 각각의 x17 → float32 [len(eids), 17] (eids 순서 그대로, 중복 허용).

    - day='YYYY-MM-DD' 를 주면 대상이 모두 그날 거래여야 함(아니면 ValueError — T5 '그날에 없는 edge_id → 거부').
    - day 가 없으면 원장에 있는 아무 거래나 받음(SALT/TGT 의 30일 창 이웃 거래처럼). 값은 각자 자기 시각 기준.
    - 원장에 없는 edge_id(중복으로 지운 행 · 자르는 시각 이후 행 · 범위 밖)는 ValueError.
    - test 봉인: 대상 날짜가 2022-10-15 이상이면 AML_TEST_UNSEAL=1 없이는 PermissionError(LG.check_seal).
    stats(dict)를 주면 시간 · 모은 행 수 · 샤드 수 · 메모리를 적어 줌."""
    T0 = time.time(); st = stats if stats is not None else {}
    say = log or (lambda s: None)
    q = np.asarray(eids, np.int64).reshape(-1)
    if day is not None:
        LG.check_seal(day, 'compute_x17 day')
    if len(q) == 0:
        return np.zeros((0, 17), np.float32)

    # 1) 대상 위치: 고유 edge_id 만 계산하고 끝에 질의 순서로 펼침. 원장 eid 는 엄격 오름차순(LEDGER_FORMAT 3.1).
    ue, back = np.unique(q, return_inverse=True); back = back.reshape(-1)
    ps = _locate(L, ue, chunk)
    if (ps < 0).any():
        bad = ue[ps < 0][:10].tolist()
        raise ValueError(f'원장에 없는 edge_id {int((ps < 0).sum())}개(중복 제거 · 자르는 시각 이후 · 범위 밖), 예: {bad}')
    n = len(ue)                                                      # 이하 대상 = 고유 edge_id(위치 오름차순)
    R = _readers(L, ('ts', 'src', 'dst', 'pcur', 'rcur', 'pz', 'az'))
    try:
        t = R['ts'].take(ps).astype(np.int64); s = R['src'].take(ps).astype(np.int64); d = R['dst'].take(ps).astype(np.int64)
        pcur = R['pcur'].take(ps); rcur = R['rcur'].take(ps); pz = R['pz'].take(ps); az = R['az'].take(ps)
    finally:
        for r in R.values():
            r.close()
    LG.check_seal(str(np.datetime64(LG.BASE_DATE) + np.timedelta64(int(t.max()) // 1440, 'D')), 'compute_x17 edges')
    if day is not None:
        dd = LG.minutes_of(day) // 1440; off = t // 1440 != dd
        if off.any():
            raise ValueError(f'{day} 날 거래가 아닌 edge_id {int(off.sum())}개, 예: {ue[off][:10].tolist()}')
    d0 = (t // 1440) * 1440                                          # 대상 날 00:00 (DP:260 t.day::TIMESTAMP)
    cut = int(t.max())                                               # 쓰는 이력: ts ≤ t−1 ≤ max(t)−1 → ts < cut
    assert cut + 61 < SH, '분이 키 자리(2^20)를 넘음'
    st['targets'] = len(q); st['unique_targets'] = n; st['cut_minute'] = cut
    st['t_locate'] = round(time.time() - T0, 2)

    # 2) 필요한 무리: 보낸 계좌 S · 받은 계좌 R · 쌍(s→d) · 역쌍(d→s). 쌍 코드 = src·NA + dst (int64 < 2^43).
    NA = len(L['acct_ent'])
    uS = np.unique(s); uR = np.unique(d)
    pc = s * NA + d; upc = np.unique(pc); pidx = np.searchsorted(upc, pc)     # 대상의 쌍 번호
    rc = d * NA + s; urc = np.unique(rc); ridx = np.searchsorted(urc, rc)     # 대상의 역쌍 번호(d→s)

    # 3) 샤드 계획: 계좌 무게 = (S 이면 보낸 수) + (R 이면 받은 수). 원장 전체 bincount(상한).
    t1 = time.time()
    deg_o, deg_i = _degrees(L, chunk)
    w = np.zeros(NA, np.int64); w[uS] += deg_o[uS]; w[uR] += deg_i[uR]
    shards = _plan_shards(w, budget_rows)
    st['t_degrees'] = round(time.time() - t1, 2); st['shards'] = len(shards); st['hist_rows_upper'] = int(w.sum())
    say(f'[x17] 대상 {n:,} · 계좌 S {len(uS):,} R {len(uR):,} · 쌍 {len(upc):,} · 이력 상한 {int(w.sum()):,}행 · 샤드 {len(shards)}')

    # 정수 통계 칸: p_* 쌍 · s_* 보냄 · q_* 받음 · r_day 역쌍
    p_n = np.zeros(n, np.int64); p_last = np.full(n, -1, np.int64); p_1h = np.zeros(n, np.int64)
    p_day = np.zeros(n, np.int64); p_lastday = np.full(n, -1, np.int64); r_day = np.zeros(n, np.int64)
    s_last = np.full(n, -1, np.int64); s_1h = np.zeros(n, np.int64)
    q_last = np.full(n, -1, np.int64); q_1h = np.zeros(n, np.int64)
    rows_got = dict(send=0, recv=0, pair=0, rev=0)
    t_scan = t_feat = 0.0
    R = _readers(L, ('ts', 'src', 'dst'))
    try:
        for a0, a1 in shards:
            t2 = time.time()
            inS = np.zeros(NA, bool); inR = np.zeros(NA, bool)
            kS = uS[(uS >= a0) & (uS < a1)]; kR = uR[(uR >= a0) & (uR < a1)]
            inS[kS] = True; inR[kR] = True
            # 보냄 · 받음 키는 상한 크기로 미리 잡아 채움(리스트 + concatenate 의 2배 꼭대기를 피함)
            bS = np.empty(int(deg_o[kS].sum()), np.int64); nS = 0
            bR = np.empty(int(deg_i[kR].sum()), np.int64); nR = 0
            lP, lV = [], []
            for c0 in range(0, L.n, chunk):
                ts_c = R['ts'].slice(c0, c0 + chunk); keep = ts_c < cut      # 가장 늦은 대상 시각 전 행만
                if not keep.any():
                    continue
                src_c = R['src'].slice(c0, c0 + chunk); dst_c = R['dst'].slice(c0, c0 + chunk)
                # (a) 보냄 이력: src ∈ S (DP:293 PARTITION BY src). 쌍 이력(PARTITION BY src,dst)은 이 부분집합.
                m = keep & inS[src_c]
                if m.any():
                    sv = src_c[m].astype(np.int64); tv = ts_c[m].astype(np.int64); dv = dst_c[m].astype(np.int64)
                    k = sv * SH + tv; bS[nS:nS + len(k)] = k; nS += len(k)
                    code = sv * NA + dv; j = np.searchsorted(upc, code); j[j == len(upc)] = 0
                    hit = upc[j] == code
                    if hit.any():
                        lP.append(j[hit].astype(np.int64) * SH + tv[hit])
                # (b) 받음 이력: dst ∈ R (DP:293 PARTITION BY dst)
                m = keep & inR[dst_c]
                if m.any():
                    k = dst_c[m].astype(np.int64) * SH + ts_c[m]; bR[nR:nR + len(k)] = k; nR += len(k)
                # (c) 역쌍 이력: src = d, dst = s 인 행 → dst ∈ S 인 행 중 코드가 역쌍에 있는 것
                #     (DP:263 ASOF LEFT JOIN r ON t.src=r.dst AND t.dst=r.src AND t.day>r.day)
                m = keep & inS[dst_c]
                if m.any():
                    code = src_c[m].astype(np.int64) * NA + dst_c[m]; j = np.searchsorted(urc, code); j[j == len(urc)] = 0
                    hit = urc[j] == code
                    if hit.any():
                        lV.append(j[hit].astype(np.int64) * SH + ts_c[m][hit])
            kSd = bS[:nS]; kSd.sort(); kRd = bR[:nR]; kRd.sort()
            kP = np.concatenate(lP) if lP else np.zeros(0, np.int64); del lP; kP.sort()
            kV = np.concatenate(lV) if lV else np.zeros(0, np.int64); del lV; kV.sort()
            rows_got['send'] += nS; rows_got['recv'] += nR; rows_got['pair'] += len(kP); rows_got['rev'] += len(kV)
            t3 = time.time(); t_scan += t3 - t2

            # 4) 이 샤드에 속한 대상의 통계 (보냄 · 쌍 · 역쌍은 s 의 샤드, 받음은 d 의 샤드)
            ii = np.flatnonzero((s >= a0) & (s < a1))
            if len(ii):
                _, s_last[ii], s_1h[ii] = _hist(kSd, s[ii], t[ii])
                p_n[ii], p_last[ii], p_1h[ii], p_day[ii], p_lastday[ii] = _hist(kP, pidx[ii], t[ii], d0[ii])
                r_day[ii] = _hist(kV, ridx[ii], t[ii], d0[ii])[3]
            ii = np.flatnonzero((d >= a0) & (d < a1))
            if len(ii):
                _, q_last[ii], q_1h[ii] = _hist(kRd, d[ii], t[ii])
            t_feat += time.time() - t3
            st.setdefault('mem_after_shard', []).append(_proc_mem())
            del bS, bR, kSd, kRd, kP, kV, inS, inR
    finally:
        for r in R.values():
            r.close()

    # 5) 열 조립: float64 로 계산 → float32 (DP:309 cast(... AS FLOAT))
    t4 = time.time()
    X = np.zeros((n, 17), np.float64)
    gap = lambda last, ref: np.where(last >= 0, _ln_int(np.where(last >= 0, 2 + ref - last, 1)), 0.0)  # ln(2+분), 없으면 0
    X[:, 0] = _ln_int(1 + p_n)                                       # DP:301 ln(1+pair_prior_count)
    X[:, 1] = gap(p_last, t)                                         # DP:298 pair_gap_minutes
    X[:, 2] = _ln_int(1 + p_1h)                                      # DP:301 ln(1+pair_count_1h)
    X[:, 3] = gap(s_last, t)                                         # DP:298 sender_gap_minutes
    X[:, 4] = _ln_int(1 + s_1h)                                      # DP:301 sender_count_1h
    X[:, 5] = gap(q_last, t)                                         # DP:298 receiver_gap_minutes
    X[:, 6] = _ln_int(1 + q_1h)                                      # DP:301 receiver_count_1h
    X[:, 8] = _ln_int(1 + p_day)                                     # DP:302 ln(1+pair_history_count)
    X[:, 9] = _ln_int(1 + r_day)                                     # DP:302 ln(1+reverse_history_count)
    X[:, 10] = gap(p_lastday, d0)                                    # DP:260, 303 pair_age_at_day_start(없으면 −1 → 0)
    # KYC · 통화 · 자기 송금 (DP:304-307). acct_ent −1 = KYC 없음(entity_id NULL), acct_country −1 = 'UNKNOWN'.
    ent = np.asarray(L['acct_ent']); cty = np.asarray(L['acct_country'])
    es, ed, cs, cd = ent[s], ent[d], cty[s], cty[d]
    X[:, 11] = (es >= 0) & (es == ed)                                # DP:304 sk.entity_id IS NOT NULL AND sk=dk
    kn = (cs >= 0) & (cd >= 0)
    X[:, 12] = kn & (cs == cd)                                       # DP:305 same_known_country
    X[:, 13] = kn                                                    # DP:306 both_countries_known
    X[:, 14] = pcur == rcur                                          # DP:307 payment_currency = receiving_currency
    X[:, 15] = s == d                                                # DP:307 t.src = t.dst
    X32 = X.astype(np.float32)
    X32[:, 7] = pz                                                   # DP:299 paid_currency_z (원장에 이미 float32)
    X32[:, 16] = az                                                  # NBA:134-138 amt_z_pay_ccy (원장에 이미 float32)
    if not np.isfinite(X32).all():                                   # DP:315-317 비유한 값이면 멈춤(자동 대체 없음)
        raise ValueError('x17 에 NaN/Inf 가 있음')
    st['rows_collected'] = rows_got
    st['t_scan'] = round(t_scan, 2); st['t_feat'] = round(t_feat + time.time() - t4, 2)
    st['t_total'] = round(time.time() - T0, 2); st['mem_end'] = _proc_mem()
    say(f'[x17] 끝 {st["t_total"]}초 (훑기 {st["t_scan"]} · 계산 {st["t_feat"]}) · 모은 이력 {rows_got}')
    return X32[back]


def x17_day(L, day, eids=None, **kw):
    """그날 D 의 대상 거래 x17. eids 가 없으면 그날 패턴 행 전부(pat_eid, edge_id 오름차순 = val55 순서)."""
    LG.check_seal(day, 'x17_day')
    if eids is None:
        eids = np.asarray(L['pat_eid'])[np.asarray(L['pat_day']) == LG.minutes_of(day) // 1440]
    return compute_x17(L, eids, day=day, **kw)


def x17_table(L, eids, **kw):
    """여러 소비자(55열 · SALT ea/tx · TGT x/tx)가 함께 쓰도록 고유 edge_id 오름차순 표로 계산 → (eid[u], X[u, 17])."""
    u = np.unique(np.asarray(eids, np.int64))
    return u, compute_x17(L, u, **kw)


def x17_take(table, q):
    """x17_table 결과에서 edge_id q 의 행을 꺼냄(없으면 KeyError)."""
    u, X = table; q = np.asarray(q, np.int64)
    j = np.searchsorted(u, q); j = np.minimum(j, len(u) - 1)
    if not (u[j] == q).all():
        raise KeyError(f'x17 표에 없는 edge_id {int((u[j] != q).sum())}개')
    return X[j]
'''

In [ ]:
# @title 코드 ③ 계좌 19열(node19.py): D 마감 30일 창 · 창 읽기 — 숨김 · 고치지 마세요
_MODULE_SRC['node19'] = r'''"""node19 — D일 마감 30일 창 [D−29, D] 의 계좌 19열(압축 원장만으로, DuckDB 없이).

쓰는 곳: LightGBM 55열의 0:19(src) · 19:38(dst), SALT C · TC B 묶음 캐시 `nx`, TGT B0 캐시 `nu` · `nv`.
입력: ledger.load_ledger(...) 가 돌려준 Ledger(형식은 LEDGER_FORMAT.md). numpy 만 씀.

원래 코드(읽기 전용) 약칭:
  (경로는 팀 작업 공간의 Final_preprocess/ 부터)
  DP = Final_preprocess/태훈님_전처리/feature_code_40_20260921/CODE/daily_pipeline.py
  T9 = Final_preprocess/HI-Large_EDA/tree9_lib.py

원래 규칙(DP:343-389 load_day 의 노드 부분, T9:178-193 node_values 가 글자 그대로 씀)
  - 창: hist = f.day ≥ D−29 AND f.day ≤ D (DP:349 start=day−(history_days−1), DP:353-356). history_days = 30
    (split_60_20_20/plan.json data_config). D 하루 전체가 들어감 — 대상 거래 뒤의 같은 날 거래도 포함(일 마감).
  - 행 = 중복을 지운 거래 전부(features 표 = tx 표, DP:313-319). 원장 행과 같음.
  - 결과 행 = 창에 src 나 dst 로 나온 계좌만(active_days JOIN kyc, DP:377). 그 밖 계좌는 값이 없음
    → 여기서는 NaN 행으로 돌려줌(T9:244-245 join_day 가 못 찾은 노드를 NaN 으로 두는 것과 같은 뜻).
  - 값: float64 로 계산 → np.float32(DP:379-389). 19열 = NODE_FEATURES(DP:33-37) 23열 중
    [0,1,2,3,4,5,8,10,12,13,14..22] (net_count_ratio · reciprocity_ratio · mean_received_currency_z ·
    received_outlier_fraction 4열은 55열에서 빠짐, tg_data.py:17-22 · aml_feature_schema.csv).

19열 (열 번호 = 이 모듈 출력 순서)
   0 log_out_count            ln1p(#창 거래 src=a)                                       DP:358 out_n, 384
   1 log_in_count             ln1p(#창 거래 dst=a)                                       DP:361 in_n
   2 log_out_neighbors        ln1p(count DISTINCT dst | src=a) — 자기 송금이면 a 자신 포함   DP:358 out_u
   3 log_in_neighbors         ln1p(count DISTINCT src | dst=a)                           DP:361 in_u
   4 log_self_count           ln1p(#src=dst=a)                                           DP:359 self_n
   5 log_reciprocal_neighbors ln1p(#{b≠a : a→b, b→a 둘 다 창에 있음})                     DP:363-364
   6 mean_paid_currency_z     avg(FLOAT paid_currency_z | src=a), 없으면 0                DP:359 out_z, 374 coalesce
   7 paid_outlier_fraction    avg(|pz| ≥ 3 ? 1 : 0 | src=a), 없으면 0                     DP:360 out_tail
   8 log_currency_count       ln1p(|{src=a 의 지불 통화} ∪ {dst=a 의 받는 통화}|)           DP:365-366
   9 log_active_days          ln1p(#{a 가 src 나 dst 로 나온 날})                          DP:367-368
  10-15 type_*               entity_type == ENTITY_TYPES[k] (UNKNOWN 은 모두 0)            DP:387
  16 log_entity_observed_accounts ln1p(#창 활동 계좌 중 같은 entity_id), entity 없으면 0  DP:372
  17 log_entity_out_diversity ln1p(#DISTINCT target_entity | source_entity = E(a))          DP:369-371 ent_out
  18 log_entity_in_diversity  ln1p(#DISTINCT source_entity | target_entity = E(a))          DP:369-371 ent_in
      (entity 쌍은 두 쪽 entity 가 모두 있는 창 거래 전체에서 DISTINCT, 같은 entity 쌍 포함. 계좌 a 가
       그 쌍에 직접 없어도 같은 entity 면 같은 값. entity 없으면 0 — DP:378-379 LEFT JOIN + coalesce)

계산 방법(메모리 · 시간)
  - 원장은 시간순이 아니므로 ts 를 묶음(기본 8M행)으로 훑어 창 행만 고름. memmap 대신 파일에서 바로 읽어
    (np.fromfile) 프로세스 RSS 에 파일 페이지를 남기지 않음.
  - 행 단위 집계는 묶음마다 바로 누적: 개수 = bincount, 통화 · 날 = 비트맵(계좌 × 15통화, 계좌 × 30일),
    pz 합 = np.add.at(행 순서대로 차례로 더함 → 묶음 크기와 무관하게 같은 값 = 창 전체 bincount 와 같음).
  - DISTINCT 계좌 쌍 = (src·n + dst) int64 정렬 + 이웃 비교(np.unique 를 쓰지 않음 — 메모리 3.6배).
    reciprocal = (min, max) 키를 다시 정렬해 두 번 나온 쌍. entity 쌍 = DISTINCT 계좌 쌍 → entity 쌍 정렬 + 비교.
  - 10-06 기준 창 55.0M행: 쌍 키 440 MB 가 가장 큼. 실측(2026-10-17, nice 19 · CPU · 2스레드, test_node19.py):
    하루 5.0~11.5초(창 훑기 6~7초 · 쌍 1.8초 · entity 0.3초 · 표 1.5초), 프로세스 VmHWM 1.18~1.32 GiB.

검증(test_node19.py, logs/node19_summary.json) — 모두 비트 단위로 같음(최대 |차| 0.0)
  - 합성 원장 32경우: 원래 SQL(DP load_day 노드 부분을 소스 글자 그대로 DuckDB 에서 실행)과 19열 · 활동 계좌 집합.
  - val 17일: val_pattern55 X[:,0:19] · X[:,19:38](24,280행 × 2), SALT 캐시 nx 444,528행(묶음 그래프를 원장에서
    다시 지어 ei · tg · eid · grp 까지 같음을 확인한 뒤 노드 대응).
  - val 17일 + train 4일(08-08 · 08-20 · 09-07 · 09-27): 원래 tgnn36 하루 캐시(DP 로 지은 day=D/node_x · 만들 때 임시 폴더에 있던 것)의 창 활동 계좌 전부
    (21일 40,678,254행 × 19열), 활동 계좌 집합도 같음.

소수 끝자리
  - DuckDB avg(FLOAT)는 double 합 / 개수(병렬 순서). 여기는 edge_id 오름차순으로 차례로 더한 double 합 / 개수.
    합의 순서가 달라 double 끝자리가 다를 수 있으나 float32 로 자르면 거의 사라짐(비교는 1e-6 허용, 실측 보고).
  - ln1p 는 원래처럼 numpy np.log1p(float64) → float32.
"""
from __future__ import annotations

import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '2')

import resource
import time

import numpy as np

try:                                   # 같은 폴더의 ledger.py(봉인 · 날짜 도구). 노트북에서는 같은 셀 안에 있음.
    import ledger as _LG
    check_seal, minutes_of = _LG.check_seal, _LG.minutes_of
    BASE_DATE, ENTITY_TYPES = _LG.BASE_DATE, _LG.ENTITY_TYPES
except ImportError:                    # pragma: no cover
    _LG = None
    BASE_DATE = '2022-08-01'
    ENTITY_TYPES = ['Corporation', 'Partnership', 'Sole Proprietorship', 'Individual', 'Country', 'Direct']  # DP:27

    def minutes_of(date_str):
        return int((np.datetime64(date_str) - np.datetime64(BASE_DATE)).astype('timedelta64[m]').astype(np.int64))

    def check_seal(date_str, what):
        if str(date_str)[:10] >= '2022-10-15' and os.environ.get('AML_TEST_UNSEAL') != '1':
            raise PermissionError(f'[test 봉인] {what}({str(date_str)[:10]})')

HISTORY_DAYS = 30                      # plan.json data_config.history_days (DP:49 기본값도 30)
TRAIN_START = '2022-08-08'             # DP:350 — 보정 기간(08-01~08-07)에는 예측하지 않음
DAY_MIN = 1440

# DP:33-37 NODE_FEATURES 23열 → 55열에 쓰는 19열(이름은 T9:47 처럼 공백 → _)
NODE23 = ['log_out_count', 'log_in_count', 'log_out_neighbors', 'log_in_neighbors', 'log_self_count',
          'log_reciprocal_neighbors', 'net_count_ratio', 'reciprocity_ratio', 'mean_paid_currency_z',
          'mean_received_currency_z', 'paid_outlier_fraction', 'received_outlier_fraction', 'log_currency_count',
          'log_active_days'] + ['type_' + x for x in ENTITY_TYPES] + [
          'log_entity_observed_accounts', 'log_entity_out_diversity', 'log_entity_in_diversity']
KEEP19 = [0, 1, 2, 3, 4, 5, 8, 10, 12, 13] + list(range(14, 23))
NODE19_NAMES = [NODE23[i].replace('type_Sole Proprietorship', 'type_Sole_Proprietorship') for i in KEEP19]
assert len(NODE19_NAMES) == 19


# ───────────────────────── 원장 열 읽기 ─────────────────────────
class _ColReader:
    """원장 열 하나의 [lo, hi) 행을 읽음. memmap 이면 파일에서 직접(np.fromfile), 아니면 메모리 배열을 자름(합성 시험)."""

    def __init__(self, arr):
        self.arr = arr; self.f = None
        fn = getattr(arr, 'filename', None)
        if isinstance(arr, np.memmap) and fn is not None and arr.flags.c_contiguous:
            self.f = open(fn, 'rb'); self.off = int(arr.offset); self.dt = arr.dtype

    def __call__(self, lo, hi):
        if self.f is None:
            return np.array(self.arr[lo:hi])
        self.f.seek(self.off + lo * self.dt.itemsize)
        a = np.fromfile(self.f, self.dt, hi - lo)
        if len(a) != hi - lo:
            raise IOError(f'원장 열을 끝까지 읽지 못함: {self.f.name} [{lo}, {hi})')
        return a

    def close(self):
        if self.f is not None:
            self.f.close(); self.f = None


def _check_day(L, D, what):
    """봉인 · 기간 확인. D 는 'YYYY-MM-DD'."""
    D = str(D)[:10]
    check_seal(D, what)                                       # T9:70-80 과 같은 뜻 — 2022-10-15 이후는 거부
    if D < TRAIN_START:                                       # DP:350
        raise ValueError(f'{what}({D}): 보정 기간에는 예측하지 않음(DP:350, {TRAIN_START} 이후만)')
    if D > str(L.meta['d_max'])[:10]:
        raise ValueError(f'{what}({D}): 원장 d_max {L.meta["d_max"]} 보다 늦은 날 — 원장을 더 늦게까지 지어야 함')
    return D


def window_bounds(D):
    """D 의 30일 창 [lo_ts, hi_ts) (분). DP:349 start = D − 29일, DP:356 f.day ≤ D."""
    d0 = minutes_of(D)
    return d0 - (HISTORY_DAYS - 1) * DAY_MIN, d0 + DAY_MIN


def window_count(L, D):
    """meta.day_counts 로 창 행 수(미리 자리 잡기용). 없으면 None."""
    dc = L.meta.get('day_counts')
    if not dc:
        return None
    days = np.datetime64(D) - np.arange(HISTORY_DAYS)
    return int(sum(int(dc.get(str(d), 0)) for d in days))


def iter_window(L, D, cols, chunk_rows=8_000_000, with_pos=False):
    """원장을 행 순서(eid 오름차순)대로 묶음 단위로 훑어 창 [D−29, D] 행만 돌려줌. dict(열 → 배열) 를 차례로 yield.
    ts 를 먼저 보고 창 밖 행은 다른 열을 쓰기 전에 버림."""
    lo_t, hi_t = window_bounds(D)
    n = L.n
    cols = [c for c in cols if c != 'ts']
    rts = _ColReader(L['ts']); rd = {c: _ColReader(L[c]) for c in cols}
    try:
        for s in range(0, n, chunk_rows):
            e = min(n, s + chunk_rows)
            t = rts(s, e)
            m = (t >= lo_t) & (t < hi_t)
            if not m.any():
                continue
            idx = np.flatnonzero(m)
            out = {'ts': t[idx]}
            for c in cols:
                out[c] = rd[c](s, e)[idx]
            if with_pos:
                out['pos'] = (idx + s).astype(np.int64)
            yield out
    finally:
        rts.close()
        for r in rd.values():
            r.close()


def read_window(L, D, cols=('src', 'dst', 'ts', 'eid'), chunk_rows=8_000_000, with_pos=False):
    """창 [D−29, D] 의 원장 행을 열별 배열로 모음(eid 오름차순). SALT 목록 · TGT 시퀀스 · node19(W=) 가 함께 쓸 수 있음.
    10-06 기준 55.0M행 × (4 B × 열 수). meta.day_counts 로 자리를 미리 잡고 끝에 행 수를 확인함."""
    D = _check_day(L, D, 'read_window')
    want = list(dict.fromkeys(list(cols) + ['ts']))
    k = window_count(L, D)
    if k is None:                                              # day_counts 가 없으면 한 번 세고 다시 훑음
        k = sum(len(b['ts']) for b in iter_window(L, D, [], chunk_rows))
    W = {c: np.empty(k, L[c].dtype) for c in want}
    if with_pos:
        W['pos'] = np.empty(k, np.int64)
    f = 0
    for b in iter_window(L, D, want, chunk_rows, with_pos):
        m = len(b['ts'])
        if f + m > k:
            raise ValueError(f'창 행 수가 meta.day_counts 합({k:,})보다 많음')
        for c, v in b.items():
            W[c][f:f + m] = v
        f += m
    if f != k:
        raise ValueError(f'창 행 수 {f:,} ≠ meta.day_counts 합 {k:,}')
    W['_day'] = D
    return W


# ───────────────────────── node19 ─────────────────────────
class Node19Day:
    """D 하루의 계좌 19열 표. X[a] = 계좌 a 의 19열(float32), 창에 없는 계좌는 NaN 행. active[a] = 창에 나옴."""

    def __init__(self, day, X, active, stats):
        self.day, self.X, self.active, self.stats = day, X, active, stats

    def take(self, accounts, strict=False):
        """계좌 번호 배열 → [len, 19] float32. strict=True 면 창에 없는 계좌가 있을 때 멈춤."""
        a = np.asarray(accounts, np.int64)
        if len(a) and (a.min() < 0 or a.max() >= len(self.active)):
            raise IndexError('계좌 번호가 원장 계좌표 범위 밖')
        miss = ~self.active[a]
        if strict and miss.any():
            raise ValueError(f'{self.day}: 창에 없는 계좌 {int(miss.sum())}개(원래 load_day 결과에 행이 없음, DP:377)')
        return self.X[a]

    def __repr__(self):
        return f'Node19Day({self.day}, active={int(self.active.sum()):,})'


def _sorted_distinct(k):
    """int64 배열을 제자리 정렬한 뒤 서로 다른 값만 돌려줌(np.unique 대신 — 메모리 절약)."""
    if len(k) == 0:
        return k
    k.sort()
    keep = np.empty(len(k), bool); keep[0] = True
    np.not_equal(k[1:], k[:-1], out=keep[1:])
    return k[keep]


def node19_day(L, D, W=None, chunk_rows=8_000_000, log=None) -> Node19Day:
    """원장 L 에서 D 일 마감 계좌 19열을 모든 계좌에 대해 계산(창 밖 계좌는 NaN).
    W: read_window(L, D, cols ⊇ src,dst,ts,pz,pcur,rcur) 결과를 주면 원장을 다시 읽지 않음(결과는 비트 단위로 같음)."""
    D = _check_day(L, D, 'node19')
    t_start = time.time(); tm = {}
    n = int(len(L['acct_ent']))                                   # 계좌 수(계좌표 + 계좌표 밖)
    ent = np.asarray(L['acct_ent']).astype(np.int64)              # −1 = KYC 없음(DP:188-192 LEFT JOIN 의 NULL)
    etype = np.asarray(L['acct_etype'])                           # 0..5 = ENTITY_TYPES, −1 = UNKNOWN
    n_ent = int(len(L['ent_id'])) if 'ent_id' in L else int(ent.max()) + 1
    C = int(len(L.meta['vocab']['currency']))                     # 지불 · 받는 통화가 같은 vocab(LEDGER_FORMAT 3.1)
    lo_t, _ = window_bounds(D); day_lo = lo_t // DAY_MIN

    # 행 단위 누적(묶음마다)
    out_n = np.zeros(n, np.int64); in_n = np.zeros(n, np.int64); self_n = np.zeros(n, np.int64)
    tail_n = np.zeros(n, np.int64); sum_pz = np.zeros(n, np.float64)
    cur_mark = np.zeros(n * C, bool)                              # (계좌, 통화) 있음 — DP:365-366 UNION
    day_mark = np.zeros(n * HISTORY_DAYS, bool)                   # (계좌, 날) 있음 — DP:367-368 UNION
    k_need = len(W['ts']) if W is not None else window_count(L, D)
    if k_need is None:
        k_need = sum(len(b['ts']) for b in iter_window(L, D, [], chunk_rows))
    pkey = np.empty(k_need, np.int64)                             # 계좌 쌍 키 src·n + dst(자기 송금 포함)
    fill = 0

    def add(src, dst, ts, pz, pc, rc):
        nonlocal fill
        s = src.astype(np.int64); d = dst.astype(np.int64)
        out_n[:] += np.bincount(s, minlength=n)                   # DP:358 count(*) GROUP BY src
        in_n[:] += np.bincount(d, minlength=n)                    # DP:361 count(*) GROUP BY dst
        self_n[:] += np.bincount(s[s == d], minlength=n)          # DP:359 FILTER(WHERE src=dst)
        np.add.at(sum_pz, s, pz.astype(np.float64))               # DP:359 avg(paid_currency_z): FLOAT → double 합
        tail_n[:] += np.bincount(s[np.abs(pz) >= 3], minlength=n)  # DP:360 abs(FLOAT pz) ≥ 3
        if len(pc) and (int(pc.max()) >= C or int(rc.max()) >= C or int(pc.min()) < 0 or int(rc.min()) < 0):
            raise ValueError('통화 코드가 meta.vocab.currency 범위 밖')
        cur_mark[s * C + pc] = True                               # (src, 지불 통화)
        cur_mark[d * C + rc] = True                               # (dst, 받는 통화)
        dd = ts.astype(np.int64) // DAY_MIN - day_lo              # 창 안 날 번호 0..29
        day_mark[s * HISTORY_DAYS + dd] = True
        day_mark[d * HISTORY_DAYS + dd] = True
        k = len(s)
        if fill + k > len(pkey):
            raise ValueError('창 행 수가 예상보다 많음')
        pkey[fill:fill + k] = s * n + d
        fill += k

    t0 = time.time()
    need = ('src', 'dst', 'ts', 'pz', 'pcur', 'rcur')
    if W is None:
        for b in iter_window(L, D, need, chunk_rows):
            add(b['src'], b['dst'], b['ts'], b['pz'], b['pcur'], b['rcur'])
    else:
        if W.get('_day') not in (None, D):
            raise ValueError(f'W 는 {W.get("_day")} 창 — {D} 와 다름')
        miss = [c for c in need if c not in W]
        if miss:
            raise ValueError(f'W 에 열이 없음: {miss}')
        for s0 in range(0, len(W['ts']), chunk_rows):
            sl = slice(s0, s0 + chunk_rows)
            add(W['src'][sl], W['dst'][sl], W['ts'][sl], W['pz'][sl], W['pcur'][sl], W['rcur'][sl])
    if fill != len(pkey):
        raise ValueError(f'창 행 수 {fill:,} ≠ 예상 {len(pkey):,}')
    tm['scan_s'] = round(time.time() - t0, 2)
    n_rows = fill

    # DISTINCT 계좌 쌍(DP:358 count DISTINCT dst · DP:361 count DISTINCT src · DP:363 pairs · DP:369 ent_pairs 의 재료)
    t0 = time.time()
    U = _sorted_distinct(pkey); del pkey
    ps = U // n; pd_ = U - ps * n; del U
    out_u = np.bincount(ps, minlength=n)                          # DISTINCT dst per src(자기 자신 포함)
    in_u = np.bincount(pd_, minlength=n)                          # DISTINCT src per dst
    n_pairs = int(len(ps))
    # reciprocal(DP:363-364): src≠dst 인 DISTINCT 쌍 (a,b) 중 (b,a) 도 있는 것 → a 에 1. 무순서 키가 두 번 나오면 양쪽 모두 +1.
    m = ps != pd_
    a_ = np.minimum(ps[m], pd_[m]); b_ = np.maximum(ps[m], pd_[m])
    rk = a_ * n + b_; del a_, b_, m
    rk.sort()
    both = rk[1:][rk[1:] == rk[:-1]]; del rk
    rec = np.bincount(both // n, minlength=n) + np.bincount(both % n, minlength=n)
    n_recip_pairs = int(len(both)); del both
    tm['pairs_s'] = round(time.time() - t0, 2)

    # entity 쌍(DP:369-371): 두 쪽 entity 가 모두 있는 창 거래의 DISTINCT (source_entity, target_entity)
    t0 = time.time()
    es = ent[ps]; ed = ent[pd_]; del ps, pd_
    ok = (es >= 0) & (ed >= 0)
    ek = es[ok] * n_ent + ed[ok]; del es, ed, ok
    EU = _sorted_distinct(ek); del ek
    ent_out = np.bincount(EU // n_ent, minlength=n_ent)           # DP:370 GROUP BY source_entity
    ent_in = np.bincount(EU % n_ent, minlength=n_ent)             # DP:371 GROUP BY target_entity
    n_ent_pairs = int(len(EU)); del EU
    tm['entity_s'] = round(time.time() - t0, 2)

    # 결과 표(DP:372-389)
    t0 = time.time()
    active = (out_n + in_n) > 0                                   # DP:367-368 active_days 에 나온 계좌
    has_ent = ent >= 0
    obs = np.bincount(ent[active & has_ent], minlength=n_ent)     # DP:372 count(*) OVER(PARTITION BY entity_id)
    ent_c = np.where(has_ent, ent, 0)
    obs_a = np.where(has_ent, obs[ent_c], 0)                      # CASE WHEN entity_id IS NULL THEN 0
    eo_a = np.where(has_ent, ent_out[ent_c], 0)                   # coalesce(eo.ent_out_n, 0)
    ei_a = np.where(has_ent, ent_in[ent_c], 0)
    n_cur = cur_mark.reshape(n, C).sum(1); del cur_mark
    n_days = day_mark.reshape(n, HISTORY_DAYS).sum(1); del day_mark
    on = out_n.astype(np.float64)
    with np.errstate(invalid='ignore', divide='ignore'):
        out_z = np.where(out_n > 0, sum_pz / on, 0.0)             # coalesce(o.out_z, 0)
        out_tail = np.where(out_n > 0, tail_n / on, 0.0)          # coalesce(o.out_tail, 0)
    f64 = lambda x: np.asarray(x, np.float64)
    fields = [np.log1p(f64(out_n)), np.log1p(f64(in_n)), np.log1p(f64(out_u)), np.log1p(f64(in_u)),
              np.log1p(f64(self_n)), np.log1p(f64(rec)), out_z, out_tail,
              np.log1p(f64(n_cur)), np.log1p(f64(n_days))]
    fields += [(etype == k).astype(np.float64) for k in range(len(ENTITY_TYPES))]   # DP:387 원핫, UNKNOWN(−1)·기타 → 0
    fields += [np.log1p(f64(obs_a)), np.log1p(f64(eo_a)), np.log1p(f64(ei_a))]
    X = np.stack(fields, axis=1).astype(np.float32)               # DP:389 .astype(np.float32)
    del fields
    if not np.isfinite(X[active]).all():                          # DP:389 finite('daily node features', …)
        raise ValueError('node19: NaN/Inf 값')
    X[~active] = np.nan
    tm['assemble_s'] = round(time.time() - t0, 2)
    tm['total_s'] = round(time.time() - t_start, 2)
    stats = dict(day=D, window_rows=n_rows, active_accounts=int(active.sum()), distinct_pairs=n_pairs,
                 reciprocal_pairs=n_recip_pairs, entity_pairs=n_ent_pairs, timings=tm,
                 maxrss_gib=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20, 3),
                 source='W' if W is not None else 'ledger')
    if log:
        log(f'[node19] {D}: 창 {n_rows:,}행 · 활동 계좌 {stats["active_accounts"]:,} · 쌍 {n_pairs:,} · '
            f'entity 쌍 {n_ent_pairs:,} · {tm}')
    return Node19Day(D, X, active, stats)


def node19(L, D, accounts, W=None, strict=False, **kw):
    """계좌 번호 배열 accounts 의 D 일 19열 [len, 19] float32(창에 없는 계좌는 NaN, strict=True 면 멈춤)."""
    return node19_day(L, D, W=W, **kw).take(accounts, strict=strict)


def target_accounts(L, eids):
    """edge_id 배열 → (src, dst) 계좌 번호. 원장에 없는 edge_id 면 멈춤(T5: 그날에 없는 edge_id 거부)."""
    e = np.asarray(eids, np.int64)
    eid = L['eid']
    if len(e) and (e.min() < 0 or e.max() > np.iinfo(eid.dtype).max):
        raise ValueError('edge_id 범위 밖')
    p = np.searchsorted(eid, e.astype(eid.dtype))                # 같은 dtype 으로 찾아야 memmap 전체를 int64 로 복사하지 않음
    p = np.minimum(p, len(eid) - 1)
    ok = np.asarray(eid[p]).astype(np.int64) == e
    if not ok.all():
        raise ValueError(f'원장에 없는 edge_id {int((~ok).sum())}개(중복으로 지웠거나 자르는 시각 이후): {e[~ok][:5].tolist()}')
    return np.asarray(L['src'][p]).astype(np.int64), np.asarray(L['dst'][p]).astype(np.int64), p
'''

In [ ]:
# @title 코드 ④ SALT-GNN C · TransformerConv B 하루 묶음 캐시(salt_inputs.py) — 숨김 · 고치지 마세요
_MODULE_SRC['salt_inputs'] = r'''"""salt_inputs — SALT C · TC B 하루 묶음 캐시(nx, ei, ea, tg, tx, y, eid, grp)를 압축 원장에서 다시 만든다.

쓰는 곳: SALT-GNN C · TransformerConv B 두 모델이 같은 하루 캐시를 읽음(aml8.py:1267-1291 predict_salt, Store aml8.py:327-356).
입력: ledger.load_ledger(...) 의 Ledger, node19.py(계좌 19열), x17.py(거래 17열). numpy 만 씀(DuckDB · PyG 없음).

원래 코드(읽기 전용) 약칭 — 경로는 팀 작업 공간의 Final_preprocess/ 부터
  DP   = Final_preprocess/태훈님_전처리/feature_code_40_20260921/CODE/daily_pipeline.py
  C1   = Final_preprocess/HI-Large_EDA/Transformer모델/SALT_GNN/개선_유형8_20261013/c1_cache.py
  CM   = 같은 폴더 common.py
  NB   = Final_preprocess/HI-Large_EDA/tgnn36_20260928/개선실험_보정_어려운정상_이웃_20261001/
         TGNN9클래스_FraudGT판_이웃확대_SALT_20261006.ipynb  (셀 번호 · 셀 안 줄 번호)
  K100 = Final_preprocess/HI-Large_EDA/이웃100_20261013/k100_ctx.py
  A8   = Final_preprocess/HI-Large_EDA/모델묶음_유형8_최종후보_20261016/aml8.py

원래 규칙과 여기서의 대응
  1. 대상 거래 = 그날 D 의 패턴 행(y 0~7), edge_id 오름차순.
     - 원래: 하루 대상 target_ids = 그날 거래 전부 ORDER BY edge_id (DP:395-397), 그중 y9 ∈ 0..7 의 자리(C1:33).
     - 여기: 원장 pat_eid[pat_day == D] (이미 edge_id 오름차순). 순서가 같으므로 묶음도 같음.
  2. 묶음 = default_rng([20261013, dayofyear(D)]).permutation(대상 수) 로 섞어 128개씩 (C1:32-35, GROUP=128 CM:36).
  3. 받은 목록 L(a) = 창 [D−29, D] 거래 중 dst = a 인 것(받음 + 자기 송금), ts DESC · edge_id DESC (DP:392-394).
     - 하루 문맥(context)은 계좌마다 L(a) 의 앞 K 개, dst 오름차순으로 이어 붙인 것(DP:393-394 QUALIFY · ORDER BY).
     - K: 그날 씨앗 계좌(패턴 행의 양 끝)는 100(K100:1-9, 67-83), 그 밖은 64(nb_cache_ids). 2홉은 앞 10개만 쓰므로
       여기서는 '씨앗 100 · 그 밖 10' 으로 만든다 — 샘플러가 실제로 읽는 칸이 원래와 하나도 다르지 않음.
  4. 묶음 그래프(NB 셀 35:138-159 CappedNeighborSampler.batch, k1=100 · k2=10 = CM:27-32, 83-84 FGT_NB_CAPS=100,10, C1:43):
     - seeds = unique(대상 src ∪ dst)
     - first = 씨앗마다 문맥 목록 앞 100 · second = unique(first 의 src) 마다 문맥 목록 앞 10
     - ci = union1d(first, second)  → 문맥 자리 오름차순 = (dst ASC, ts DESC, edge_id DESC) 순서
     - nodes = unique(seeds ∪ src[ci] ∪ dst[ci]) — 계좌 번호 순서. 원장 acct id 순서 = 원래 node_id 순서
       (LEDGER_FORMAT 2.3 · node19 시험에서 2,109,728 계좌 일대일 · 순서 보존 확인).
     - ei = (searchsorted(nodes, src[ci]), searchsorted(nodes, dst[ci])), tg = 대상 양 끝의 같은 자리.
  5. 저장 꼴(C1:44-62): nx float32 [ΣN,19] · ei int32 [ΣE,2] · ea float32 [ΣE,17] · tg int32 [ΣT,2] · tx float32 [ΣT,17]
     · y int8 [ΣT] · eid int64 [ΣT] · grp int64 [G,6] = (n0, n1, e0, e1, t0, t1).
     - nx = node19(D) 의 그 계좌 행(DP:384-389 → node19.py), ea = 문맥 거래 x17, tx = 대상 거래 x17(x17.py).

쓰는 법
  P = salt_plan(L, D, W)                         # 묶음 · 목록 · 그래프 모양(정수만). P.need_eids = x17 이 필요한 거래 전부
  A = salt_arrays(P, T19, xtab)                  # T19 = node19.node19_day(L, D, W=W), xtab = x17.x17_table(L, P.need_eids ∪ …)
  A, P = build_salt_day(L, D)                    # 위 셋을 한 번에(W · T19 · xtab 를 주면 다시 계산하지 않음)
  save_salt_day(A, out_dir)                      # C1 과 같은 파일 꼴로 저장(+ _DONE)

거부(T5)
  - D ≥ 2022-10-15: PermissionError(ledger.check_seal). D < 2022-08-08 또는 원장 d_max 이후: ValueError(node19 와 같은 검사).
  - target_eids 를 주었을 때 원장에 없거나 그날 거래가 아닌 edge_id: ValueError.
"""
from __future__ import annotations

import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '2')

import datetime
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

import sys
sys.path.insert(0, str(Path(__file__).resolve().parent))
import ledger as LG    # noqa: E402
import node19 as N19   # noqa: E402
import x17 as X17      # noqa: E402

# ───────────────────────── 고정 상수 ─────────────────────────
GROUP = 128            # CM:36 GROUP — 묶음 하나의 대상 거래 수
GROUP_SEED = 20261013  # C1:34 default_rng([20261013, dayofyear])
K1, K2 = 100, 10       # 1홉 · 2홉 상한(CM:27-32 SALT_K=100 · CM:83-84 FGT_NB_CAPS=K,10 · C1:43 assert (k1, k2) == (KNB, 10))
FIELDS = ('nx', 'ei', 'ea', 'tg', 'tx', 'y', 'eid', 'grp')   # C1:59-62 저장 파일


def day_of_year(D: str) -> int:
    """pd.Timestamp(D).dayofyear 와 같은 값(1월 1일 = 1)."""
    return datetime.date.fromisoformat(str(D)[:10]).timetuple().tm_yday


def make_groups(n: int, D: str):
    """C1:32-35 groups_of: 대상 자리 0..n−1 을 고정 난수로 섞어 128개씩. 반환 = 자리 배열 목록."""
    perm = np.random.default_rng([GROUP_SEED, day_of_year(D)]).permutation(n)
    return [perm[s:s + GROUP] for s in range(0, n, GROUP)]


# ───────────────────────── 대상 거래 ─────────────────────────
def day_targets(L, D: str, target_eids=None):
    """그날 대상 거래 → (eid int64 오름차순, y int8, src int64, dst int64, mode).
    기본(target_eids=None) = 그날 패턴 행 전부(pat_day == D) — 원래 C1:33 의 y9 ∈ 0..7 자리와 같은 집합 · 같은 순서.
    target_eids 를 주면 그 목록(고유 · 오름차순으로 정리)을 쓴다. y 는 패턴 행이면 pat_y, 아니면 −1.
    이 경우 128개 묶음 구성이 원래와 달라지므로(같은 묶음에 든 다른 행이 바뀜) 값은 유효하지만 val 결과와 그대로 비교할 수는 없음."""
    di = LG.minutes_of(D) // 1440
    pday = np.asarray(L['pat_day']).astype(np.int64)
    pe_day = np.asarray(L['pat_eid'])[pday == di].astype(np.int64)
    py_day = np.asarray(L['pat_y'])[pday == di].astype(np.int8)
    if len(pe_day) > 1 and not (np.diff(pe_day) > 0).all():
        raise ValueError(f'{D}: 원장 패턴 행이 edge_id 오름차순이 아님(LEDGER_FORMAT 3.3)')
    if target_eids is None:
        eid, y, mode = pe_day, py_day, 'patterns'
    else:
        eid = np.unique(np.asarray(target_eids, np.int64)); mode = 'eids'
        j = np.minimum(np.searchsorted(pe_day, eid), max(len(pe_day) - 1, 0))
        hit = (pe_day[j] == eid) if len(pe_day) else np.zeros(len(eid), bool)
        y = np.where(hit, py_day[j] if len(pe_day) else -1, -1).astype(np.int8)
    if len(eid) == 0:
        raise ValueError(f'{D}: 대상 거래가 없음(DP:398 No target transactions)')
    src, dst, pos = N19.target_accounts(L, eid)            # 원장에 없는 edge_id 면 ValueError(T5)
    tday = np.asarray(L['ts'][pos]).astype(np.int64) // 1440
    if (tday != di).any():                                 # T5: 그날에 없는 거래는 거부
        bad = eid[tday != di][:5].tolist()
        raise ValueError(f'{D}: 그날 거래가 아닌 edge_id {int((tday != di).sum())}개, 예: {bad}')
    return eid, y, src, dst, mode


# ───────────────────────── 받은 목록(하루 문맥) ─────────────────────────
def _sorted_lists(W, sel):
    """창 행 번호 sel → (dst ASC, ts DESC, edge_id DESC) 로 정렬한 행 번호(DP:393-394 의 PARTITION · ORDER 와 같은 순서)."""
    d = W['dst'][sel].astype(np.int64)
    t = W['ts'][sel].astype(np.int64)
    e = W['eid'][sel].astype(np.int64)
    return sel[np.lexsort((-e, -t, d))]                    # lexsort: 마지막 키(d)가 첫째 정렬 키


def _cap_rows(W, rows, cap):
    """정렬된 행(rows)에서 계좌(dst)마다 앞 cap[dst] 개만 남김(DP:393 QUALIFY row_number() ≤ K)."""
    if len(rows) == 0:
        return rows
    d = W['dst'][rows]
    st = np.flatnonzero(np.r_[True, d[1:] != d[:-1]])      # 계좌 구간의 시작
    cnt = np.diff(np.r_[st, len(d)])
    rank = np.arange(len(d)) - np.repeat(st, cnt)          # 구간 안 순번 0, 1, 2, …
    return rows[rank < cap[d]]


def day_context(W, n_acct: int, seeds, k_seed=K1, k_other=K2):
    """하루 문맥 = 필요한 계좌의 받은 목록 앞 K 개를 dst 오름차순으로 이어 붙인 창 행 번호.
    - 씨앗(그날 대상 양 끝): 앞 k_seed(100) — K100:67-83 이 씨앗 목록만 100 으로 늘린 것과 같음.
    - 씨앗 목록에 나온 보낸 계좌(2홉 후보): 앞 k_other(10). 원래 문맥은 64 지만 2홉은 앞 10 만 읽음(NB 셀 35:151).
    반환: (행 번호 int64 [C] — 문맥 순서, 통계 dict)."""
    t0 = time.time()
    cap = np.zeros(n_acct, np.int16)
    seeds = np.unique(np.asarray(seeds, np.int64))
    cap[seeds] = k_seed
    # 1차: 씨앗이 받은 거래
    sel1 = np.flatnonzero(cap[W['dst']] > 0)
    c1 = _cap_rows(W, _sorted_lists(W, sel1), cap)
    # 2차: 씨앗 목록의 보낸 계좌 중 씨앗이 아닌 것(씨앗이면 이미 앞 100 을 가짐 → 그 앞 10 = 2홉 목록)
    w2 = np.unique(W['src'][c1].astype(np.int64))
    w2 = w2[cap[w2] == 0]
    cap2 = np.zeros(n_acct, np.int16); cap2[w2] = k_other
    sel2 = np.flatnonzero(cap2[W['dst']] > 0)
    c2 = _cap_rows(W, _sorted_lists(W, sel2), cap2)
    # 두 묶음의 계좌는 서로 겹치지 않음 → dst 로만 안정 정렬하면 계좌 안 순서(ts DESC, edge_id DESC)가 유지됨
    rows = np.r_[c1, c2]
    rows = rows[np.argsort(W['dst'][rows], kind='stable')]
    st = dict(seed_accounts=int(len(seeds)), hop2_accounts=int(len(w2)), rows_seed_lists_full=int(len(sel1)),
              rows_hop2_lists_full=int(len(sel2)), context_rows=int(len(rows)), sec=round(time.time() - t0, 2))
    return rows, st


class CappedSampler:
    """NB 셀 35:138-159 CappedNeighborSampler (+ 셀 13:2-25 = DP:406-434 FixedNeighborSampler) 의 계산을 그대로 옮김.
    문맥 배열(src, dst)은 dst 오름차순 · 계좌 안 (ts DESC, edge_id DESC)."""

    def __init__(self, ctx_src, ctx_dst, k1=K1, k2=K2):
        self.src = np.asarray(ctx_src, np.int64); self.dst = np.asarray(ctx_dst, np.int64)
        self.k1, self.k2 = k1, k2
        self.dest, self.start, self.count = np.unique(self.dst, return_index=True, return_counts=True)   # DP:413
        assert np.all(self.dst[:-1] <= self.dst[1:])                                                    # DP:414

    def incoming_k(self, nodes, k):
        """NB 셀 35:142-148 — 계좌마다 문맥 목록 앞 min(개수, k) 칸의 문맥 자리."""
        nodes = np.unique(nodes); i = np.searchsorted(self.dest, nodes); ok = i < len(self.dest)
        ok[ok] = self.dest[i[ok]] == nodes[ok]; i = i[ok]
        if not len(i):
            return np.empty(0, np.int64)
        counts = np.minimum(self.count[i], k); starts = self.start[i]
        offsets = np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)
        return np.repeat(starts, counts) + offsets

    def batch(self, tsrc, tdst):
        """NB 셀 35:149-159 batch 의 정수 부분. 반환 (ci 문맥 자리, nodes 계좌 번호, ei [2,E] int64, tg [2,T] int64)."""
        seeds = np.unique(np.r_[tsrc, tdst])                                        # 대상 거래의 보낸 · 받은 계좌
        first = self.incoming_k(seeds, self.k1); second = self.incoming_k(self.src[first], self.k2)   # 1홉 · 2홉 = 들어오는 엣지만
        ci = np.union1d(first, second)
        nodes = np.unique(np.r_[seeds, self.src[ci], self.dst[ci]])
        ei = np.stack([np.searchsorted(nodes, self.src[ci]), np.searchsorted(nodes, self.dst[ci])])
        tg = np.stack([np.searchsorted(nodes, tsrc), np.searchsorted(nodes, tdst)])
        return ci, nodes, ei, tg


# ───────────────────────── 하루 계획(정수 부분) ─────────────────────────
@dataclass
class SaltPlan:
    day: str
    mode: str                      # 'patterns'(기본, val 과 비교 가능) | 'eids'(목록을 받은 경우)
    target_eid: np.ndarray         # int64 [T] 오름차순
    target_y: np.ndarray           # int8 [T]
    target_src: np.ndarray         # int64 [T] 원장 계좌 번호
    target_dst: np.ndarray
    groups: list                   # 묶음마다 대상 자리 배열(C1:35)
    ctx_src: np.ndarray            # int64 [C] 문맥 거래 보낸 계좌
    ctx_dst: np.ndarray            # int64 [C] 문맥 거래 받은 계좌(오름차순)
    ctx_eid: np.ndarray            # int64 [C] 문맥 거래 edge_id
    parts: list                    # 묶음마다 (ci, nodes, ei, tg)
    need_eids: np.ndarray          # x17 이 필요한 edge_id 전부(고유 · 오름차순) = 묶음 엣지 ∪ 대상
    stats: dict = field(default_factory=dict)


def salt_plan(L, D: str, W=None, target_eids=None, log=None) -> SaltPlan:
    """D 하루의 묶음 · 받은 목록 · 묶음 그래프 모양을 원장에서 계산(실수 피처 없이 정수만).
    W: node19.read_window(L, D, cols ⊇ src, dst, ts, eid) 결과(W['_day'] == D 여야 함). 없으면 여기서 읽음(4열 · 10-06 기준 0.88 GB)."""
    T0 = time.time()
    D = N19._check_day(L, D, 'salt_plan')                  # 봉인 · 기간 검사(T9:70-80, DP:350, d_max)
    eid, y, tsrc, tdst, mode = day_targets(L, D, target_eids)
    groups = make_groups(len(eid), D)
    if W is None:
        W = N19.read_window(L, D, cols=('src', 'dst', 'ts', 'eid'))
    elif W.get('_day') != D:
        raise ValueError(f'W 는 {W.get("_day")} 창 — {D} 창이 필요함')
    n_acct = int(len(L['acct_ent']))
    rows, cst = day_context(W, n_acct, np.r_[tsrc, tdst])
    ctx_src = W['src'][rows].astype(np.int64); ctx_dst = W['dst'][rows].astype(np.int64); ctx_eid = W['eid'][rows].astype(np.int64)
    sm = CappedSampler(ctx_src, ctx_dst)
    parts = []; used = []
    for ix in groups:
        ci, nodes, ei, tg = sm.batch(tsrc[ix], tdst[ix])
        parts.append((ci, nodes, ei, tg)); used.append(ctx_eid[ci])
    need = np.unique(np.r_[np.concatenate(used) if used else np.zeros(0, np.int64), eid])
    st = dict(targets=int(len(eid)), groups=len(groups), node_rows=int(sum(len(p[1]) for p in parts)),
              edge_rows=int(sum(len(p[0]) for p in parts)), unique_edges=int(len(need)), context=cst,
              window_rows=int(len(W['dst'])), sec=round(time.time() - T0, 2))
    if log:
        log(f'[salt_plan] {D}: 대상 {st["targets"]:,} · 묶음 {st["groups"]} · 노드 행 {st["node_rows"]:,} · '
            f'엣지 행 {st["edge_rows"]:,} · x17 필요 {st["unique_edges"]:,} · 문맥 {cst} · {st["sec"]}초')
    return SaltPlan(D, mode, eid, y, tsrc, tdst, groups, ctx_src, ctx_dst, ctx_eid, parts, need, st)


# ───────────────────────── 실수 피처 붙이기 ─────────────────────────
def salt_arrays(P: SaltPlan, T19, xtab) -> dict:
    """C1:44-62 build_day 의 저장 배열 8개.
    T19: node19.node19_day(L, P.day, ...) 결과(.take). xtab: x17.x17_table(L, eids ⊇ P.need_eids) 결과 (eid 오름차순, X)."""
    if getattr(T19, 'day', P.day) != P.day:
        raise ValueError(f'node19 표의 날({T19.day}) ≠ 묶음의 날({P.day})')
    acc = {k: [] for k in ('nx', 'ei', 'ea', 'tg', 'tx')}; grp = []; n0 = e0 = t0 = 0
    for ix, (ci, nodes, ei, tg) in zip(P.groups, P.parts):
        acc['nx'].append(np.asarray(T19.take(nodes, strict=True), np.float32))   # NB 셀 35:155-156 'Unknown node mapping' 과 같은 뜻
        acc['ei'].append(ei.T.astype(np.int32))                                   # C1:47
        acc['ea'].append(np.asarray(X17.x17_take(xtab, P.ctx_eid[ci]), np.float32))
        acc['tg'].append(tg.T.astype(np.int32))                                   # C1:48
        acc['tx'].append(np.asarray(X17.x17_take(xtab, P.target_eid[ix]), np.float32))
        grp.append((n0, n0 + len(nodes), e0, e0 + ei.shape[1], t0, t0 + len(ix)))    # C1:49
        n0 += len(nodes); e0 += ei.shape[1]; t0 += len(ix)
    allpos = np.concatenate(P.groups)                                             # C1:58
    A = {k: np.concatenate(v) for k, v in acc.items()}
    A['y'] = P.target_y[allpos].astype(np.int8)                                   # C1:61
    A['eid'] = P.target_eid[allpos].astype(np.int64)
    A['grp'] = np.array(grp, np.int64).reshape(-1, 6)                             # C1:62
    for k in ('nx', 'ea', 'tx'):                                                  # DP:389, 400-401 finite(...) 와 같은 확인
        if not np.isfinite(A[k]).all():
            raise ValueError(f'{P.day}: {k} 에 NaN/Inf')
    return A


def build_salt_day(L, D: str, W=None, T19=None, xtab=None, target_eids=None, log=None):
    """D 하루 SALT/TC 캐시 배열 8개를 원장에서 바로. 반환 (A dict, SaltPlan).
    W · T19 · xtab 를 주면(같은 날 다른 소비자와 함께 계산한 것) 다시 계산하지 않음."""
    D = str(D)[:10]
    need_node = T19 is None
    if W is None:
        cols = ('src', 'dst', 'ts', 'eid', 'pz', 'pcur', 'rcur') if need_node else ('src', 'dst', 'ts', 'eid')
        W = N19.read_window(L, D, cols=cols)
    P = salt_plan(L, D, W=W, target_eids=target_eids, log=log)
    if need_node:
        T19 = N19.node19_day(L, D, W=W if all(c in W for c in ('pz', 'pcur', 'rcur')) else None, log=log)
    if xtab is None:
        xtab = X17.x17_table(L, P.need_eids, log=log)
    return salt_arrays(P, T19, xtab), P


def save_salt_day(A: dict, out_dir):
    """C1:57-62 와 같은 파일 꼴로 저장(nx.npy … grp.npy + _DONE)."""
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    for k in FIELDS:
        np.save(out / f'{k}.npy', A[k])
    (out / '_DONE').write_text(out.name.replace('day=', ''))


def load_salt_day(day_dir) -> dict:
    """C1 꼴 하루 캐시 폴더 → 배열 dict(묶음 data/salt_cache_K100/val/day=D 를 읽어 대조할 때)."""
    d = Path(day_dir)
    return {k: np.load(d / f'{k}.npy') for k in FIELDS}
'''

In [ ]:
# @title 코드 ⑤ TGT B0 하루 시퀀스 캐시(tgt_inputs.py) — 숨김 · 고치지 마세요
_MODULE_SRC['tgt_inputs'] = r'''"""tgt_inputs — TGT B0 하루 시퀀스 캐시(eid + 14키)를 압축 원장 + x17.py + node19.py 로 만든다(numpy 만).

결과 dict 는 원래 t1_cache.build 가 저장한 data/tgt_cache_K100/val/<D>.npz 와 같은 키 · 순서 · dtype · shape:
  eid[B] i64, x[A,100,17] f16, dir[A,100] i8, dt[A,100] f32, bkt[A,100] i8, cp[A,100] i64, mask[A,100] bool,
  inv[B,2] i64, dtt[B,2,100] f32, qdt[B,2] f32, uv[B,2] i64, tx[B,17] f32, nu[B,19] f32, nv[B,19] f32, y[B] i64
  (t1_cache.py:28 `np.savez(tmp, eid=..., **b)`, b = tgt_data.make_batch 기본 경로 dict 순서 tgt_data.py:164-167).
cp · uv 만 번호 체계가 다름: 원래는 파일 전체 node_id(DP:175-178), 여기는 원장 acct id(계좌표 사전순 순위).
두 번호는 일대일 · 순서 보존(ledger T1-e)이라 inv · 계좌 순서는 같고, 모델은 cp · uv 를 같음 비교에만 씀
(aml8.py:519-524 ncoe_counts, :580 ns) → 로짓이 같음. 대조는 계좌 키(은행, 계좌)로 함(t3_tgt.py).

원래 코드(읽기 전용) 약칭:
  TGD = Transformer모델/기법실행_20261008/code/tgt_data.py      (Store.seqs · make_batch · bucket)
  BTD = Transformer모델/TGT/code/build_tgt_data.py               (계좌별 색인: dir · cp 규칙)
  RTH = Transformer모델/TGT/code/reorder_ties_hash.py            (같은 분 동점 해시)
  T1C = Transformer모델/TGT_유형8_기법_20261013/t1_cache.py       (하루 캐시 = 패턴 행 전부로 make_batch 한 번)
  CMT = Transformer모델/TGT_유형8_기법_20261013/common_t.py       (K = 100 · 키 목록)
  AML = 모델묶음_유형8_최종후보_20261016/aml8.py                  (모델 · 예측)
  DP  = 태훈님_전처리/feature_code_40_20260921/CODE/daily_pipeline.py

규칙(원래 → 여기)
  1. 대상 = 그날 패턴 행(y 0~7) 전부, edge_id 오름차순(T1C:23 pos = (y ≥ 0) & (y < 8), day.tid 순서).
     원장 pat_eid[pat_day == D] 는 (날, edge_id) 오름차순이고 val55 와 같음(ledger T1-d).
     te = 대상 거래의 분(TGD:130 store.tsmin[tid]) = 원장 ts. nu · nv = node_x[src] · node_x[dst](TGD:128-129, 134).
  2. 계좌 집합 acc = np.unique(u ∪ v) (TGD:159). 원장 acct id 순서 = 원래 node_id 순서 → inv 가 비트 단위로 같음.
  3. 계좌 a 의 거래 목록(BTD:80-84): 받음(a = dst ≠ src) dir 0 · cp = src / 보냄(a = src ≠ dst) dir 1 · cp = dst /
     자기 이체(src = dst = a) dir 2 · cp = a, 한 번만. 라벨로 거르지 않음.
  4. 순서(RTH:19-33): (계좌, 분, h32(edge_id)) 오름차순, 해시까지 같으면 edge_id 오름차순
     (원래 BTD:84 ORDER BY a, m, edge_id 뒤 안정 lexsort). h32(e) = ((e·0x9E3779B97F4A7C15) mod 2^64) >> 32.
  5. 창(TGD:94-96): [D−29일 00:00, D+1일 00:00) — D 하루 전체(대상 뒤 같은 날 거래 포함). di−29 < 0 이면 아래 끝 0(:95).
     원장 ts ≥ 0 이라 창 아래 끝이 음수면 자연히 처음부터임.
  6. 끝에서 K=100 개(start = max(end − K, lo), :96) → 칸 0..n−1 에 오래된 것부터. mask = 칸 < n(:98).
     빈 칸 n..K−1 은 칸 0 을 복사(x · dir · cp · ts, :99) → dt · bkt · dtt 도 칸 0 값, mask False.
  7. x = float16(x17[eid])(:102), anchor = 계좌 창 마지막 거래 분(:104), gap = anchor − ts, dt = float32(−gap)(:164),
     bkt = gap // 1440 → 0 / 1 / 2~6 / 7+(:49-52), dtt = float32(ts[inv] − te)(:162), qdt = float32(te − anchor[inv])(:163),
     uv = [u, v](:165), tx = float32(x17[tid])(:166), y = int64(:167).
  8. K = 100: CMT:23 TGT_K=100 을 CMT:82 가 TD.K_NB 로 넣음. AML:1298 이 100 을 확인함. WIN_DAYS 30 · BASE 08-01(TGD:32-33).

대상 바꾸기(eids=…): 원래 make_batch 값은 계좌마다 독립이라(TGD:145 주석, T1C:30-35 --verify) 대상 일부만 줘도 그 행 · 계좌 값은
전체로 만든 것과 같음. 패턴 행이 아니면 y = −1(모델은 y 를 쓰지 않음).

쓰는 법:
  C = tgt_day(L, '2022-10-06')                              # 혼자: 창 훑기 · x17 · node19 를 스스로 함
  P = plan_day(L, D, W=W); tab = X17.x17_table(L, np.union1d(P.need_eids, 다른_소비자_eid))
  C = finish_day(P, x17tab=tab, n19=N19.node19_day(L, D, W=W))   # 노트북: 창 W · x17 표 · node19 를 함께 씀
"""
from __future__ import annotations

import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '2')

import resource
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import ledger as LG  # noqa: E402
import node19 as N19  # noqa: E402
import x17 as X17  # noqa: E402

# ───────────────────────── 상수(원래 값 그대로) ─────────────────────────
K_NB = 100                      # CMT:23-25 TGT_K=100 · CMT:82 TD.K_NB = KNB · AML:1298 assert 100 (TGD:32 기본 64 는 덮어씀)
WIN_DAYS = 30                   # TGD:32
BASE_DATE = LG.BASE_DATE        # '2022-08-01' — TGD:33 BASE, 원장 ts 의 0점
DAY_MIN = 1440
TRAIN_START = '2022-08-08'      # DP:350 — 보정 주(08-01~08-07)에는 대상이 없음
DIR_RECV, DIR_SENT, DIR_SELF = 0, 1, 2          # BTD:81-83 (meta dir_code '0' 받음 · '1' 보냄 · '2' 자기 이체)
TIE_MUL = np.uint64(0x9E3779B97F4A7C15)         # RTH:6, 19-20
ACC_KEYS = ('x', 'dir', 'dt', 'bkt', 'cp', 'mask')                  # CMT:29 계좌(시퀀스) 단위
TGT_KEYS = ('inv', 'dtt', 'qdt', 'uv', 'tx', 'nu', 'nv', 'y')       # CMT:30 대상 거래 단위
CACHE_KEYS = ('eid',) + ACC_KEYS + TGT_KEYS                         # T1C:28 저장 순서
CACHE_DTYPES = dict(eid=np.int64, x=np.float16, dir=np.int8, dt=np.float32, bkt=np.int8, cp=np.int64, mask=np.bool_,
                    inv=np.int64, dtt=np.float32, qdt=np.float32, uv=np.int64, tx=np.float32, nu=np.float32,
                    nv=np.float32, y=np.int64)


def h32(e: np.ndarray) -> np.ndarray:
    """같은 (계좌, 분) 동점을 끊는 고정 해시. RTH:19-20 글자 그대로(uint64 곱은 2^64 에서 돌아감)."""
    return ((np.asarray(e).astype(np.uint64) * TIE_MUL) >> np.uint64(32)).astype(np.int64)


def bucket(gap_min: np.ndarray) -> np.ndarray:
    """anchor 기준 24시간 단위 구간 0 / 1 / 2~6 / 7 이상. TGD:49-52 글자 그대로."""
    b = gap_min // 1440
    return np.select([b <= 0, b == 1, b <= 6], [0, 1, 2], 3).astype(np.int8)


def day_index(D: str) -> int:
    """BASE 부터 날 수(TGD:45-46 day_index 와 같음)."""
    return LG.minutes_of(D) // DAY_MIN


def _check_day(L, D: str) -> str:
    """봉인 · 기간 확인. 2022-10-15 이후는 AML_TEST_UNSEAL=1 없이 PermissionError(ledger.check_seal = T9:70-80 뜻)."""
    D = str(D)[:10]
    LG.check_seal(D, 'tgt_inputs')
    if D < TRAIN_START:
        raise ValueError(f'tgt_inputs({D}): 보정 기간에는 대상이 없음(DP:350, {TRAIN_START} 이후만)')
    if D > str(L.meta['d_max'])[:10]:
        raise ValueError(f'tgt_inputs({D}): 원장 d_max {L.meta["d_max"]} 보다 늦은 날 — 원장을 더 늦게까지 지어야 함')
    return D


def _mem():
    d = {}
    try:
        for line in open('/proc/self/status'):
            if line.startswith(('VmHWM', 'VmRSS', 'RssAnon', 'RssFile')):
                k, v = line.split(':'); d[k] = round(int(v.split()[0]) / 2 ** 20, 3)
    except OSError:
        pass
    d['ru_maxrss_gib'] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20, 3)
    d['cgroup_anon_gib'] = round(LG.cgroup_anon_gib(), 3)
    return d


# ───────────────────────── 대상 ─────────────────────────
def targets_of_day(L, D: str, eids=None):
    """그날 대상 → (eid 오름차순 int64, y int64, src acct id int64, dst acct id int64, te 분 int64).
    eids=None: 그날 Patterns.txt 행 전부(T1C:23 = val55 그날 행). eids 를 주면 고유 · 오름차순으로 정렬해 씀.
    그날 거래가 아니거나 원장에 없는 edge_id(중복으로 지움 · 자르는 시각 이후 · 범위 밖)면 ValueError(T5)."""
    di = day_index(D)
    pe = np.asarray(L['pat_eid'], np.int64); pd_ = np.asarray(L['pat_day'], np.int64); py = np.asarray(L['pat_y'], np.int64)
    sel = pd_ == di
    if eids is None:
        e = pe[sel]
        y = py[sel]
        if len(e) > 1 and not (np.diff(e) > 0).all():
            raise ValueError('원장 pat_eid 가 그날 안에서 edge_id 오름차순이 아님(LEDGER_FORMAT 3.3 위반)')
    else:
        e = np.unique(np.asarray(eids, np.int64).reshape(-1))
        pe_d, py_d = pe[sel], py[sel]                      # 그날 패턴 행(edge_id 오름차순)
        j = np.minimum(np.searchsorted(pe_d, e), max(len(pe_d) - 1, 0))
        hit = (pe_d[j] == e) if len(pe_d) else np.zeros(len(e), bool)
        y = np.where(hit, py_d[j] if len(pe_d) else -1, -1).astype(np.int64)   # 패턴 행이 아니면 −1(모델은 y 를 안 씀)
    if len(e) == 0:
        raise ValueError(f'{D}: 대상이 없음')
    s, d, p = N19.target_accounts(L, e)                    # 원장에 없는 edge_id 면 ValueError
    t = np.asarray(L['ts'][p]).astype(np.int64)
    bad = (t // DAY_MIN) != di
    if bad.any():
        raise ValueError(f'{D}: 그날 거래가 아닌 edge_id {int(bad.sum())}개: {e[bad][:5].tolist()}')
    return e, y, s, d, t


# ───────────────────────── 1단계: 계좌 시퀀스 계획(x17 · node19 전) ─────────────────────────
class TGTPlan:
    """plan_day 결과. 시퀀스 칸의 edge_id · 시각 · 방향 · 상대 계좌와 대상 정보를 담음. need_eids = x17 이 필요한 거래."""

    def __init__(self, **kw):
        self.__dict__.update(kw)

    def __repr__(self):
        return f'TGTPlan({self.day}, targets={len(self.eid):,}, accounts={len(self.acc):,}, need_eids={len(self.need_eids):,})'


def _collect(src, dst, ts, eid, isacc, parts):
    """창 행 묶음 하나에서 대상 계좌의 거래를 BTD:80-84 규칙으로 뽑아 parts 에 붙임."""
    src = src.astype(np.int64, copy=False); dst = dst.astype(np.int64, copy=False)
    ins, ind = isacc[src], isacc[dst]
    if not (ins.any() or ind.any()):
        return
    selfm = src == dst
    m0 = ind & ~selfm                                      # 받음: a = dst, cp = src (BTD:81)
    m1 = ins & ~selfm                                      # 보냄: a = src, cp = dst (BTD:82)
    m2 = ins & selfm                                       # 자기 이체: a = src, cp = src, 한 번만 (BTD:83)
    for m, a, code, c in ((m0, dst, DIR_RECV, src), (m1, src, DIR_SENT, dst), (m2, src, DIR_SELF, src)):
        k = int(m.sum())
        if k:
            parts['A'].append(a[m]); parts['E'].append(eid[m].astype(np.int64)); parts['T'].append(ts[m].astype(np.int64))
            parts['R'].append(np.full(k, code, np.int8)); parts['C'].append(c[m])


def plan_day(L, D: str, eids=None, W=None, K: int = K_NB, chunk_rows: int = 8_000_000, log=None) -> TGTPlan:
    """D 하루 TGT 시퀀스 계획. W = node19.read_window(L, D, cols ⊇ src,dst,ts,eid) 를 주면 원장을 다시 훑지 않음."""
    T0 = time.time(); say = log or (lambda s: None); st = {}
    D = _check_day(L, D); di = day_index(D)
    eid, y, u, v, te = targets_of_day(L, D, eids)
    acc, inv = np.unique(np.concatenate([u, v]), return_inverse=True)          # TGD:159
    inv = inv.reshape(2, -1).T.copy().astype(np.int64)                           # [B, 2]
    n_acct = len(L['acct_bank'])
    isacc = np.zeros(n_acct, bool); isacc[acc] = True
    lo_t, hi_t = N19.window_bounds(D)                                            # [D−29 00:00, D+1 00:00) = TGD:94-96
    assert lo_t == (di - WIN_DAYS + 1) * DAY_MIN and hi_t == (di + 1) * DAY_MIN
    st['t_targets'] = round(time.time() - T0, 2)

    # (1) 창 행 중 대상 계좌가 끝점인 거래만 모음(원장 행 순서 = edge_id 오름차순)
    t1 = time.time(); parts = {k: [] for k in 'AETRC'}; n_win = 0
    if W is not None:
        if str(W.get('_day', D)) != D:
            raise ValueError(f'W 는 {W.get("_day")} 창임 — {D} 가 아님')
        for c in ('src', 'dst', 'ts', 'eid'):
            if c not in W:
                raise ValueError(f'W 에 {c} 열이 없음(read_window cols 에 넣어야 함)')
        n_win = len(W['ts'])
        for s0 in range(0, n_win, chunk_rows):                                   # 묶음으로 나눠 임시 메모리를 줄임
            sl = slice(s0, min(n_win, s0 + chunk_rows))
            _collect(W['src'][sl], W['dst'][sl], W['ts'][sl], W['eid'][sl], isacc, parts)
    else:
        for b in N19.iter_window(L, D, ('src', 'dst', 'eid'), chunk_rows):       # ts 로 먼저 창을 고른 뒤 다른 열을 읽음
            n_win += len(b['ts'])
            _collect(b['src'], b['dst'], b['ts'], b['eid'], isacc, parts)
    if not parts['A']:
        raise ValueError(f'{D}: 대상 계좌의 창 거래가 없음')
    A = np.concatenate(parts['A']); E = np.concatenate(parts['E']); T = np.concatenate(parts['T'])
    R = np.concatenate(parts['R']); C = np.concatenate(parts['C']); del parts
    assert (T >= lo_t).all() and (T < hi_t).all(), '창 밖 거래가 섞임'
    st['t_scan'] = round(time.time() - t1, 2); st['window_rows'] = int(n_win); st['account_rows'] = int(len(A))

    # (2) 계좌 → 분 → h32(edge_id) → edge_id 오름차순(RTH:33 lexsort((h32, t, owner)) 를 BTD:84 edge_id 순서 위에서 안정 정렬한 것과 같음)
    t2 = time.time()
    o = np.lexsort((E, h32(E), T, A))
    A, E, T, R, C = A[o], E[o], T[o], R[o], C[o]; del o
    stt = np.searchsorted(A, acc, 'left'); en = np.searchsorted(A, acc, 'right')
    nA = len(acc)
    # (3) 끝 K 개 · 빈 칸 = 칸 0 복사(TGD:94-104)
    st2 = np.maximum(en - K, stt); n = en - st2
    if not (n >= 1).all():
        raise AssertionError('대상 거래의 끝점인데 그날 거래가 없다(TGD:97)')
    ar = np.arange(K); mask = ar[None, :] < n[:, None]
    p = np.where(mask, st2[:, None] + ar[None, :], st2[:, None])                # TGD:99 빈 칸 = 첫 칸(mask 로 뺌)
    slot_eid = E[p]; slot_ts = T[p]; slot_dir = R[p]; slot_cp = C[p]
    anchor = T[en - 1]                                                           # TGD:104 계좌 창 마지막 거래 시각
    assert (slot_ts[mask] < hi_t).all() and (slot_ts[mask] >= lo_t).all(), '창 밖(다음 날 이후 또는 30일 전) 거래(TGD:101)'
    need = np.unique(np.concatenate([slot_eid[mask], eid]))                      # x17 이 필요한 거래(빈 칸은 칸 0 과 같은 거래)
    st['t_sort_slots'] = round(time.time() - t2, 2)
    st.update(targets=int(len(eid)), accounts=int(nA), real_slots=int(mask.sum()), full_accounts=int((n == K).sum()),
              need_eids=int(len(need)), t_plan=round(time.time() - T0, 2))
    say(f'[tgt] {D} 계획: 대상 {len(eid):,} · 계좌 {nA:,} · 창 {n_win:,}행 중 계좌 거래 {st["account_rows"]:,} · '
        f'실제 칸 {st["real_slots"]:,} · x17 필요 {len(need):,} · {st["t_plan"]}초')
    return TGTPlan(day=D, di=di, K=K, eid=eid, y=y, u=u, v=v, te=te, acc=acc, inv=inv, slot_eid=slot_eid, slot_ts=slot_ts,
                   dir=slot_dir.astype(np.int8), cp=slot_cp.astype(np.int64), mask=mask, anchor=anchor, need_eids=need, stats=st)


# ───────────────────────── 2단계: 배열 채우기(TGD:159-167) ─────────────────────────
def finish_day(P: TGTPlan, x17tab=None, n19=None, L=None, W=None, log=None) -> dict:
    """계획 P 에 x17 · node19 를 붙여 캐시 dict(CACHE_KEYS 순서) 를 만듦.
    x17tab = x17.x17_table 결과(P.need_eids 를 모두 포함해야 함). 없으면 L 로 계산.
    n19 = node19.node19_day(L, P.day) 결과. 없으면 L(+W) 로 계산."""
    T0 = time.time(); say = log or (lambda s: None); st = P.stats
    if x17tab is None:
        if L is None:
            raise ValueError('x17tab 이 없으면 L 이 필요함')
        t = time.time(); x17tab = X17.x17_table(L, P.need_eids); st['t_x17'] = round(time.time() - t, 2)
    if n19 is None:
        if L is None:
            raise ValueError('n19 가 없으면 L 이 필요함')
        t = time.time(); n19 = N19.node19_day(L, P.day, W=W); st['t_node19'] = round(time.time() - t, 2)
    if getattr(n19, 'day', P.day) != P.day:
        raise ValueError(f'node19 는 {n19.day} 날 표임 — {P.day} 가 아님')
    t = time.time(); A, K = P.mask.shape
    x = X17.x17_take(x17tab, P.slot_eid.reshape(-1)).astype(np.float16).reshape(A, K, 17)   # TGD:102 np.asarray(x17[eid], float16)
    gap = P.anchor[:, None] - P.slot_ts                                                      # TGD:161 ≥ 0
    dtt = P.slot_ts[P.inv] - P.te[:, None, None]                                             # TGD:162 [B, 2, K]
    qdt = P.te[:, None] - P.anchor[P.inv]                                                    # TGD:163 [B, 2] ≤ 0
    out = dict(eid=P.eid.astype(np.int64),
               x=x, dir=P.dir, dt=(-gap).astype(np.float32), bkt=bucket(gap), cp=P.cp, mask=P.mask,
               inv=P.inv, dtt=dtt.astype(np.float32), qdt=qdt.astype(np.float32), uv=np.stack([P.u, P.v], 1).astype(np.int64),
               tx=X17.x17_take(x17tab, P.eid).astype(np.float32),                            # TGD:114, 166 target_x = x17[tid]
               nu=n19.take(P.u, strict=True).astype(np.float32),                            # TGD:129, 166 node_x[src]
               nv=n19.take(P.v, strict=True).astype(np.float32),
               y=P.y.astype(np.int64))
    for k in CACHE_KEYS:                                                                     # dtype 고정(원래 캐시와 같게)
        assert out[k].dtype == CACHE_DTYPES[k], (k, out[k].dtype)
    if not (np.isfinite(out['nu']).all() and np.isfinite(out['nv']).all()):
        raise ValueError('nu/nv 에 NaN — 창에 없는 계좌(있을 수 없음)')
    st['t_assemble'] = round(time.time() - t, 2); st['t_finish'] = round(time.time() - T0, 2); st['mem'] = _mem()
    say(f'[tgt] {P.day} 채움 {st["t_finish"]}초 · x {x.shape} · 대상 {len(P.eid):,}')
    return {k: out[k] for k in CACHE_KEYS}


def tgt_day(L, D: str, eids=None, W=None, x17tab=None, n19=None, K: int = K_NB, stats=None, log=None) -> dict:
    """D 하루 TGT 캐시(원래 tgt_cache_K100/<D>.npz 와 같은 꼴). cp · uv 는 원장 acct id."""
    P = plan_day(L, D, eids=eids, W=W, K=K, log=log)
    out = finish_day(P, x17tab=x17tab, n19=n19, L=L, W=W, log=log)
    if stats is not None:
        stats.update(P.stats)
    return out


def save_npz(cache: dict, path) -> None:
    """원래와 같은 무압축 npz(T1C:27-28 tmp → rename)."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp.npz')
    np.savez(tmp, **{k: cache[k] for k in CACHE_KEYS}); tmp.rename(path)


def account_keys(L, ids) -> np.ndarray:
    """원장 acct id → '은행|계좌' 문자열(원래 node_mapping 의 bank · account 와 같은 정규화, DP:175-178). 대조 · 표시용."""
    ids = np.asarray(ids, np.int64)
    u, back = np.unique(ids.reshape(-1), return_inverse=True)
    b = np.asarray(L['acct_bank'])[u]; a = np.asarray(L['acct_num'])[u]
    k = np.char.add(np.char.add(b.astype('U'), '|'), a.astype('U'))
    return k[back].reshape(ids.shape)
'''

In [ ]:
# @title 코드 ⑥ SALT-GNN C · TransformerConv B 모델(salt_models.py · 순수 torch) — 숨김 · 고치지 마세요
_MODULE_SRC['salt_models'] = r'''"""salt_models — SALT-GNN C · TransformerConv B 예측(순수 torch, torch_geometric 없음).

입력: salt_inputs.salt_arrays(...) 가 만든 하루 캐시 배열 dict(nx, ei, ea, tg, tx, y, eid, grp) 또는 그 목록(여러 날).
가중치: aml8_모델.pt 의 models['salt_c'] · models['tc_b'] (state_dict 안에 z-표준화 버퍼 zn.nm/ns/em/es · SALT δ 포함).

원래 코드(읽기 전용) 약칭
  A8  = Final_preprocess/HI-Large_EDA/모델묶음_유형8_최종후보_20261016/aml8.py
  PYG = torch_geometric 2.8.0 (설치본): utils/_softmax.py:82-92 · utils/_scatter.py:68-100 · nn/dense/linear.py:127
  PT  = 만들 때 쓴 순수 torch 시험판(PyG 판과 차 0 을 10-06 · 6시드에서 확인 · 이 모듈이 그것을 옮긴 것)

옮긴 것
  - TransformerConv(A8:234-319 = NB 셀 15) → TransformerConvPT. 파라미터 이름(lin_key/query/value/edge/skip/beta)이 같아
    state_dict 를 strict 로 그대로 읽음. PyG Linear 는 F.linear 와 같은 계산(PYG linear.py:127).
    · softmax: out = exp(a − max_dst) / (sum_dst + 1e−16) (PYG _softmax.py:82-92).
    · 'add' 모으기 = scatter_add_, mean = 합 / max(개수, 1), min · max = scatter_reduce_(include_self=False) (PYG _scatter.py:68-100).
  - salt_build_model(A8:370-479 = CM:152-263) 의 ZNorm · head · TC2(실행 B) · SALTLayer2 · SALT2(실행 C) 를 글자 그대로,
    TC 자리에 TransformerConvPT, `from torch_geometric.utils import scatter` 자리에 scatter_pt 만 바꿈.
  - 예측 반복(A8:1267-1291 predict_salt = c2_train.py:75-79): 묶음 4개(PER_BATCH)씩 노드 번호를 밀어 붙여 한 배치 →
    softmax(logits.float()). Store.batch(A8:346-356) 와 같은 꼴(MemStore).
  - 시드 평균(A8:1356): float32 확률을 float64 로 바꿔 평균 → seed_mean.

배치 구성과 끝자리(2026-10-17 실측, CPU · 2스레드, test_salt_models.py)
  - 원래 실행(aml8.py predict_salt)은 val 17일을 날 순서로 이어 붙여 묶음 4개씩 배치를 만든다. 그래서 날 경계를 넘는 배치가 생김
    (예: 10-06 앞에 묶음 94개 → 10-06 첫 배치는 10-05 묶음 2개와 함께 감).
  - 묶음 그래프끼리는 서로 떨어져 있지만, 배치 행 수가 달라지면 CPU 행렬곱의 계산 순서가 달라져 float32 끝자리가 바뀜.
    10-06 을 그날만 돌리면 17일 실행의 그날 행과 최대 3.3e-6(SALT C) · 2.4e-7(TC B) 다름, 1등 유형은 같음.
    앞 묶음 수가 4의 배수인 09-28 · 10-14 는 0.
  - 그래서 predict_salt 에 17일 배열을 날 순서 목록으로 주면 원래 CPU 실행과 비트 단위로 같고(차 0),
    일부 날만 주면 그 날들끼리 배치를 만든 값(차 수 e-6 이하)이 나옴.
"""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PER_BATCH, K8 = 4, 8                    # A8:295 GROUP, PER_BATCH, K8 = 128, 4, 8
# A8:301-306 실행 설정 중 최종 후보가 쓰는 B · C (desc 는 설명 글이라 뺌 — 모델 파일 cfg 와 숫자 열을 대조함)
RUNS = {'B': dict(model='tc2', H=64, heads=2, lr=0.001, p=0.1, pf=0.3, znorm=True),
        'C': dict(model='salt2', H=64, heads=2, lr=0.001, p=0.1, pf=0.3, znorm=True)}
MODEL_RUN = {'salt_c': 'C', 'tc_b': 'B'}


# ───────────────────────── PyG 2.8.0 의 scatter · softmax 를 순수 torch 로 ─────────────────────────
def _bc(index, src):
    """PYG _scatter.py broadcast: index 를 src 모양으로 펼침."""
    return index.view((-1,) + (1,) * (src.dim() - 1)).expand_as(src)


def scatter_pt(src, index, dim=0, dim_size=None, reduce='sum'):
    """torch_geometric.utils.scatter(dim=0) 와 같은 계산(PYG _scatter.py:68-100)."""
    assert dim == 0
    N = int(index.max()) + 1 if dim_size is None else dim_size
    size = (N,) + tuple(src.shape[1:])
    if reduce in ('sum', 'add'):
        return src.new_zeros(size).scatter_add_(0, _bc(index, src), src)
    if reduce == 'mean':
        cnt = src.new_zeros(N); cnt.scatter_add_(0, index, src.new_ones(src.size(0))); cnt = cnt.clamp(min=1)
        out = src.new_zeros(size).scatter_add_(0, _bc(index, src), src)
        return out / _bc(cnt, out)
    if reduce in ('min', 'max'):
        return src.new_zeros(size).scatter_reduce_(0, _bc(index, src), src, reduce='a' + reduce, include_self=False)
    raise ValueError(reduce)


def softmax_pt(src, index, num_nodes):
    """torch_geometric.utils.softmax(src, index, ptr=None, num_nodes) (PYG _softmax.py:82-92)."""
    m = scatter_pt(src.detach(), index, 0, num_nodes, 'max')
    out = (src - m.index_select(0, index)).exp()
    s = scatter_pt(out, index, 0, num_nodes, 'sum') + 1e-16
    return out / s.index_select(0, index)


class TransformerConvPT(nn.Module):
    """A8:233-291 TransformerConv(MessagePassing, aggr='add', flow source→target) 의 eval 계산. 파라미터 이름 같음."""

    def __init__(self, in_channels, out_channels, heads=1, concat=True, beta=False, dropout=0., edge_dim=None, bias=True, root_weight=True):
        super().__init__()
        self.H, self.C, self.concat, self.root_weight = heads, out_channels, concat, root_weight
        self.beta = beta and root_weight; self.dropout = dropout
        self.lin_key = nn.Linear(in_channels, heads * out_channels, bias=bias)
        self.lin_query = nn.Linear(in_channels, heads * out_channels, bias=bias)
        self.lin_value = nn.Linear(in_channels, heads * out_channels, bias=bias)
        self.lin_edge = nn.Linear(edge_dim, heads * out_channels, bias=False) if edge_dim is not None else None
        # A8:247-252: lin_skip 은 root_weight 와 상관없이 만듦(state_dict 에 있음)
        self.lin_skip = nn.Linear(in_channels, heads * out_channels if concat else out_channels, bias=bias)
        self.lin_beta = nn.Linear(3 * heads * out_channels if concat else 3 * out_channels, 1, bias=False) if self.beta else None

    def forward(self, x, edge_index, edge_attr=None):
        H, C = self.H, self.C; N = x.size(0); src, dst = edge_index[0], edge_index[1]
        q = self.lin_query(x).view(-1, H, C); k = self.lin_key(x).view(-1, H, C); v = self.lin_value(x).view(-1, H, C)
        q_i = q.index_select(0, dst); k_j = k.index_select(0, src); v_j = v.index_select(0, src)   # i = 받는 계좌, j = 보낸 이웃
        if self.lin_edge is not None:
            e = self.lin_edge(edge_attr).view(-1, H, C); k_j = k_j + e                       # 키에 거래 피처(A8:281-283)
        a = (q_i * k_j).sum(dim=-1) / math.sqrt(C)
        a = softmax_pt(a, dst, N)                                                           # 받은 이웃끼리 합 1(A8:285)
        msg = v_j
        if self.lin_edge is not None:
            msg = msg + e                                                                    # 값에도 거래 피처(A8:289)
        msg = msg * a.view(-1, H, 1)
        out = scatter_pt(msg, dst, 0, N, 'sum')                                              # aggr='add'
        out = out.view(-1, H * C) if self.concat else out.mean(dim=1)
        if self.root_weight:
            xr = self.lin_skip(x)
            if self.lin_beta is not None:
                b = self.lin_beta(torch.cat([out, xr, out - xr], -1)).sigmoid(); out = b * xr + (1 - b) * out
            else:
                out = out + xr
        return out


# ───────────────────────── 모델(A8:370-479 글자 그대로 · TC/scatter 만 바꿈) ─────────────────────────
def build_model(run, meta):
    """A8:370 salt_build_model(run, G, meta) 의 B(tc2) · C(salt2) 부분. meta = 모델 파일 salt_meta(znorm · delta)."""
    TC = TransformerConvPT; c = RUNS[run]; _scatter = scatter_pt

    def head(H, n_edge, k, pf):
        return nn.Sequential(nn.Linear(2 * H + n_edge, H), nn.ReLU(), nn.Dropout(pf), nn.Linear(H, k))   # A8:375-376

    class ZNorm(nn.Module):
        """A8:378-386 입력 z-표준화. 대상 거래 피처에는 엣지 통계."""
        def __init__(self, s):
            super().__init__()
            for k in ('nm', 'ns', 'em', 'es'):
                self.register_buffer(k, torch.tensor(s[k], dtype=torch.float32))

        def forward(self, x, ea, tx):
            return (x - self.nm) / self.ns, (ea - self.em) / self.es, (tx - self.em) / self.es

    class TC2(nn.Module):
        """A8:403-421 실행 B: h ← (h + drop(ReLU(BN(conv(h)))))/2."""
        def __init__(self, H, heads, p, pf, s, n_node=19, n_edge=17):
            super().__init__()
            self.zn = ZNorm(s); self.node_emb = nn.Linear(n_node, H)
            self.convs = nn.ModuleList([TC(H, H // heads, heads=heads, edge_dim=n_edge, dropout=0.) for _ in range(2)])
            self.bns = nn.ModuleList([nn.BatchNorm1d(H) for _ in range(2)]); self.drop = nn.Dropout(p)
            self.head = head(H, n_edge, K8, pf)

        def forward(self, x, ei, ea, tg, tx):
            x, ea, tx = self.zn(x, ea, tx); h = self.node_emb(x)
            for conv, bn in zip(self.convs, self.bns):
                pre = bn(conv(h, ei, ea)); br = self.drop(torch.relu(pre))
                h = 0.5 * (h + br)
            return self.head(torch.cat([h[tg[0]], h[tg[1]], tx], 1))

    class SALTLayer2(nn.Module):
        """A8:425-459 SALT-Trans 한 층(통계 · attention · 융합)."""
        def __init__(self, H, heads, n_edge, S, p, eps=1e-5):
            super().__init__()
            self.eps = eps
            self.msg_stat = nn.Linear(H, S); self.edge_stat = nn.Linear(n_edge, S, bias=False)
            self.att = TC(H, H // heads, heads=heads, edge_dim=n_edge, dropout=0., root_weight=False)
            self.stat_h = nn.Linear(H, H); self.stat_g = nn.Linear(12 * S, H, bias=False); self.bn_s = nn.BatchNorm1d(H); self.stat_out = nn.Linear(H, H)
            self.fuse_h = nn.Linear(H, H); self.fuse_s = nn.Linear(H, H, bias=False); self.fuse_a = nn.Linear(H, H, bias=False)
            self.bn_f = nn.BatchNorm1d(H); self.fuse_out = nn.Linear(H, H); self.drop = nn.Dropout(p)

        def stats(self, h, ei, ea, delta):                                          # A8:437-446
            src, dst = ei[0], ei[1]
            m = self.msg_stat(h)[src] + self.edge_stat(ea)
            nd, inv = torch.unique(dst, return_inverse=True); M = nd.numel()
            mean = _scatter(m, inv, 0, dim_size=M, reduce='mean')
            std = (_scatter(m * m, inv, 0, dim_size=M, reduce='mean') - mean * mean).clamp(min=1e-5).sqrt()
            mn = _scatter(m, inv, 0, dim_size=M, reduce='min'); mx = _scatter(m, inv, 0, dim_size=M, reduce='max')
            a = torch.cat([mean, std, mn, mx], 1)
            gam = torch.log(torch.bincount(inv, minlength=M).to(h.dtype) + 1).unsqueeze(1)
            return nd, torch.cat([a, a * (gam / delta), a * (delta / gam.clamp(min=self.eps))], 1)

        def forward(self, h, ei, ea, delta):                                        # A8:448-459 (진단 갈래 뺌 — diag=None 일 때와 같음)
            nd, g = self.stats(h, ei, ea, delta)
            s_pre = self.bn_s(self.stat_h(h).index_add(0, nd, self.stat_g(g)))
            h_stat = self.stat_out(torch.relu(s_pre))
            h_att = self.att(h, ei, ea)
            f_pre = self.bn_f(self.fuse_h(h) + self.fuse_s(h_stat) + self.fuse_a(h_att))
            br = self.fuse_out(self.drop(torch.relu(f_pre)))
            return 0.5 * (h + br)

    class SALT2(nn.Module):
        """A8:461-472 실행 C."""
        def __init__(self, H, heads, p, pf, s, delta, n_node=19, n_edge=17):
            super().__init__()
            self.zn = ZNorm(s); self.node_emb = nn.Linear(n_node, H)
            self.layers = nn.ModuleList([SALTLayer2(H, heads, n_edge, H, p) for _ in range(2)])
            self.head = head(H, n_edge, K8, pf); self.register_buffer('salt_delta', torch.tensor(float(delta)))

        def forward(self, x, ei, ea, tg, tx):
            x, ea, tx = self.zn(x, ea, tx); h = self.node_emb(x)
            for layer in self.layers:
                h = layer(h, ei, ea, self.salt_delta)
            return self.head(torch.cat([h[tg[0]], h[tg[1]], tx], 1))

    if c['model'] == 'tc2':
        return TC2(c['H'], c['heads'], c['p'], c['pf'], meta['znorm'])
    if c['model'] == 'salt2':
        return SALT2(c['H'], c['heads'], c['p'], c['pf'], meta['znorm'], meta['delta'])
    raise ValueError(run)


# ───────────────────────── 하루 캐시 배열 → 배치 ─────────────────────────
class MemStore:
    """A8:327-356 Store 와 같은 꼴을 메모리 배열 목록(날 순서)으로. 대상 순서 = 날 순서 · 날 안 묶음 순서."""
    FIELDS = ('nx', 'ei', 'ea', 'tg', 'tx')

    def __init__(self, days_arrays):
        if isinstance(days_arrays, dict):
            days_arrays = [days_arrays]
        self.parts, refs, tpos, base = [], [], [], 0
        for di, A in enumerate(days_arrays):
            self.parts.append(A)
            for g in A['grp']:
                refs.append((di, *g)); tpos.append((base + g[4], base + g[5]))
            base += len(A['y'])
        self.ref = np.array(refs, np.int64); self.tpos = np.array(tpos, np.int64)
        self.y = np.concatenate([np.asarray(A['y']).astype(np.int64) for A in self.parts])
        self.eid = np.concatenate([np.asarray(A['eid']) for A in self.parts])
        assert self.tpos[-1, 1] == len(self.y)

    def batch(self, gids, dev):
        """A8:346-356 Store.batch 와 같은 계산(묶음마다 노드 번호를 밀어 붙임)."""
        xs, eis, eas, tgs, txs, off = [], [], [], [], [], 0
        for g in gids:
            di, n0, n1, e0, e1, t0, t1 = self.ref[g]; A = self.parts[di]
            xs.append(A['nx'][n0:n1]); eis.append(A['ei'][e0:e1] + off); eas.append(A['ea'][e0:e1])
            tgs.append(A['tg'][t0:t1] + off); txs.append(A['tx'][t0:t1]); off += n1 - n0
        t = lambda a, dt: torch.from_numpy(np.ascontiguousarray(a)).to(dev, dt, non_blocking=True)
        return (t(np.concatenate(xs), torch.float32), t(np.concatenate(eis).T, torch.long), t(np.concatenate(eas), torch.float32),
                t(np.concatenate(tgs).T, torch.long), t(np.concatenate(txs), torch.float32))


def check_model_entry(M, key):
    """모델 파일 항목이 이 모듈의 실행 설정과 같은지(A8:1270 assert RUNS[run] == m['cfg'] 와 같은 뜻, 설명 글 desc 는 뺌)."""
    m = M['models'][key]; run = m['run']
    assert run == MODEL_RUN[key], f'{key}: 실행 {run} ≠ {MODEL_RUN[key]}'
    cfg = {k: v for k, v in m['cfg'].items() if k != 'desc'}
    assert cfg == RUNS[run], f'{key}: 실행 설정이 모델 파일과 다르다 {cfg} vs {RUNS[run]}'
    return m, run


def predict_salt(M, key, days_arrays, dev='cpu', seeds=None):
    """key = 'salt_c' | 'tc_b'. days_arrays = 하루 캐시 배열 dict 또는 날 순서 목록.
    반환 (eid int64 [T] 캐시 순서, {seed: P float32 [T, 8]}). 원래 A8:1267-1291 의 반복과 같음(정렬은 호출하는 쪽에서)."""
    m, run = check_model_entry(M, key)
    va = MemStore(days_arrays)
    out = {}
    for seed in (seeds or m['seeds']):
        torch.manual_seed(seed)                                                     # A8:1279
        model = build_model(run, M['salt_meta'])
        model.load_state_dict(m['state_dict'][f's{seed}'], strict=True)            # 이름이 같아 strict 로 읽힘
        model = model.to(dev)
        with torch.no_grad():                                                       # A8:1283-1288 (c2_train.py:75-79 글자 그대로)
            model.eval(); Pc = np.empty((len(va.y), 8), np.float32)
            for b in range(0, len(va.ref), PER_BATCH):
                gids = np.arange(b, min(b + PER_BATCH, len(va.ref))); bt = va.batch(gids, dev)
                Pc[va.tpos[gids[0], 0]:va.tpos[gids[-1], 1]] = torch.softmax(model(*bt).float(), 1).cpu().numpy()
        out[seed] = Pc
        del model
    return va.eid.astype(np.int64), out


def seed_mean(P_by_seed):
    """A8:1356 와 같은 시드 평균: float32 확률 → float64 → 평균(최종 섞기 final_probs 의 입력)."""
    return np.mean([P.astype(np.float64) for P in P_by_seed.values()], 0)


def align(src_eid, want_eid):
    """src_eid 순서의 행을 want_eid 순서로 놓는 인덱스(A8:359-367 align_to_val55 와 같은 뜻 · 집합이 같아야 함)."""
    o = np.argsort(src_eid, kind='stable'); se = src_eid[o]
    i = np.minimum(np.searchsorted(se, want_eid), len(se) - 1)
    if not (se[i] == want_eid).all() or len(src_eid) != len(want_eid):
        raise ValueError(f'행 집합이 다르다: 맞음 {int((se[i] == want_eid).sum())} / 원하는 {len(want_eid)} / 있는 {len(src_eid)}')
    return o[i]
'''

In [ ]:
# @title 코드 ⑦ TGT B0 모델(묶음 aml8.py 4 · 5 구역 + 예측 반복) — 숨김 · 고치지 마세요
_MODULE_SRC['tgt_model'] = r'''"""tgt_model — TGT B0 모델(묶음 aml8.py 4 · 5 구역 글자 그대로) + 메모리 하루 캐시 + 예측 반복. 노트북용 · 2026-10-17 · Claude Code.

원래: 모델묶음_유형8_최종후보_20261016/aml8.py 줄 483-1163 (aml8.py sha256 9f628c02c4ecdca1 · 이 구간 sha256 e2cbfe69004a77e7).
  그 안의 `# ── 원래:` 줄이 팀 원본(tgt_model.py · train_tgt.py · common_t.py)의 file:line 이다.
  B0(옵션 모두 끔 · 출력 8)가 실제로 쓰는 것: TE · ncoe_counts · TGTND · TGTND8 · tgt_build_model. 순수 torch(PyG 없음).
맨 끝 [노트북] 구역: aml8.py:1167-1196 DayCache 와 같은 batch · order_of 를 메모리 배열로(MemDayCache), aml8.py:1294-1323 예측 반복(predict_tgt).
"""
from __future__ import annotations  # aml8.py:20 (파일 맨 위에만 둘 수 있어 여기로)

# ═════════════════ 4. TGT 모델 — 팀 기법실행 tgt_model.py 전체 ═════════════════
# ── 원래: HI-Large_EDA/Transformer모델/기법실행_20261008/code/tgt_model.py 줄 1-657 · sha256 b23dbf63a27e13c7 — 파일 전체. TGTND8 이 옵션 가지마다 이 파일의 다른 정의를 불러 잘라 내지 않았다. B0(옵션 모두 끔 · 출력 8)가 실제로 쓰는 것: TE · ncoe_counts · TGTND · N_NODE/N_EDGE · _sub_linear_rows · TGTND8
"""TGT-ND 모델 — 계좌-일 시퀀스 인코더 + 대상 질의 교차 attention + 9클래스 헤드. 2026-10-04 · Claude 작성.
설계 ../설계_TGT_맞춤안_20261002.md 3.1(반박 검토 반영판).

  토큰 h_j = Linear(x17) + 방향 임베딩 + Linear(TE(dt_j)) + 24시간 구간 임베딩        (위치 인코딩 없음 → 같은 분 안 순서에 불변)
  계좌-일 인코더: 2층 pre-LN Transformer(d 64 · head 4 · FF 128) → H_a, 평균 h_a
  질의 q_e = MLP[Linear(엣지 17) + Linear(node_x_u ‖ node_x_v) + Linear(TE(qdt_u) ‖ TE(qdt_v))]
  키 · 값 = LN[H + 쪽 임베딩 + MLP(NCoE 2열) + Linear(TE(t_j − t_e))]   → 교차 attention(1층 · head 4) → c_e
  헤드 MLP[c_e ‖ q_e ‖ h_u ‖ h_v ‖ Linear(NCoE 요약 3)] → 9 로짓
time_on=False 면 dt · dtt · qdt · 구간을 모두 0 으로 둔다(설계 6절 3 (c) '시간 끔' 팔).
pc_dim > 0 (10-05 시도): 질의에 Linear(대상 거래 패턴 개수 pc_dim 열)을 더한다. 이 층은 맨 마지막에 만들어 다른 층의 초기값은 pc_dim = 0 과 같다.
[10-06 비교 실험 — 사용자 승인] 새 층을 만들지 않으므로 초기값 · 기본 동작은 그대로다.
  te_fixed=True(B 팔): 세 TE 의 주파수 ω 를 학습하지 않는다(초기값 1분 ~ 10만 분 주기에 고정, TIDFormer 절제 근거).
  no_ncoe=True(C 팔): NCoE 2열과 헤드의 NCoE 요약 3개를 0 으로 둔다(정보만 끄고 층은 남김).
"""
# from __future__ import annotations  # [묶음] 파일 맨 위로 옮김(파이썬은 __future__ 를 맨 위에만 허용)

import math

import torch
import torch.nn as nn


class TE(nn.Module):
    """상대 시간 인코딩 [cos(ωΔt), sin(ωΔt)] · 학습 주파수(초기 주기 1분 ~ 10만 분). sin 이 부호(전/후)를 보존한다."""

    def __init__(self, n: int = 16):
        super().__init__()
        self.w = nn.Parameter(torch.tensor([2 * math.pi / 10 ** (5 * i / (n - 1)) for i in range(n)], dtype=torch.float32))

    def forward(self, dt: torch.Tensor) -> torch.Tensor:
        z = dt.float().unsqueeze(-1) * self.w
        return torch.cat([torch.cos(z), torch.sin(z)], -1)


def ncoe_counts(cp_u, cp_v, m_u, m_v):
    """DyGFormer 식 이웃 동시출현: 토큰마다 [S_u 안 같은 상대 수, S_v 안 같은 상대 수] (두 열을 이어 붙여 송·수신 비대칭 유지)."""
    allcp = torch.cat([cp_u, cp_v], 1)                                                       # [B, 2K]
    cu = ((allcp.unsqueeze(2) == cp_u.unsqueeze(1)) & m_u.unsqueeze(1)).sum(-1)
    cv = ((allcp.unsqueeze(2) == cp_v.unsqueeze(1)) & m_v.unsqueeze(1)).sum(-1)
    return torch.stack([cu, cv], -1).float()                                                  # [B, 2K, 2]


class TGTND(nn.Module):
    def __init__(self, d: int = 64, heads: int = 4, layers: int = 2, ff: int = 128, drop: float = 0.1, time_on: bool = True, ckpt: bool = True,
                 pc_dim: int = 0, te_fixed: bool = False, no_ncoe: bool = False):
        super().__init__()
        self.time_on, self.ckpt = time_on, ckpt                          # ckpt: 학습 때 인코더 층 gradient checkpointing(배치 4,096 을 GPU 3.5 GiB 안에)
        self.tok = nn.Linear(17, d); self.dir = nn.Embedding(3, d); self.bkt = nn.Embedding(4, d)
        self.te_tok = TE(16); self.te_tok_p = nn.Linear(32, d)
        layer = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.side = nn.Embedding(2, d)
        self.ncoe = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, d))
        self.te_key = TE(16); self.te_key_p = nn.Linear(32, d); self.kv_norm = nn.LayerNorm(d)
        self.q_edge = nn.Linear(17, d); self.q_node = nn.Linear(38, d); self.te_q = TE(16); self.q_te_p = nn.Linear(64, d)
        self.q_mlp = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
        self.xattn = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.ns = nn.Linear(3, 16)
        self.head = nn.Sequential(nn.LayerNorm(4 * d + 16), nn.Linear(4 * d + 16, 128), nn.GELU(), nn.Dropout(drop), nn.Linear(128, 9))
        self.pc_dim = pc_dim
        if pc_dim:                                                                            # 맨 마지막(위 층들의 초기값을 바꾸지 않게)
            self.q_pc = nn.Linear(pc_dim, d)
        self.te_fixed, self.no_ncoe = te_fixed, no_ncoe                                       # [10-06] 새 매개변수 없음 → 난수 소비 · 초기값 그대로
        if te_fixed:
            for t_ in (self.te_tok, self.te_key, self.te_q):
                t_.w.requires_grad_(False)

    def forward(self, b: dict) -> torch.Tensor:
        z = (lambda t: t if self.time_on else torch.zeros_like(t))
        mask = b['mask']                                                                       # [A, K] True = 있음
        h = self.tok(b['x'].float()) + self.dir(b['dir'].long()) + self.te_tok_p(self.te_tok(z(b['dt']))) + self.bkt(z(b['bkt'].long()))
        if self.ckpt and self.training:                                                        # 값은 같고 메모리만 줄인다(역전파 때 다시 계산)
            for layer in self.enc.layers:
                h = torch.utils.checkpoint.checkpoint(layer, h, None, ~mask, use_reentrant=False)
            H = h
        else:
            H = self.enc(h, src_key_padding_mask=~mask)                                        # [A, K, d]
        mf = mask.unsqueeze(-1).to(H.dtype)
        mean = (H * mf).sum(1) / mf.sum(1).clamp(min=1)                                       # [A, d]
        iu, iv = b['inv'][:, 0], b['inv'][:, 1]
        m_u, m_v = mask[iu], mask[iv]; cp_u, cp_v = b['cp'][iu], b['cp'][iv]
        B, K = m_u.shape
        cnt = ncoe_counts(cp_u, cp_v, m_u, m_v)
        if self.no_ncoe:                                                                      # [10-06 C 팔]
            cnt = torch.zeros_like(cnt)
        kv = torch.cat([H[iu] + self.side.weight[0], H[iv] + self.side.weight[1]], 1)         # [B, 2K, d]
        kv = kv + self.ncoe(torch.log1p(cnt)).to(kv.dtype) + self.te_key_p(self.te_key(z(b['dtt'].reshape(B, 2 * K)))).to(kv.dtype)
        kv = self.kv_norm(kv)
        qte = self.te_q(z(b['qdt'])).reshape(B, -1)                                             # [B, 64]
        qin = self.q_edge(b['tx']) + self.q_node(torch.cat([b['nu'], b['nv']], 1)) + self.q_te_p(qte)
        if self.pc_dim:
            qin = qin + self.q_pc(b['pc'])
        q = self.q_mlp(qin)
        c, _ = self.xattn(q.unsqueeze(1), kv, kv, key_padding_mask=~torch.cat([m_u, m_v], 1), need_weights=False)
        u, v = b['uv'][:, 0], b['uv'][:, 1]
        ns = torch.stack([((cp_u == v.unsqueeze(1)) & m_u).sum(1), ((cp_v == u.unsqueeze(1)) & m_v).sum(1),
                          ((cnt[:, :K, 1] > 0) & m_u).sum(1)], 1).float()                       # [S_u 안 v 횟수, S_v 안 u 횟수, 공통 상대 토큰 수]
        if self.no_ncoe:                                                                      # [10-06 C 팔]
            ns = torch.zeros_like(ns)
        zz = torch.cat([c.squeeze(1).float(), q.float(), mean[iu].float(), mean[iv].float(), self.ns(torch.log1p(ns)).float()], 1)   # 혼합정밀에서 dtype 을 맞춘다
        return self.head(zz)



# ═══════════════════════ [2026-10-09 기법실행_20261008 추가 — 위 TGTND · TE · ncoe_counts 는 원본(TGT/code/tgt_model.py)과 한 글자도 다르지 않다] ═══════════════════════
# TGTND8 = TGTND + 플래그로 켜는 변형(Case 1-A · 1-B · 2-A · 2-B · 2-C). 모든 옵션을 끄면 forward 의 연산 순서까지 TGTND 와 같다(test_flags.py 1번 검사: 초기값 · 로짓 차이 0).
# 초기값 원칙: 기준 층을 기준과 같은 순서로 먼저 만들고(super().__init__ → 같은 시드면 같은 난수), 열 제거는 난수를 쓰지 않는 '잘라 내기'로,
#   새 인코더(FT · MLP)는 그 뒤에 만든다. 그래서 바꾸지 않는 층의 초기값은 B0 와 비트 단위로 같다(짝 비교에서 시드 잡음을 줄이려고).
#
# 열 제거(--drop … --drop-mode remove, 기본):
#   입력에서 열을 빼고(index_select) 첫 사영 층(q_edge 17→d · q_node 38→d · tok 17→d)을 남는 열 수로 줄인다. 줄인 층의 가중치는 B0 와 같은 난수로
#   만든 전체 모양에서 남는 열만 잘라 쓴다. Linear 첫 층에서는 '0 으로 만들기'와 수학적으로 같다(빠진 열 가중치의 기울기가 늘 0 → Adam 갱신 0 ·
#   기울기 자르기 노름 기여 0). 차이는 파라미터 수가 (빠진 열 수 × d) 만큼 줄어드는 것뿐이고, 그것은 '입력이 줄었다'는 Case 정의 그 자체라
#   비교 가능성을 해치지 않는다고 본다(추정). FT · MLP 입력부에서는 0 을 넣은 열도 토큰 b_j 또는 은닉 편향으로 남으므로 '0 = 빼기'가 아니다 → 제거가 맞다.
#   --drop-mode zero 는 비교용(열에 0 을 곱함 · 모양 그대로).
# 모델 안 입력 제거(--drop-te dt,dtt,qdt · --drop-bucket · --drop-ncoe, --internal-mode remove 기본):
#   그 항을 아예 더하지 않는다(쓰지 않는 층은 지운다). ncoe 제거는 키·값의 NCoE 2열 항과 헤드의 요약 3열(ns → 16차원)을 모두 빼고, 헤드 첫 LayerNorm ·
#   Linear 의 입력을 272 → 256 으로 잘라 낸다(잘라 내기 · 난수 없음).
#   --internal-mode zero 는 기존 --time-off · --no-ncoe 와 같은 '0 으로 만들기'(학습되는 상수 항이 남음 — 10-06 C 팔과 같은 방식).
import importlib
import sys
from contextlib import contextmanager

import numpy as np
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as _ckpt

N_NODE, N_EDGE = 19, 17
KYC_NODE = list(range(10, 16))                     # node_x 10~15 = KYC 유형 원핫(aml_feature_schema.csv:12-17 · 확인_1/1-1_피처분류.csv 행 10~15)

# ── FT-Transformer · MLP 입력부(기법실행_20261008/bench/input_encoders.py 의 정의를 그대로 옮김 — 1-3 CPU 추정에 쓴 것과 같은 구조)
# FT 기본값 근거(추정): Gorishniy et al. 2021 · Grinsztajn 2022 표가 옮겨 적은 FT 기본값(bench/input_encoders.py:12-18).
FT_PRESETS = {
    'paper': dict(d_token=192, n_layers=3, n_heads=8, ffn_factor=4 / 3, attn_drop=0.2, ffn_drop=0.1, res_drop=0.0),   # 논문 기본값 옮김(추정)
    'tgtw': dict(d_token=64, n_layers=3, n_heads=4, ffn_factor=4 / 3, attn_drop=0.2, ffn_drop=0.1, res_drop=0.0),     # TGT 폭(d 64 · head 4)에 맞춘 판(추정 · 제안)
    'small': dict(d_token=32, n_layers=2, n_heads=4, ffn_factor=4 / 3, attn_drop=0.2, ffn_drop=0.1, res_drop=0.0),    # 위치 ②(토큰) 메모리 대비 작은 판(추정 · 제안)
}


@contextmanager
def _keep_rng():
    """이 안에서 만든 층은 CPU 난수 상태를 소비하지 않은 것으로 되돌린다(잘라 낸 층을 만들 때만 씀)."""
    st = torch.get_rng_state()
    try:
        yield
    finally:
        torch.set_rng_state(st)


def _sub_linear(lin: nn.Linear, cols) -> nn.Linear:
    with _keep_rng():
        new = nn.Linear(len(cols), lin.out_features, bias=lin.bias is not None)
    with torch.no_grad():
        new.weight.copy_(lin.weight[:, list(cols)])
        if lin.bias is not None:
            new.bias.copy_(lin.bias)
    return new


def _sub_layernorm(ln: nn.LayerNorm, cols) -> nn.LayerNorm:
    with _keep_rng():
        new = nn.LayerNorm(len(cols), eps=ln.eps)
    with torch.no_grad():
        new.weight.copy_(ln.weight[list(cols)]); new.bias.copy_(ln.bias[list(cols)])
    return new


class ReGLU(nn.Module):
    def forward(self, x):
        a, b = x.chunk(2, -1)
        return a * F.relu(b)


class FTBlock(nn.Module):
    """PreNorm Transformer 층 하나. first=True 면 attention 앞 LayerNorm 을 뺀다(논문 p.5). cls_only=True 면 [CLS] 만 질의(rtdl 기본 · 추정)."""

    def __init__(self, d, heads, dh, attn_drop, ffn_drop, res_drop, first, cls_only):
        super().__init__()
        self.attn_norm = None if first else nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=attn_drop, batch_first=True)
        self.ffn_norm = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 2 * dh), ReGLU(), nn.Dropout(ffn_drop), nn.Linear(dh, d))
        self.res_drop = nn.Dropout(res_drop)
        self.cls_only = cls_only

    def forward(self, x):
        xn = x if self.attn_norm is None else self.attn_norm(x)
        q = xn[:, :1] if self.cls_only else xn
        if self.cls_only:
            x = x[:, :1]
        a, _ = self.attn(q, xn, xn, need_weights=False)
        x = x + self.res_drop(a)
        return x + self.res_drop(self.ffn(self.ffn_norm(x)))


class FTEncoder(nn.Module):
    """[..., n_feat] 수치 → [..., d_out]. 수치 토큰 T_j = b_j + x_j·W_j · [CLS] 를 붙여 PreNorm L층 · 출력 = Linear(ReLU(LayerNorm(T_CLS))).
    cat_groups: 배타적 원핫 열 묶음들 → 묶음마다 범주 토큰 1개(룩업 + 편향)."""

    def __init__(self, n_feat, d_out, d_token, n_layers, n_heads, ffn_factor, attn_drop, ffn_drop, res_drop, ckpt=False, cat_groups=()):
        super().__init__()
        s = 1 / math.sqrt(d_token)                                   # rtdl 'uniform' 토큰 초기화(추정)
        cat_cols = {c for g in cat_groups for c in g}
        num_idx = [j for j in range(n_feat) if j not in cat_cols]
        self.n_cat = len(cat_groups)
        self.weight = nn.Parameter(torch.empty(len(num_idx), d_token).uniform_(-s, s))
        self.bias = nn.Parameter(torch.empty(len(num_idx) + self.n_cat, d_token).uniform_(-s, s))
        self.cls = nn.Parameter(torch.empty(d_token).uniform_(-s, s))
        if self.n_cat:
            self.register_buffer('num_idx', torch.tensor(num_idx, dtype=torch.long), persistent=False)
            self.register_buffer('cat_cols', torch.tensor([c for g in cat_groups for c in g], dtype=torch.long), persistent=False)
            self.cat_sizes = [len(g) for g in cat_groups]
            self.register_buffer('cat_off', torch.tensor([0] + list(np.cumsum(self.cat_sizes)[:-1]), dtype=torch.long), persistent=False)
            self.emb = nn.Embedding(sum(self.cat_sizes), d_token)
            nn.init.uniform_(self.emb.weight, -s, s)
        dh = int(d_token * ffn_factor)
        self.blocks = nn.ModuleList([FTBlock(d_token, n_heads, dh, attn_drop, ffn_drop, res_drop, i == 0, i == n_layers - 1) for i in range(n_layers)])
        self.out = nn.Sequential(nn.LayerNorm(d_token), nn.ReLU(), nn.Linear(d_token, d_out))
        self.ckpt = ckpt                                             # 학습 때 층별 gradient checkpointing(값 같음 · 메모리만 줄임)

    def forward(self, x):
        lead = x.shape[:-1]
        x = x.reshape(-1, x.shape[-1]).float()
        if self.n_cat:
            xn = x.index_select(1, self.num_idx)
            oh = x.index_select(1, self.cat_cols).split(self.cat_sizes, 1)
            ci = torch.stack([g.argmax(1) for g in oh], 1) + self.cat_off              # [N, n_cat] 배타적 원핫 → 범주 번호
            t = torch.cat([xn.unsqueeze(-1) * self.weight, self.emb(ci)], 1) + self.bias
        else:
            t = x.unsqueeze(-1) * self.weight + self.bias             # [N, n_feat, d]
        t = torch.cat([self.cls.expand(t.shape[0], 1, -1), t], 1)
        for blk in self.blocks:
            t = _ckpt(blk, t, use_reentrant=False) if (self.ckpt and self.training) else blk(t)
        return self.out(t[:, 0]).reshape(*lead, -1)


class MLPEncoder(nn.Module):
    """2-B 대조군: Linear-ReLU-Dropout × n_layers → Linear(d_out). 은닉 폭은 solve_hidden 으로 짝 FT 와 파라미터 수를 맞춘다."""

    def __init__(self, n_in, d_out, hidden, n_layers=3, drop=0.1):
        super().__init__()
        layers_, d = [], n_in
        for _ in range(n_layers):
            layers_ += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(drop)]
            d = hidden
        layers_.append(nn.Linear(d, d_out))
        self.net = nn.Sequential(*layers_)

    def forward(self, x):
        return self.net(x.float())


def mlp_params(n_in, d_out, h, n_layers):
    return n_in * h + h + (n_layers - 1) * (h * h + h) + h * d_out + d_out


def solve_hidden(n_in, d_out, n_layers, target):
    return min(range(1, 4096), key=lambda h: abs(mlp_params(n_in, d_out, h, n_layers) - target))


def ft_params(n_feat, d_out, d_token, n_layers, n_heads, ffn_factor, cat_groups=(), **_):
    dh = int(d_token * ffn_factor); d = d_token
    n_cat_cols = sum(len(g) for g in cat_groups); n_num = n_feat - n_cat_cols; n_cat = len(cat_groups)
    tok = 2 * n_num * d + n_cat * d + n_cat_cols * d
    blk = (4 * d * d + 4 * d) + 2 * d + (d * 2 * dh + 2 * dh) + (dh * d + d)
    return tok + d + n_layers * blk + (n_layers - 1) * 2 * d + 2 * d + d * d_out + d_out


# ═══════════════════════ [2026-10-10 추가 — 2-D TabNet · 4-A Mamba · 4-B PatchTST · Case 6] ═══════════════════════
# 모두 '기준 층을 먼저 만들고 새 층은 그 뒤에' 원칙(위 머리말)을 그대로 따른다 → 바꾸지 않는 층의 초기값은 B0 와 비트 단위로 같다.
# 공식 패키지는 쓸 때만 불러온다(기본 경로 B0 는 triton · mamba_ssm · pytorch_tabnet 을 import 하지 않음).
TABNET_PATH = None  # [묶음] 원래 '/workspace/.pylibs/tabnet' — TabNet 입력부 전용 · B0 에서는 쓰지 않음
MAMBA_PATH = None  # [묶음] 원래 '/workspace/.pylibs/mamba' — Mamba 시퀀스 인코더 전용 · B0 에서는 쓰지 않음

# 2-D TabNet 크기. pkg = pytorch-tabnet 기본값(tab_network.py:42-56 TabNetEncoder 인자 기본값 · abstract_model.py:56-57 momentum 0.02 · lambda_sparse 1e-3).
#   matched = pkg 에서 n_d = n_a 만 바꿔 같은 자리 FT(--ft-q-preset)의 파라미터 수에 가장 가깝게(2-B MLP 의 맞춤과 같은 방식 · 제안).
TABNET_PKG = dict(n_d=8, n_a=8, n_steps=3, gamma=1.3, n_independent=2, n_shared=2, epsilon=1e-15, virtual_batch_size=128, momentum=0.02,
                  mask_type='sparsemax')
# 4-A Mamba 크기(Mamba-1 블록 = mamba_ssm.Mamba · mamba_simple.py:32-50 인자). official = 공식 기본값(d_state 16 · d_conv 4 · expand 2) ·
#   matched = expand 1(혼합층 파라미터가 B0 self-attention 과 거의 같아짐 — 16,320 vs 16,640 · 제안).
MAMBA_PRESETS = {'official': dict(d_state=16, d_conv=4, expand=2), 'matched': dict(d_state=16, d_conv=4, expand=1)}


def _import_from(path, mod):
    if path not in sys.path:
        sys.path.insert(0, path)
    return importlib.import_module(mod)


def _sub_linear_rows(lin: nn.Linear, rows) -> nn.Linear:
    """출력 행만 남긴 Linear(난수 소비 없음). Case 6 출력층 9 → 7."""
    with _keep_rng():
        new = nn.Linear(lin.in_features, len(rows), bias=lin.bias is not None)
    with torch.no_grad():
        new.weight.copy_(lin.weight[list(rows)])
        if lin.bias is not None:
            new.bias.copy_(lin.bias[list(rows)])
    return new


class TabNetQ(nn.Module):
    """2-D: 공식 pytorch_tabnet.tab_network.TabNetEncoder + 마지막 사상 Linear(n_d → d_out, bias 없음).
    TabNetNoEmbeddings(tab_network.py:399-502)와 같은 구성이다. 다만 TabNetNoEmbeddings 가 만들고 쓰지 않는 initial_bn(:451 · forward 는 :490-492 에서
    encoder 의 initial_bn 만 씀)은 빼서 '기울기 없는 파라미터'가 생기지 않게 했다. forward 는 출력과 함께 희소 정규화 항 M_loss 를 self.aux 에 남긴다
    (공식 학습식 loss − λ_sparse · M_loss, abstract_model.py:527-531 — 학습 루프에서 --tabnet-lambda-sparse 로 더함).
    group_attention_matrix: 공식 코드는 일반 텐서 속성이라 .to(device) 에 따라가지 않는다(공식 TabModel 은 만들 때 device 로 옮김, abstract_model.py:621) → 버퍼로 바꿔 둔다."""

    def __init__(self, n_in, d_out, groups=(), **cfg):
        super().__init__()
        TN = _import_from(TABNET_PATH, 'pytorch_tabnet.tab_network'); TU = _import_from(TABNET_PATH, 'pytorch_tabnet.utils')
        gm = TU.create_group_matrix([list(g) for g in groups], n_in) if groups else None
        self.enc = TN.TabNetEncoder(input_dim=n_in, output_dim=d_out, group_attention_matrix=gm, **cfg)
        G = self.enc.group_attention_matrix.detach().clone().float(); del self.enc.group_attention_matrix
        self.enc.register_buffer('group_attention_matrix', G, persistent=False)
        self.final_mapping = nn.Linear(cfg['n_d'], d_out, bias=False)
        TN.initialize_non_glu(self.final_mapping, cfg['n_d'], d_out)       # TabNetNoEmbeddings :486-487 와 같은 초기화
        self.aux = None

    def forward(self, x):
        steps, m_loss = self.enc(x.float())
        self.aux = m_loss
        return self.final_mapping(torch.sum(torch.stack(steps, 0), 0))


def tabnet_params(n_in, d_out, groups=(), **cfg) -> int:
    with _keep_rng():
        return sum(p.numel() for p in TabNetQ(n_in, d_out, groups, **cfg).parameters())


def solve_tabnet_width(n_in, d_out, groups, base_cfg, target) -> int:
    """n_d = n_a = w 만 바꿔 파라미터 수가 target 에 가장 가까운 w(1~256). 만들어서 센다(난수 상태는 되돌림)."""
    best, bw = None, None
    for w in range(1, 257):
        n = tabnet_params(n_in, d_out, groups, **dict(base_cfg, n_d=w, n_a=w))
        if best is None or abs(n - target) < best:
            best, bw = abs(n - target), w
        if n > target and best is not None and n - target > best:
            break
    return bw


def _mamba_ref_forward(m, hs):
    """CPU 검사용 경로: 공식 Mamba.forward(mamba_simple.py:120-206)의 느린 경로를 같은 순서로 따르고, CUDA 커널 두 곳만 공식 패키지 안의 대체로 바꾼다
    (causal_conv1d_fn 이 없을 때의 nn.Conv1d 분기 = mamba_simple.py:164-165 · selective_scan_fn → 공식 참조 구현 selective_scan_ref = selective_scan_interface.py:115).
    GPU 에서는 이 함수를 쓰지 않는다(공식 fused 경로 mamba_inner_fn). [10-10 r2 고침] 두 경로가 같은 값인지는 **확인하지 못했다(미확인)** — 대조 함수 bench/smoke_bench_new_encoders.py 의 mamba_kernel_check 가 V100(sm_70)에서 첫 CUDA 호출(causal_conv1d_fwd 'no kernel image')에서 멈춰 대조 전에 끝났다(bench/results_new/smoke_mamba_1009_142244.json "steps": [] · _오류.txt). 앞 판 문구 "GPU 스모크에서 대조한다(bench/smoke_new_encoders.py)"의 파일 이름은 없는 파일이었다. CPU 에서 확인한 것은 이 함수 = 공식 Mamba.forward 의 '커널 → 공식 참조 구현' 분기(test_flags_v3.log 7절)뿐이다."""
    sel = _import_from(MAMBA_PATH, 'mamba_ssm.ops.selective_scan_interface')
    batch, seqlen, dim = hs.shape
    xz = (m.in_proj.weight @ hs.reshape(batch * seqlen, dim).t()).reshape(-1, batch, seqlen).permute(1, 0, 2)    # "d (b l) -> b d l"
    if m.in_proj.bias is not None:
        xz = xz + m.in_proj.bias.to(dtype=xz.dtype)[:, None]
    A = -torch.exp(m.A_log.float())
    x, z = xz.chunk(2, dim=1)
    x = m.act(m.conv1d(x)[..., :seqlen])
    x_dbl = m.x_proj(x.permute(0, 2, 1).reshape(batch * seqlen, -1))                                              # "b d l -> (b l) d"
    dt, B, C = torch.split(x_dbl, [m.dt_rank, m.d_state, m.d_state], dim=-1)
    dt = (m.dt_proj.weight @ dt.t()).reshape(-1, batch, seqlen).permute(1, 0, 2)
    B = B.reshape(batch, seqlen, -1).permute(0, 2, 1).contiguous(); C = C.reshape(batch, seqlen, -1).permute(0, 2, 1).contiguous()
    y = sel.selective_scan_ref(x, dt, A, B, C, m.D.float(), z=z, delta_bias=m.dt_proj.bias.float(), delta_softplus=True)
    return m.out_proj(y.permute(0, 2, 1))


def mamba_apply(m, hs):
    """GPU = 공식 Mamba.forward(fused CUDA 경로) · CPU = _mamba_ref_forward(공식 참조 구현)."""
    return m(hs) if hs.is_cuda else _mamba_ref_forward(m, hs)


def _rev_index(mask):
    """실제 칸(앞쪽 n 칸)만 뒤집는 색인(패딩 칸은 제자리). mask 는 '앞 n 칸 = 실제' 꼴이어야 한다(tgt_data.seqs · seqs_past)."""
    K = mask.shape[1]; n = mask.sum(1); ar = torch.arange(K, device=mask.device)
    return torch.where(ar[None] < n[:, None], n[:, None] - 1 - ar[None], ar[None].expand(len(n), K))


def _gather_t(x, idx):
    return torch.gather(x, 1, idx.unsqueeze(-1).expand(-1, -1, x.shape[-1]))


class MambaEncLayer(nn.Module):
    """4-A: B0 의 TransformerEncoderLayer(norm_first · d 64 · FF 128 · ReLU · dropout 0.1)에서 self-attention 자리만 Mamba 로 바꾼 층.
    x = x + drop(Mamba(LN(x))) ; x = x + drop(FF(LN(x))) — FF · LN · dropout 은 B0 층과 같은 구성(새로 만든 층이라 초기값은 다름).
    bi=True: 앞→뒤 Mamba + 뒤→앞 Mamba(실제 칸만 뒤집어 넣고 되돌림 · 방향마다 따로인 파라미터 — Vim 식, 추정) 를 더한다.
    uni(인과): 칸 j 의 출력은 칸 ≤ j 만 본다. 패딩은 맨 뒤라 실제 칸 출력에 영향이 없다(test_flags 검사)."""

    def __init__(self, d, ff, drop, mcfg, bi, n_layers):
        super().__init__()
        MS = _import_from(MAMBA_PATH, 'mamba_ssm.modules.mamba_simple'); MX = _import_from(MAMBA_PATH, 'mamba_ssm.models.mixer_seq_simple')
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.lin1, self.lin2 = nn.Linear(d, ff), nn.Linear(ff, d)
        self.drop, self.drop1, self.drop2 = nn.Dropout(drop), nn.Dropout(drop), nn.Dropout(drop)
        self.mix = MS.Mamba(d, **mcfg)
        self.mix_b = MS.Mamba(d, **mcfg) if bi else None
        for m_ in (self.mix, self.mix_b):                                   # 공식 모델(MixerModel)과 같은 초기화 함수(mixer_seq_simple.py:86-115 · FF 가 있어 잔차 2개/층)
            if m_ is not None:
                m_.apply(lambda mod: MX._init_weights(mod, n_layer=n_layers, n_residuals_per_layer=2))

    def forward(self, x, mask):
        h = self.norm1(x)
        y = mamba_apply(self.mix, h)
        if self.mix_b is not None:
            ri = _rev_index(mask)
            y = y + _gather_t(mamba_apply(self.mix_b, _gather_t(h, ri)), ri)
        x = x + self.drop1(y)
        return x + self.drop2(self.lin2(self.drop(F.relu(self.lin1(self.norm2(x))))))


class MambaSeqEncoder(nn.Module):
    def __init__(self, d, ff, drop, layers, mcfg, bi):
        super().__init__()
        self.layers = nn.ModuleList([MambaEncLayer(d, ff, drop, mcfg, bi, layers) for _ in range(layers)])

    def forward(self, h, mask, ckpt):
        n = mask.sum(1); ar = torch.arange(mask.shape[1], device=mask.device)
        assert bool((mask == (ar[None] < n[:, None])).all()), 'Mamba 는 실제 칸이 앞쪽에 모인 시퀀스만 받는다(tgt_data 규칙)'
        for lay in self.layers:
            h = _ckpt(lay, h, mask, use_reentrant=False) if ckpt else lay(h, mask)
        return h


class PatchSeqEncoder(nn.Module):
    """4-B: PatchTST(Nie et al. 2023, arXiv:2211.14730)의 '패칭' — 연속 P 칸을 한 토큰으로(보폭 S) · 학습 위치 임베딩 · Transformer 인코더.
    이 자리에 맞춘 점(추정 · 논문과 다른 점은 실행명령.md 11절):
      - 입력은 B0 와 같은 토큰 임베딩 h(64차원)이고, 패치 하나 = P 칸 × 64 를 펼쳐 Linear(P·64 → 64). 채널 독립(CI)은 넣지 않았다.
      - 칸 수가 계좌마다 다르므로 최근 거래가 맨 끝에 오게 오른쪽 정렬한 뒤 앞에서부터 자른다(패치 위치 = '가장 최근에서 몇 번째 묶음').
        실제 칸이 하나도 없는 패치는 attention 키에서 가린다. 패딩 칸 값은 0 으로 만든 뒤 묶는다 → 실제 칸 출력은 패딩 값과 무관(test_flags 검사).
      - 층은 B0 인코더와 같은 nn.TransformerEncoderLayer(d 64 · head 4 · FF 128 · pre-LN). 논문 공식 코드의 BatchNorm 대신 LayerNorm(패딩 패치 때문).
      - 칸별 출력이 필요하므로(교차 attention 키 · 평균) 칸 j 출력 = h_j + Linear(z_패치(j)) (겹치는 패치는 평균) 로 되돌린다."""

    def __init__(self, d, heads, ff, drop, layers, K, P, S):
        super().__init__()
        assert 1 <= S <= P <= K and (K - P) % S == 0, f'패치 길이 {P} · 보폭 {S} 가 {K} 칸과 맞지 않는다((K − P) 가 S 의 배수여야 함)'
        self.K, self.P, self.S, self.N = K, P, S, (K - P) // S + 1
        self.emb = nn.Linear(P * d, d)
        self.pos = nn.Parameter(torch.empty(self.N, d).uniform_(-0.02, 0.02))    # PatchTST 공식 코드 positional_encoding('zeros', learn_pe=True) 와 같은 초기화(추정 · 원문 코드 미대조)
        self.pdrop = nn.Dropout(drop)
        layer = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.unpatch = nn.Linear(d, d)
        cnt = torch.zeros(K)
        for p in range(self.N):
            cnt[p * S:p * S + P] += 1
        self.register_buffer('cnt', cnt, persistent=False)

    def forward(self, h, mask, ckpt):
        A_, K, d = h.shape; assert K == self.K
        n = mask.sum(1); ar = torch.arange(K, device=h.device)
        assert bool((mask == (ar[None] < n[:, None])).all()), 'PatchTST 판은 실제 칸이 앞쪽에 모인 시퀀스만 받는다(tgt_data 규칙)'
        shift = (K - n)[:, None]
        gi = (ar[None] - shift) % K                                              # 오른쪽 정렬: 굴린 자리 j ← 원래 칸 gi[a, j]
        hz = h * mask.unsqueeze(-1).to(h.dtype)
        hr = _gather_t(hz, gi); mr = torch.gather(mask, 1, gi)
        pt = hr.unfold(1, self.P, self.S).permute(0, 1, 3, 2).reshape(A_, self.N, self.P * d)
        pv = mr.unfold(1, self.P, self.S).any(-1)                                 # [A, N] 실제 칸이 있는 패치
        z = self.pdrop(self.emb(pt) + self.pos)
        if ckpt:
            for lay in self.enc.layers:
                z = _ckpt(lay, z, None, ~pv, use_reentrant=False)
        else:
            z = self.enc(z, src_key_padding_mask=~pv)
        u = self.unpatch(z)
        acc = torch.zeros(A_, K, d, device=h.device, dtype=u.dtype)
        for p in range(self.N):
            acc[:, p * self.S:p * self.S + self.P] += u[:, p:p + 1]
        outr = hr + acc / self.cnt[None, :, None].to(acc.dtype)
        return _gather_t(outr, (ar[None] + shift) % K)                          # 원래 자리로 되돌림


class TGTND8(TGTND):
    """TGTND + Case 플래그. 인자 설명은 train_tgt.py --help 와 모듈 머리말(이 블록 위) 참고."""

    def __init__(self, drop_node=(), drop_edge_q=(), drop_edge_tok=(), drop_mode='remove', drop_te=(), drop_bkt=False, drop_ncoe=False,
                 internal_mode='remove', q_enc='linear', q_cols=None, q_cfg=None, tok_enc='linear', tok_cfg=None, mlp_layers=3, mlp_drop=0.1,
                 mlp_q_hidden=None, mlp_tok_hidden=None, q_ckpt=False, cat_kyc=False,
                 tabnet_cfg=None, tabnet_match=False, seq_enc='attn', seq_cfg=None, out_keep=None, **base_kw):
        # [10-10] tabnet_cfg/tabnet_match: q_enc='tabnet'(2-D) · seq_enc: 'attn'(B0) | 'mamba'(4-A) | 'patchtst'(4-B) · out_keep: 출력 클래스 번호 목록(Case 6 = [0,1,2,3,4,6,8])
        super().__init__(**base_kw)                                   # ① 기준 층(같은 난수 순서)
        d = self.q_mlp[1].in_features
        assert drop_mode in ('remove', 'zero') and internal_mode in ('remove', 'zero')
        assert set(drop_te) <= {'dt', 'dtt', 'qdt'}, drop_te
        self.node_keep = [i for i in range(N_NODE) if i not in set(drop_node)]
        self.eq_keep = [j for j in range(N_EDGE) if j not in set(drop_edge_q)]
        self.et_keep = [j for j in range(N_EDGE) if j not in set(drop_edge_tok)]
        nk38 = self.node_keep + [N_NODE + i for i in self.node_keep]
        self.drop_mode = drop_mode
        for nm in ('sel_q', 'sel_n', 'sel_t'):                         # remove: index 버퍼 · zero: 0/1 곱 버퍼 · 바꾸는 열이 없으면 None(연산 안 함)
            self.register_buffer(nm, None, persistent=False)
        if drop_mode == 'remove':
            if len(self.eq_keep) < N_EDGE:
                self.sel_q = torch.tensor(self.eq_keep, dtype=torch.long)
            if len(self.node_keep) < N_NODE:
                self.sel_n = torch.tensor(nk38, dtype=torch.long)
            if len(self.et_keep) < N_EDGE:
                self.sel_t = torch.tensor(self.et_keep, dtype=torch.long)
            if self.sel_q is not None:                                 # ② 첫 층 잘라 내기(난수 없음)
                self.q_edge = _sub_linear(self.q_edge, self.eq_keep)
            if self.sel_n is not None:
                self.q_node = _sub_linear(self.q_node, nk38)
            if self.sel_t is not None:
                self.tok = _sub_linear(self.tok, self.et_keep)
            n_eq, n_nd, n_et = len(self.eq_keep), len(self.node_keep), len(self.et_keep)
        else:
            def _m(n, keep):
                v = torch.zeros(n); v[keep] = 1.0; return v
            if len(self.eq_keep) < N_EDGE:
                self.sel_q = _m(N_EDGE, self.eq_keep)
            if len(self.node_keep) < N_NODE:
                self.sel_n = _m(2 * N_NODE, nk38)
            if len(self.et_keep) < N_EDGE:
                self.sel_t = _m(N_EDGE, self.et_keep)
            n_eq, n_nd, n_et = N_EDGE, N_NODE, N_EDGE
        # ③ 모델 안 입력: 0 으로 만들기(zero_set) 또는 항 빼기(rm)
        self.zero_set, self.rm = set(), set()
        names = set(drop_te) | ({'bkt'} if drop_bkt else set())
        if internal_mode == 'zero':
            self.zero_set |= names
            if drop_ncoe:
                self.no_ncoe = True                                    # 기존 C 팔과 같은 0 만들기(tgt_model.py:84-85, 98-99)
        else:
            self.rm |= names | ({'ncoe'} if drop_ncoe else set())
            for nm, mods in (('dt', ('te_tok', 'te_tok_p')), ('dtt', ('te_key', 'te_key_p')), ('qdt', ('te_q', 'q_te_p')), ('bkt', ('bkt',)),
                             ('ncoe', ('ncoe', 'ns'))):
                if nm in self.rm:
                    for m_ in mods:
                        delattr(self, m_)
            if 'ncoe' in self.rm:                                       # 헤드 입력 272 → 256(ns 16차원을 잘라 냄)
                keep = list(range(4 * d))
                self.head[0] = _sub_layernorm(self.head[0], keep); self.head[1] = _sub_linear(self.head[1], keep)
        # ④ 새 입력부(기준 층 뒤에 만든다)
        self.q_kind, self.tok_kind, self.q_cols = q_enc, tok_enc, q_cols
        if q_enc != 'linear':
            assert q_cols in ('edge17', 'edge_node55'), q_cols
            n_in = n_eq + (2 * n_nd if q_cols == 'edge_node55' else 0)
            groups = ()
            if cat_kyc:
                assert q_cols == 'edge_node55', '--ft-cat-kyc 는 계좌 열이 FT 에 들어갈 때(edge_node55)만'
                assert all(c in self.node_keep for c in KYC_NODE), 'KYC 6열 중 일부를 빼면 범주 토큰으로 묶을 수 없다'
                pos = [self.node_keep.index(c) if drop_mode == 'remove' else c for c in KYC_NODE]   # 질의 입력 = tx ‖ nu ‖ nv 안의 KYC 자리
                groups = ([n_eq + i for i in pos], [n_eq + n_nd + i for i in pos])
            assert q_cfg is not None
            self.q_ft_params = ft_params(n_in, d, cat_groups=groups, **q_cfg)
            if q_enc == 'ft':
                self.enc_q = FTEncoder(n_in, d, ckpt=q_ckpt, cat_groups=groups, **q_cfg)
            elif q_enc == 'mlp':
                h = mlp_q_hidden or solve_hidden(n_in, d, mlp_layers, self.q_ft_params)
                self.enc_q = MLPEncoder(n_in, d, h, mlp_layers, mlp_drop); self.mlp_q_hidden = h
            elif q_enc == 'tabnet':                                    # [10-10] 2-D · KYC 범주 묶음은 TabNet 의 group_attention_matrix(공식 기능)로
                assert tabnet_cfg is not None
                cfg = dict(tabnet_cfg)
                if tabnet_match:
                    w = solve_tabnet_width(n_in, d, groups, cfg, self.q_ft_params); cfg.update(n_d=w, n_a=w)
                self.tabnet_cfg = cfg
                self.enc_q = TabNetQ(n_in, d, groups, **cfg)
            else:
                raise ValueError(q_enc)
            del self.q_edge
            if q_cols == 'edge_node55':
                del self.q_node
        if tok_enc != 'linear':
            assert tok_cfg is not None
            self.tok_ft_params = ft_params(n_et, d, **tok_cfg)
            if tok_enc == 'ft':
                self.enc_tok = FTEncoder(n_et, d, ckpt=self.ckpt, **tok_cfg)
            elif tok_enc == 'mlp':
                h = mlp_tok_hidden or solve_hidden(n_et, d, mlp_layers, self.tok_ft_params)
                self.enc_tok = MLPEncoder(n_et, d, h, mlp_layers, mlp_drop); self.mlp_tok_hidden = h
            else:
                raise ValueError(tok_enc)
            del self.tok
        # ⑤ [10-10] 시퀀스 인코더(4-A · 4-B) — 기준 인코더 self.enc 를 지우고 그 자리를 바꾼다(그 뒤에 만듦 → 다른 층 초기값 = B0)
        self.seq_kind = seq_enc
        if seq_enc != 'attn':
            L0 = self.enc.layers[0]; heads_ = L0.self_attn.num_heads; ff_ = L0.linear1.out_features; drop_ = L0.dropout.p; nl_ = len(self.enc.layers)
            self.b0_enc_params = sum(p.numel() for p in self.enc.parameters())
            assert seq_cfg is not None
            if seq_enc == 'mamba':
                self.enc_seq = MambaSeqEncoder(d, ff_, drop_, nl_, MAMBA_PRESETS[seq_cfg['preset']], seq_cfg['dir'] == 'bi')
            elif seq_enc == 'patchtst':
                self.enc_seq = PatchSeqEncoder(d, heads_, ff_, drop_, nl_, seq_cfg['K'], seq_cfg['P'], seq_cfg['S'])
            else:
                raise ValueError(seq_enc)
            del self.enc
        # ⑥ [10-10] Case 6: 출력층에서 뺀 클래스 행을 잘라 냄(난수 없음 → 나머지 행 · 다른 층 초기값 = B0)
        self.out_keep = None if out_keep is None else list(out_keep)
        if self.out_keep is not None:
            self.head[-1] = _sub_linear_rows(self.head[-1], self.out_keep)

    def aux_loss(self):
        """2-D TabNet 의 희소 정규화 항 M_loss(마지막 forward 값). TabNet 이 아니면 None."""
        return getattr(self.enc_q, 'aux', None) if self.q_kind == 'tabnet' else None

    def set_head_bias(self, b9: np.ndarray):
        """출력층 bias 를 주어진 값(출력 클래스 수만큼)으로(난수 없음 · --bias-init 에서만)."""
        with torch.no_grad():
            self.head[-1].bias.copy_(torch.as_tensor(np.asarray(b9, np.float32)))

    def _cols(self, t, sel):
        if sel is None:
            return t
        return t.index_select(-1, sel) if self.drop_mode == 'remove' else t * sel

    def forward(self, b: dict) -> torch.Tensor:                        # TGTND.forward(tgt_model.py:68-101)와 같은 연산 순서 · 옵션 자리만 갈라짐
        zs = self.zero_set

        def z(name, t):
            return t if (self.time_on and name not in zs) else torch.zeros_like(t)
        mask = b['mask']                                                                       # [A, K] True = 있음
        xf = self._cols(b['x'].float(), self.sel_t)
        h = self.tok(xf) if self.tok_kind == 'linear' else self.enc_tok(xf)                    # ← 위치 ②
        h = h + self.dir(b['dir'].long())
        if 'dt' not in self.rm:
            h = h + self.te_tok_p(self.te_tok(z('dt', b['dt'])))
        if 'bkt' not in self.rm:
            h = h + self.bkt(z('bkt', b['bkt'].long()))
        if self.seq_kind != 'attn':                                                            # [10-10] 4-A · 4-B
            H = self.enc_seq(h, mask, self.ckpt and self.training)
        elif self.ckpt and self.training:
            for layer in self.enc.layers:
                h = torch.utils.checkpoint.checkpoint(layer, h, None, ~mask, use_reentrant=False)
            H = h
        else:
            H = self.enc(h, src_key_padding_mask=~mask)
        mf = mask.unsqueeze(-1).to(H.dtype)
        mean = (H * mf).sum(1) / mf.sum(1).clamp(min=1)
        iu, iv = b['inv'][:, 0], b['inv'][:, 1]
        m_u, m_v = mask[iu], mask[iv]; cp_u, cp_v = b['cp'][iu], b['cp'][iv]
        B, K = m_u.shape
        kv = torch.cat([H[iu] + self.side.weight[0], H[iv] + self.side.weight[1]], 1)
        if 'ncoe' not in self.rm:
            cnt = ncoe_counts(cp_u, cp_v, m_u, m_v)
            if self.no_ncoe:
                cnt = torch.zeros_like(cnt)
            kv = kv + self.ncoe(torch.log1p(cnt)).to(kv.dtype)
        if 'dtt' not in self.rm:
            kv = kv + self.te_key_p(self.te_key(z('dtt', b['dtt'].reshape(B, 2 * K)))).to(kv.dtype)
        kv = self.kv_norm(kv)
        tx = self._cols(b['tx'], self.sel_q)
        if self.q_kind == 'linear':                                                            # ← 위치 ①
            qin = self.q_edge(tx) + self.q_node(self._cols(torch.cat([b['nu'], b['nv']], 1), self.sel_n))
        elif self.q_cols == 'edge17':
            qin = self.enc_q(tx) + self.q_node(self._cols(torch.cat([b['nu'], b['nv']], 1), self.sel_n))
        else:
            nn_ = self._cols(torch.cat([b['nu'], b['nv']], 1), self.sel_n)
            qin = self.enc_q(torch.cat([tx, nn_], 1))
        if 'qdt' not in self.rm:
            qte = self.te_q(z('qdt', b['qdt'])).reshape(B, -1)
            qin = qin + self.q_te_p(qte)
        if self.pc_dim:
            qin = qin + self.q_pc(b['pc'])
        q = self.q_mlp(qin)
        c, _ = self.xattn(q.unsqueeze(1), kv, kv, key_padding_mask=~torch.cat([m_u, m_v], 1), need_weights=False)
        parts = [c.squeeze(1).float(), q.float(), mean[iu].float(), mean[iv].float()]
        if 'ncoe' not in self.rm:
            u, v = b['uv'][:, 0], b['uv'][:, 1]
            ns = torch.stack([((cp_u == v.unsqueeze(1)) & m_u).sum(1), ((cp_v == u.unsqueeze(1)) & m_v).sum(1),
                              ((cnt[:, :K, 1] > 0) & m_u).sum(1)], 1).float()
            if self.no_ncoe:
                ns = torch.zeros_like(ns)
            parts.append(self.ns(torch.log1p(ns)).float())
        return self.head(torch.cat(parts, 1))


# ═════════════════ 5. TGT B0 — 하루 시퀀스 캐시 · 모델 만들기 ═════════════════
# ── 원래: HI-Large_EDA/Transformer모델/TGT_유형8_기법_20261013/common_t.py 줄 29-30 · sha256 4790568f52b12131 — 캐시 열 이름
ACC_KEYS = ('x', 'dir', 'dt', 'bkt', 'cp', 'mask')                   # 계좌(시퀀스) 단위
TGT_KEYS = ('inv', 'dtt', 'qdt', 'uv', 'tx', 'nu', 'nv', 'y')        # 대상 거래 단위


# ── 원래: HI-Large_EDA/Transformer모델/기법실행_20261008/code/train_tgt.py 줄 275-288 · sha256 76e1a1cf055f510f — build_model. 인자 A 는 원래 train_tgt.parse_args + resolve(common_t.make_args) 로 만든 값을 aml8_모델.pt 에 저장해 둔 것(시드만 다름)
# ── [묶음] train_tgt.py 를 통째로 옮기지 않은 까닭: 맨 위에서 tree9_lib(/tmp 코드 · DuckDB)를 불러오고 나머지는 학습 · 채점 루프다.
def tgt_build_model(A) -> TGTND8:  # [묶음] 이름만 바꿈(원래 build_model · SALT 의 build_model 과 겹침)
    """원본 main 의 `torch.manual_seed(A.seed); torch.cuda.manual_seed_all(A.seed); model = TGTND(...)` 와 같은 순서로 만든다(CPU)."""
    torch.manual_seed(A.seed); torch.cuda.manual_seed_all(A.seed)
    return TGTND8(time_on=not A.time_off, pc_dim=A.pc_dim, te_fixed=A.te_fixed, no_ncoe=A.no_ncoe,
                  drop_node=A.drop_node, drop_edge_q=A.drop_edge_q, drop_edge_tok=A.drop_edge_tok, drop_mode=A.drop_mode,
                  drop_te=A.drop_te_set, drop_bkt=A.drop_bucket, drop_ncoe=A.drop_ncoe, internal_mode=A.internal_mode,
                  q_enc=A.q_enc, q_cols=A.ft_cols, q_cfg=A.q_cfg, tok_enc=A.tok_enc, tok_cfg=A.tok_cfg,
                  mlp_layers=A.mlp_layers, mlp_drop=A.mlp_drop,
                  mlp_q_hidden=None if A.mlp_q_hidden == 'auto' else int(A.mlp_q_hidden),
                  mlp_tok_hidden=None if A.mlp_tok_hidden == 'auto' else int(A.mlp_tok_hidden),
                  q_ckpt=A.ft_q_ckpt, cat_kyc=A.ft_cat_kyc == 'on',
                  tabnet_cfg=A.tabnet_cfg, tabnet_match=A.tabnet_preset == 'matched', seq_enc=A.seq_enc, seq_cfg=A.seq_cfg, out_keep=A.out_keep)


# ═════════════════ [노트북] 새로 씀 — 메모리 하루 캐시 · TGT B0 예측 반복 ═════════════════
# 원래 DayCache(aml8.py:1167-1196 = common_t.py:110-138)는 data/tgt_cache_K100/<split>/<D>.npz 파일을 읽었다.
# 여기서는 tgt_inputs.finish_day 가 만든 하루 dict(같은 키 · dtype · shape)를 날 순서 목록으로 받는다.
# batch() · order_of() 는 aml8.py:1180-1196 글자 그대로다(계좌는 날마다 다시 묶음).
import contextlib as _contextlib
import types as _types


class MemDayCache:
    """날 순서 하루 캐시 dict 목록(메모리). 대상 순서 = 날 순서 · 날 안 edge_id 오름차순."""

    def __init__(self, parts):
        self.parts = list(parts); refs = []
        for di, z in enumerate(self.parts):
            refs += [(di, i) for i in range(len(z['y']))]
        self.ref = np.array(refs, np.int64).reshape(-1, 2)                     # 전역 대상 번호 → (날, 날 안 번호)  (aml8.py:1177)
        self.y = np.concatenate([p['y'] for p in self.parts]); self.eid = np.concatenate([p['eid'] for p in self.parts])

    def batch(self, gidx):
        """전역 대상 번호 목록 → make_batch 와 같은 꼴의 dict(계좌는 날마다 다시 묶음). aml8.py:1180-1190 글자 그대로."""
        gidx = np.asarray(gidx); out = {k: [] for k in ACC_KEYS + TGT_KEYS}; off = 0
        for di in np.unique(self.ref[gidx, 0]):
            sel = self.ref[gidx[self.ref[gidx, 0] == di], 1]; z = self.parts[di]
            inv = z['inv'][sel]; acc = np.unique(inv.ravel()); new_inv = np.searchsorted(acc, inv)
            for k in ACC_KEYS:
                out[k].append(z[k][acc])
            out['inv'].append(new_inv + off); off += len(acc)
            for k in TGT_KEYS[1:]:
                out[k].append(z[k][sel])
        return {k: np.concatenate(v) for k, v in out.items()}

    def order_of(self, gidx):
        """batch() 가 내놓는 대상 순서(날 순서로 다시 모임)에 맞는 전역 번호. aml8.py:1192-1196 글자 그대로."""
        gidx = np.asarray(gidx)
        return np.concatenate([gidx[self.ref[gidx, 0] == di] for di in np.unique(self.ref[gidx, 0])])


def predict_tgt(M, parts, dev='cpu', eval_batch=2048, seeds=None):
    """TGT B0 시드별 확률. aml8.py:1294-1323 predict_tgt 의 반복을 옮김(캐시 파일 대신 메모리 parts).
    반환 (eid int64 [T] 캐시 순서, {seed: P float32 [T, 8]}). 정렬(align)은 부르는 쪽에서.
    - 배치 = 전역 대상 번호를 eval_batch(2048, 원래 값)씩 자름 → 날 경계를 넘는 배치도 원래와 같음.
    - cuda 면 원래와 같은 fp16 autocast(aml8.py:1304, 1318), cpu 면 fp32."""
    m = M['models']['tgt_b0']
    va = MemDayCache(parts)
    assert va.parts[0]['x'].shape[1] == m['k_nb'] == 100, '시퀀스 칸 수가 100 이 아니다'          # aml8.py:1298
    to_dev = lambda b: {k: torch.from_numpy(np.ascontiguousarray(x)).to(dev, non_blocking=True) for k, x in b.items()}  # noqa: E731  aml8.py:1303
    amp = (lambda: torch.autocast('cuda', dtype=torch.float16)) if dev == 'cuda' else _contextlib.nullcontext          # aml8.py:1304
    out = {}
    for seed in (seeds or m['seeds']):
        A = _types.SimpleNamespace(**dict(m['args'], seed=seed))              # aml8.py:1307 (시드만 다름)
        model = tgt_build_model(A)                                             # 시드는 build_model 안에서(aml8.py:1154)
        if hasattr(model, 'ckpt'):
            model.ckpt = False
        model.load_state_dict(m['state_dict'][f's{seed}'])                     # strict(기본값)
        model = model.to(dev)
        model.eval(); Pc = np.empty((len(va.y), 8), np.float32)               # aml8.py:1313-1319
        with torch.no_grad():
            for st in range(0, len(va.y), eval_batch):
                gi = np.arange(st, min(st + eval_batch, len(va.y))); b = va.batch(gi)
                with amp():
                    lg = model(to_dev(b))
                Pc[va.order_of(gi)] = torch.softmax(lg.float(), 1).cpu().numpy()
        out[seed] = Pc
        del model
    return va.eid.astype(np.int64), out
'''

In [ ]:
# @title 코드 ⑧ 도구: 코드 설치 · 자료 풀기 · LightGBM(numpy) · 섞기 · 채점 — 숨김
import base64
import contextlib
import datetime
import gc
import hashlib
import io
import json
import os
import re
import sys
import threading
import time
import types
import zlib
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(f'Python 3.11 이상이 필요합니다(지금 {sys.version.split()[0]}).')
if sys.platform.startswith('win') or not hasattr(os, 'preadv'):
    raise RuntimeError('이 노트북은 Linux 에서만 돕니다(원장 읽기에 os.preadv · resource 를 씀). 팀 JupyterLab 서버에서 열어 주세요.')
# 꼭 있어야 하는 라이브러리 3개. 없으면 무엇을 깔지 한국어로 알리고 멈춤(폴더 · 파일을 만들기 전).
try:
    import numpy as np
    import pandas as pd
except ImportError as _e:
    raise ImportError(f'{getattr(_e, "name", None) or "numpy · pandas"} 가 없습니다 → 터미널에서 `pip install numpy pandas` 를 한 뒤 '
                      '커널을 다시 시작하고 Run All 해 주세요.') from None
try:
    import torch
except ImportError:
    raise ImportError('torch 가 없습니다 → 터미널에서 `pip install torch`(2.x · CPU 판이면 충분)를 한 뒤 커널을 다시 시작하고 '
                      'Run All 해 주세요.') from None
if tuple(int(x) for x in re.findall(r'\d+', torch.__version__)[:2]) < (2, 0):
    raise ImportError(f'torch {torch.__version__} 는 옛 판입니다 → 2.0 이상이 필요합니다(2.10 에서 시험) · `pip install -U torch` 뒤 커널을 다시 시작해 주세요.')


def say(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)


# ───────────────────────── 시간 · 메모리 기록 ─────────────────────────
def _proc_status():
    """이 프로세스의 메모리(GiB): VmRSS · VmHWM(꼭대기) · RssAnon. 컨테이너 anon(/sys/fs/cgroup/memory.stat)도."""
    d = {}
    try:
        for line in open('/proc/self/status'):
            if line.startswith(('VmRSS', 'VmHWM', 'RssAnon')):
                k, v = line.split(':'); d[k] = int(v.split()[0]) / 2 ** 20
    except OSError:
        pass
    try:
        for line in open('/sys/fs/cgroup/memory.stat'):
            if line.startswith('anon '):
                d['cgroup_anon'] = int(line.split()[1]) / 2 ** 30; break
    except OSError:
        pass
    return d


class Meter:
    """단계별 걸린 초와 메모리. 0.5초마다 RssAnon · 컨테이너 anon 꼭대기를 잼(뒤에서 도는 실 하나)."""

    def __init__(self):
        self.rows = []; self.peak = {'RssAnon': 0.0, 'cgroup_anon': 0.0}; self._stop = threading.Event()
        self._th = threading.Thread(target=self._loop, daemon=True); self._th.start()

    def _loop(self):
        while not self._stop.wait(0.5):
            s = _proc_status()
            for k in self.peak:
                if k in s:
                    self.peak[k] = max(self.peak[k], s[k])

    @contextlib.contextmanager
    def stage(self, name):
        t = time.time()
        yield
        s = _proc_status()
        self.rows.append(dict(단계=name, 초=round(time.time() - t, 1), RssAnon_GiB=round(s.get('RssAnon', float('nan')), 2),
                              VmHWM_GiB=round(s.get('VmHWM', float('nan')), 2), 컨테이너anon_GiB=round(s.get('cgroup_anon', float('nan')), 2)))

    def close(self):
        self._stop.set(); self._th.join(timeout=2)
        s = _proc_status()
        for k in self.peak:
            if k in s:
                self.peak[k] = max(self.peak[k], s[k])
        self.peak['VmHWM'] = s.get('VmHWM', float('nan'))
        return {k: round(v, 2) for k, v in self.peak.items()}


# ───────────────────────── 숨긴 셀의 코드를 모듈로 설치 ─────────────────────────
_MODULE_ORDER = ('ledger', 'x17', 'node19', 'salt_inputs', 'tgt_inputs', 'salt_models', 'tgt_model')


def install_modules(scratch):
    """숨긴 셀 ①~⑦ 의 코드 글자를 SCRATCH/_code/<이름>.py 로 쓰고 같은 글자를 모듈로 실행해 sys.modules 에 넣음.
    파일로도 쓰는 까닭: ledger.py 가 meta 에 자기 코드 sha256 을 적고(Path(__file__)), 오류가 나면 줄 번호가 보이게.
    모듈끼리는 `import ledger as LG` 처럼 서로 부르므로 의존 순서대로 넣음."""
    code_dir = Path(scratch).expanduser().resolve() / '_code'
    code_dir.mkdir(parents=True, exist_ok=True)
    mods = {}
    allow = globals().get('ALLOW_MODIFIED_CODE', False) is True              # 일부러 고친 코드일 때만 설정 셀에 True
    for name in _MODULE_ORDER:
        src = _MODULE_SRC[name]
        sha = hashlib.sha256(src.encode('utf-8')).hexdigest()
        if _MODULE_SHA.get(name) != sha:
            if not allow:
                raise RuntimeError(f'숨긴 코드 셀 {name} 이 만들 때와 다릅니다(sha256 {sha[:16]} ≠ {str(_MODULE_SHA.get(name))[:16]}). '
                                   '받은 노트북 파일을 다시 열어 Run All 해 주세요. 일부러 고친 것이면 설정 셀에 '
                                   'ALLOW_MODIFIED_CODE = True 를 넣고 다시 실행하세요(결과가 원래와 달라질 수 있음).')
            say(f'[알림] 숨긴 코드 셀 {name} 이 만들 때와 다릅니다(sha256 {sha[:16]}). ALLOW_MODIFIED_CODE = True 라 고친 코드로 돕니다.')
        path = code_dir / f'{name}.py'
        if not path.exists() or path.read_text(encoding='utf-8') != src:
            path.write_text(src, encoding='utf-8')
        mod = types.ModuleType(name); mod.__file__ = str(path)
        sys.modules[name] = mod
        exec(compile(src, str(path), 'exec', dont_inherit=True), mod.__dict__)
        mods[name] = mod
    return mods


# ───────────────────────── 숨긴 자료 풀기 ─────────────────────────
def _unpack(lines, what):
    """base64 줄 목록 → zlib 풀기 → 바이트. 만들 때 잰 sha256(_PAYLOAD_SHA[what])과 다르면 멈춤."""
    try:
        raw = zlib.decompress(base64.b64decode(''.join(lines), validate=True))
        sha = hashlib.sha256(raw).hexdigest()
    except (ValueError, zlib.error):                                          # base64 · zlib 이 깨짐(binascii.Error 는 ValueError)
        raw, sha = None, '풀리지 않음'
    if sha != _PAYLOAD_SHA[what]:
        raise RuntimeError(f'숨긴 자료 셀({what})이 만들 때와 다릅니다(sha256 {sha[:16]} ≠ {_PAYLOAD_SHA[what][:16]}). '
                           '숨긴 셀은 고치지 말고, 받은 노트북 파일을 다시 열어 Run All 해 주세요.')
    return raw


def load_nn_weights():
    """GNN · TGT 가중치 9개(SALT C · TC B · TGT B0 × 시드 3) + SALT z-표준화 · δ + TGT 설정.
    aml8_모델.pt 에서 LightGBM 문자열만 뺀 dict 를 torch.save 한 것. weights_only=True(파일 안 코드를 실행하지 않음, aml8.py:1224-1226)."""
    return torch.load(io.BytesIO(_unpack(NN_B64, 'NN')), map_location='cpu', weights_only=True)


class LGBNumpy:
    """LightGBM 다중 분류 예측(860 round × 8유형 = 나무 6,880개)을 numpy 로 계산.
    나무 배열은 aml8_모델.pt models.lgb.model_string(LightGBM 4.7.0 · L31D6_m20_r0_lam0_s42 · round 860 까지)에서 풀어 낸 것.
    규칙(LightGBM tree.h NumericalDecision): decision_type 은 모두 2(숫자 · default_left · 빈 값 규칙 None) →
    NaN 은 0 으로 본 뒤 x <= 문턱 이면 왼쪽. 원점수는 round 순서대로 더함(GBDT::PredictRaw 와 같은 순서).
    확률 = softmax(원점수) 를 float32 로(aml8.py:1263 `softmax(bst.predict(X, raw_score=True)).astype(np.float32)`)."""

    def __init__(self, lines):
        z = np.load(io.BytesIO(_unpack(lines, 'LGB')), allow_pickle=False)
        self.feat = z['feat'].astype(np.int64); self.thr = z['thr']; self.left = z['left'].astype(np.int64)
        self.right = z['right'].astype(np.int64); self.leaf = z['leaf']
        self.start = np.where(z['nl'] > 1, 0, -1).astype(np.int64)            # 잎 하나뿐인 나무는 처음부터 잎 0(= ~0)
        self.T = len(self.thr); assert self.T == 860 * 8, self.T

    def raw(self, X, chunk=1000):
        X = np.asarray(X, np.float64)
        if X.ndim != 2 or X.shape[1] != 55:
            raise ValueError(f'입력은 (행, 55) 이어야 합니다: {X.shape}')
        X = np.where(np.isnan(X), 0.0, X)
        out = np.empty((len(X), 8)); tt = np.arange(self.T)
        for a in range(0, len(X), chunk):
            x = X[a:a + chunk]; n = len(x)
            node = np.repeat(self.start[None, :], n, 0)                         # 0 이상 = 분기 칸 · 음수 = 잎(~잎 번호)
            for _ in range(64):
                act = node >= 0
                if not act.any():
                    break
                nd = np.where(act, node, 0)
                go_left = np.take_along_axis(x, self.feat[tt, nd], 1) <= self.thr[tt, nd]
                node = np.where(act, np.where(go_left, self.left[tt, nd], self.right[tt, nd]), node)
            lv = self.leaf[tt, ~node]
            raw = np.zeros((n, 8))
            for it in range(self.T // 8):                                       # round 순서대로 유형별로 더함
                raw += lv[:, it * 8:(it + 1) * 8]
            out[a:a + n] = raw
        return out

    def proba(self, X):
        R = self.raw(X)
        R = R - R.max(1, keepdims=True); E = np.exp(R)                          # aml8.py:105-106 softmax
        return (E / E.sum(1, keepdims=True)).astype(np.float32)


# ───────────────────────── 섞기 · 채점 ─────────────────────────
def seed_mean(P_by_seed):
    """시드 평균: float32 확률 → float64 → 평균(aml8.py:1356 final_candidate.main 과 같은 식)."""
    return np.mean([P.astype(np.float64) for P in P_by_seed.values()], 0)


def final_probs(probs):
    """probs = {이름: 시드 평균 확률}. 로그 평균(가중 = 욕심 선택 횟수) → BIP/STACK 보정. aml8.py:1210-1215 글자 그대로."""
    L = sum(COMP[k][1] * np.log(np.maximum(probs[k], 1e-30)) for k in COMP) / sum(w for _, w in COMP.values())
    L -= L.max(1, keepdims=True); Q = np.exp(L); Q /= Q.sum(1, keepdims=True)
    Q[:, 5] *= np.exp(DELTA / 2); Q[:, 6] *= np.exp(-DELTA / 2)
    return Q / Q.sum(1, keepdims=True)


def _pr(cm):
    """유형별 정밀도 · 재현율(찍은 것이 없으면 정밀도 0 · sklearn zero_division=0 과 같음). aml8.py:134-137 글자 그대로."""
    tp = np.diag(cm); pc, rc = cm.sum(0), cm.sum(1)
    return np.where(pc > 0, tp / np.maximum(pc, 1e-12), 0.0), np.where(rc > 0, tp / np.maximum(rc, 1e-12), 0.0)


def macro_pr(P, y):
    """macro 정밀도 · 재현율(8유형 평균 · aml8.py Scorer.key 의 pr.mean() · rc.mean() 과 같은 식). y < 0(패턴 행 아님)은 뺌."""
    P = np.asarray(P); y = np.asarray(y, np.int64); m = y >= 0
    if not m.any():
        return float('nan'), float('nan'), 0
    cm = np.bincount(y[m] * 8 + P[m].argmax(1), minlength=64).reshape(8, 8).astype(np.float64)
    pr, rc = _pr(cm)
    return float(pr.mean()), float(rc.mean()), int(m.sum())

In [ ]:
# @title 코드 ⑨ 실행 순서: 설정 확인 → 원장 → 날마다 4모델 입력 → 4모델 → 섞기 → csv — 숨김
# 날마다 하는 일(각 모듈 머리말 · 규칙 옆에 원래 file:line):
#   창 읽기 = 원장에서 [D−29, D] 30일 행(node19.read_window)
#   계좌 19열 = D 마감 30일 창 계좌 피처(node19.node19_day · daily_pipeline.py:340-389)
#   묶음 · 시퀀스 계획 = SALT/TC 받은 목록 → 128개 묶음 그래프(salt_inputs.salt_plan · c1_cache.py:32-62)
#                        + TGT 계좌별 최근 100건(tgt_inputs.plan_day · tgt_data.py:94-104)
#   거래 17열 = 위 셋이 쓰는 거래 전부의 x17(x17.x17_table · daily_pipeline.py:245-330 · 2022-08-01 부터 누적)
#   4모델 입력 = LightGBM 55열 · SALT/TC 하루 캐시 8배열 · TGT 하루 캐시 15키 (원래 캐시와 비트 단위로 같음)


_DAY_HELP = "예: '2022-10-06' · 'val'(val 17일 전부) · '2022-09-28~2022-10-05'(양 끝 포함) · ['2022-09-28', '2022-10-14']"


def _one_day(s, orig):
    """'2022-10-06' · '2022/10/06'(원본 거래 파일의 꼴) · '2022-10-06 13:20' → '2022-10-06'."""
    try:
        return datetime.date.fromisoformat(str(s).strip().replace('/', '-')[:10]).isoformat()
    except ValueError:
        raise ValueError(f'TARGET_DAYS 의 {orig!r} 를 날짜로 읽지 못했습니다. {_DAY_HELP}') from None


def _expand_day(d):
    """TARGET_DAYS 의 한 칸 → 날 목록. 'val' = val 17일, 'A~B' = A 부터 B 까지(양 끝 포함)."""
    if isinstance(d, datetime.date):                                       # date · datetime · pd.Timestamp
        return [d.isoformat()[:10]]
    s = str(d).strip()
    if s.lower() == 'val':
        return list(pd.date_range(VAL_FIRST, VAL_LAST).strftime('%Y-%m-%d'))
    if '~' in s:
        a, b = (_one_day(x, d) for x in s.split('~', 1))
        if a > b:
            raise ValueError(f'TARGET_DAYS 의 {d!r}: 앞 날({a})이 뒤 날({b})보다 늦습니다. {_DAY_HELP}')
        return list(pd.date_range(a, b).strftime('%Y-%m-%d'))
    return [_one_day(s, d)]


def _norm_days(days):
    """TARGET_DAYS → 'YYYY-MM-DD' 오름차순 목록. test 봉인 · 보정 주는 여기서 먼저 거부(원장을 짓기 전)."""
    if isinstance(days, (str, datetime.date)):
        days = [days]
    out = sorted({x for d in days for x in _expand_day(d)})
    if not out:
        raise ValueError(f'TARGET_DAYS 가 비어 있습니다. {_DAY_HELP}')
    for d in out:
        if d >= TEST_START and os.environ.get('AML_TEST_UNSEAL') != '1':      # tree9_lib.py:70-80 과 같은 뜻
            raise PermissionError(f'[test 봉인] {d}: {TEST_START} 이후는 test 기간이라 분류하지 않습니다(팀 결정 전). '
                                  f'TARGET_DAYS 를 {VAL_LAST} 이전 날짜로 바꿔 주세요. 원장도 짓지 않았습니다.')
        if d < TRAIN_FIRST:                                                   # daily_pipeline.py:350
            raise ValueError(f'{d}: {TRAIN_FIRST} 이전(통화 보정 주 08-01~08-07)은 분류할 수 없습니다.')
    return out


def _load_targets(targets):
    """TARGETS → None('patterns') 또는 edge_id 고유 · 오름차순 int64."""
    if isinstance(targets, str) and targets.strip().lower() == 'patterns':
        return None
    if isinstance(targets, (str, Path)):
        p = Path(targets).expanduser()
        if not p.exists():
            raise FileNotFoundError(f"TARGETS 파일이 없습니다: {p} ('patterns' 또는 edge_id 목록 · csv 경로)")
        df = pd.read_csv(p)
        if 'edge_id' not in df.columns:
            raise ValueError(f'{p}: edge_id 열이 없습니다(열: {list(df.columns)[:10]})')
        e = df['edge_id'].to_numpy()
    else:
        e = np.asarray(targets if isinstance(targets, np.ndarray) else list(targets))
    if e.size == 0:
        raise ValueError('TARGETS edge_id 목록이 비어 있습니다.')
    ei = np.asarray(e, np.float64)
    if not (np.isfinite(ei).all() and (ei == np.round(ei)).all() and (ei >= 0).all()):
        raise ValueError('TARGETS 의 edge_id 는 0 이상 정수여야 합니다.')
    return np.unique(ei.astype(np.int64))


def _check_sources(trans_csv, accounts_csv, patterns_txt):
    """원본 3파일이 있는지 · 거래 파일 크기가 원본과 같은지. 다르면 원장을 짓기 전에 멈춤."""
    for nm, p in (('TRANS_CSV', trans_csv), ('ACCOUNTS_CSV', accounts_csv), ('PATTERNS_TXT', patterns_txt)):
        if not Path(p).expanduser().is_file():
            raise FileNotFoundError(f'{nm} 파일이 없습니다: {p}')
    size = Path(trans_csv).expanduser().stat().st_size
    if size != SOURCE_BYTES:
        raise ValueError(f'TRANS_CSV 가 원본 HI-Large_Trans.csv 와 다릅니다: 크기 {size:,} B (원본 {SOURCE_BYTES:,} B · {SOURCE_ROWS:,}행). '
                         '이 노트북은 원본 전체 파일이 행 순서 그대로 있어야 합니다. edge_id(= 원본 행 번호)가 이웃 목록 · 시퀀스의 '
                         '동점 정렬에 쓰여, 잘라 내거나 고친 파일로는 같은 결과가 나오지 않습니다(daily_pipeline.py:142, 393). 원장도 짓지 않았습니다.')


def _pick_device(name):
    import torch
    name = str(name).lower()
    if name == 'auto':
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    if name == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError("DEVICE='cuda' 인데 GPU 를 찾지 못했습니다. DEVICE 를 'auto' 나 'cpu' 로 바꿔 주세요.")
    if name not in ('cpu', 'cuda'):
        raise ValueError(f"DEVICE 는 'auto' · 'cpu' · 'cuda' 중 하나: {name!r}")
    return name


def _targets_by_day(L, LG, N19, days, tlist):
    """edge_id 목록 → 날별로 나눔. 원장에 없거나(중복으로 지움 · 봉인 뒤 · 범위 밖) TARGET_DAYS 밖이면 멈춤."""
    if tlist is None:
        return {D: None for D in days}
    try:
        _, _, pos = N19.target_accounts(L, tlist)                             # 원장에 없는 edge_id → ValueError
    except ValueError as e:
        raise ValueError(f'TARGETS: {e}. 원장에는 {LG.BASE_DATE} ~ {L.meta["d_max"]} 의 거래만 있고(완전 중복은 가장 작은 edge_id 만 남김), '
                         f'{TEST_START} 이후 거래는 test 봉인이라 넣지 않았습니다. edge_id 는 머리줄을 뺀 0부터의 행 번호(파일 줄 번호 − 2)입니다.') from None
    day0 = np.datetime64(LG.BASE_DATE)
    dstr = np.array([str(day0 + int(x)) for x in np.asarray(L['ts'][pos]).astype(np.int64) // 1440])
    bad = ~np.isin(dstr, days)
    early = bad & (dstr < TRAIN_FIRST)
    if early.any():                                                           # 보정 주 거래는 TARGET_DAYS 에 넣을 수도 없음(daily_pipeline.py:350)
        ex = ', '.join(f'{int(e)}({d})' for e, d in zip(tlist[early][:5], dstr[early][:5]))
        raise ValueError(f'TARGETS 의 edge_id {int(early.sum())}개가 {TRAIN_FIRST} 이전(통화 보정 주 08-01~08-07) 거래라 분류할 수 없습니다'
                         f'(예: {ex}). TARGETS 에서 빼 주세요.')
    if bad.any():
        ex = ', '.join(f'{int(e)}({d})' for e, d in zip(tlist[bad][:5], dstr[bad][:5]))
        raise ValueError(f'TARGETS 의 edge_id {int(bad.sum())}개가 TARGET_DAYS 밖의 날 거래입니다(예: {ex}). 그 날을 TARGET_DAYS 에 넣거나 '
                         'TARGETS 에서 빼 주세요.')
    return {D: tlist[dstr == D] for D in days}


def _build_day(L, mods, D, e_list, meter):
    """D 하루의 4모델 입력. 창 W 하나 · x17 표 하나를 세 소비자(55열 · SALT/TC · TGT)가 함께 씀."""
    X17, N19, SI, TGI = mods['x17'], mods['node19'], mods['salt_inputs'], mods['tgt_inputs']
    with meter.stage(f'{D} 창 읽기'):
        W = N19.read_window(L, D, cols=('src', 'dst', 'ts', 'eid', 'pz', 'pcur', 'rcur'))
    with meter.stage(f'{D} 계좌 19열'):
        T19 = N19.node19_day(L, D, W=W)
    with meter.stage(f'{D} 묶음 · 시퀀스 계획'):
        SP = SI.salt_plan(L, D, W=W, target_eids=e_list)
        TP = TGI.plan_day(L, D, eids=e_list, W=W)
    n_win = len(W['ts']); del W; gc.collect()
    eid = SP.target_eid
    if not np.array_equal(TP.eid, eid):
        raise AssertionError(f'{D}: SALT 와 TGT 의 대상 행이 다름')
    with meter.stage(f'{D} 거래 17열'):
        tab = X17.x17_table(L, np.union1d(np.union1d(SP.need_eids, TP.need_eids), eid))
    with meter.stage(f'{D} 4모델 입력 배열'):
        A = SI.salt_arrays(SP, T19, tab)                                     # c1_cache.py:44-62 꼴
        C = TGI.finish_day(TP, x17tab=tab, n19=T19)                          # t1_cache.py:28 꼴
        X55 = np.concatenate([T19.take(SP.target_src, strict=True), T19.take(SP.target_dst, strict=True),
                              X17.x17_take(tab, eid)], 1).astype(np.float32)   # val_pattern55 열 0:19 · 19:38 · 38:55
        if not np.isfinite(X55).all():
            raise ValueError(f'{D}: 55열에 NaN/Inf')
    info = dict(eid=eid.astype(np.int64), y=SP.target_y.astype(np.int64), ts=np.asarray(TP.te, np.int64),
                src=SP.target_src.astype(np.int64), dst=SP.target_dst.astype(np.int64))
    say(f'{D}: 대상 {len(eid):,}행 · 창 {n_win:,}행 · SALT 묶음 {len(SP.groups)} (노드 {len(A["nx"]):,} · 엣지 {len(A["ei"]):,}) · '
        f'TGT 계좌 {C["x"].shape[0]:,} · x17 {len(tab[0]):,}건')
    del T19, tab, SP, TP; gc.collect()
    return dict(day=D, info=info, X55=X55, salt=A, tgt=C)


def _make_table(L, per, final, mean):
    """결과 표: 거래마다 최종 예측 · 유형별 확률 8칸 · 모델별 예측 유형(+ Patterns.txt 정답이 있으면 정답 · 맞음)."""
    cn = np.asarray(CLASS_NAMES8)
    info = {k: np.concatenate([p['info'][k] for p in per]) for k in ('eid', 'y', 'ts', 'src', 'dst')}
    date = np.concatenate([[p['day']] * len(p['info']['eid']) for p in per])
    bank, num = np.asarray(L['acct_bank']), np.asarray(L['acct_num'])
    dec = lambda a: np.char.decode(a, 'utf-8')                                       # noqa: E731
    t = pd.Timestamp('2022-08-01') + pd.to_timedelta(info['ts'], unit='min')
    df = pd.DataFrame({'edge_id': info['eid'], 'date': date, '시각': t.strftime('%Y/%m/%d %H:%M'),
                       '보낸_은행': dec(bank[info['src']]), '보낸_계좌': dec(num[info['src']]),
                       '받은_은행': dec(bank[info['dst']]), '받은_계좌': dec(num[info['dst']]),
                       '예측': cn[final.argmax(1)], '예측확률': final.max(1).astype(np.float32)})
    for j, n in enumerate(CLASS_NAMES8):
        df[f'p_{n}'] = final[:, j].astype(np.float32)
    for k in MODEL_KEYS:
        df[f'{COL_PREFIX[k]}_예측'] = cn[mean[k].argmax(1)]
    y = info['y']
    if (y >= 0).any():
        df['정답'] = np.where(y >= 0, cn[np.maximum(y, 0)], '')
        df['맞음'] = pd.array(np.where(y >= 0, final.argmax(1) == y, False), dtype='boolean')
        df.loc[y < 0, '맞음'] = pd.NA
    return df, info


def _scores(mean, final, y, date):
    """macro 정밀도 · 재현율: 전체(정답 있는 행) · val 앞 8일 · val 뒤 9일(aml8.py:1326-1331 scores_of 와 같은 나눔)."""
    parts = [('전체', np.ones(len(y), bool))]
    if ((date >= VAL_FIRST) & (date < BACK9_FIRST)).any():
        parts.append(('val 앞8일', (date >= VAL_FIRST) & (date < BACK9_FIRST)))
    if ((date >= BACK9_FIRST) & (date <= VAL_LAST)).any():
        parts.append(('val 뒤9일', (date >= BACK9_FIRST) & (date <= VAL_LAST)))
    rows = []
    for k, P in list(mean.items()) + [('final', final)]:
        r = {'모델': '최종(섞기 + 보정)' if k == 'final' else MODEL_LABEL[k]}
        for nm, m in parts:
            p_, r_, n_ = macro_pr(P[m], y[m])
            r[f'{nm} 정밀도'] = round(p_, 4); r[f'{nm} 재현율'] = round(r_, 4); r[f'{nm} 행'] = n_
        rows.append(r)
    return pd.DataFrame(rows)


def 실행(trans_csv, accounts_csv, patterns_txt, target_days, targets='patterns', scratch='유형8_작업폴더', out_dir='.',
        device='auto', threads=2):
    """원본 3파일 → (처음 한 번) 압축 원장 → 날마다 4모델 입력 → LightGBM · SALT C · TC B · TGT B0 → 섞기 → 분류결과_<날짜>.csv."""
    T0 = time.time()
    days = _norm_days(target_days)                       # 봉인 · 날짜 확인(무엇이든 하기 전에)
    tlist = _load_targets(targets)
    _check_sources(trans_csv, accounts_csv, patterns_txt)
    dev = _pick_device(device)                           # 장치 확인도 폴더를 만들기 전에
    # 원장은 val 끝날(2022-10-14)까지 지음 — 거래 파일이 시간순이 아니라 어느 날까지 짓든 17 GB 를 끝까지 훑는 시간은 같고,
    # 그래야 기본 실행(10-06)으로 지은 원장을 val 다른 날에도 다시 씀. 날마다 쓰는 행은 그날까지뿐이라 결과는 같음(시험 T2 · T3).
    d_max = VAL_LAST if days[-1] <= VAL_LAST else days[-1]
    scratch = Path(scratch).expanduser().resolve(); out_dir = Path(out_dir).expanduser().resolve()
    scratch.mkdir(parents=True, exist_ok=True); out_dir.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(int(threads))
    say(f'시작 · 날 {len(days)}개({days[0]} ~ {days[-1]}) · 대상 {"Patterns.txt 행 전부" if tlist is None else f"edge_id {len(tlist):,}개"} · '
        f'장치 {dev} · 스레드 {torch.get_num_threads()} · 작업 폴더 {scratch}')
    if any(TRAIN_FIRST <= d <= TRAIN_LAST for d in days):
        say(f'[알림] {TRAIN_FIRST} ~ {TRAIN_LAST} 은 모델이 배운 날입니다. 그 날의 점수는 부풀려집니다.')
    meter = Meter()
    try:
        with meter.stage('코드 · 가중치 풀기'):
            mods = install_modules(scratch)
            LG, SM, TM = mods['ledger'], mods['salt_models'], mods['tgt_model']
            M = load_nn_weights(); lgbm = LGBNumpy(LGB_B64)
            assert list(M['features_55']) == FEATURES_55, '55열 이름이 가중치 파일과 다름'
            assert M['ensemble']['weights'] == {k: w for k, (_, w) in COMP.items()} and M['ensemble']['delta'] == DELTA
        old, why = LG.reuse_status(scratch / 'ledger', LG.source_fingerprints(trans_csv, accounts_csv, patterns_txt), d_max)
        fresh = old is None; del old                     # 다시 쓸 원장이 없으면 까닭 · 걸릴 시간 · 디스크를 먼저 알림
        if fresh:
            say(f'원장을 새로 짓습니다({why}). 원본 거래 파일을 끝까지 한 번 훑어 {d_max} 까지 남깁니다 · '
                f'약 {"5~6분" if LG.HAVE_ARROW else "12~13분"} · {scratch} 에 디스크 약 6 GB(원장 3.9 GB 가 남음) · 다음부터는 다시 씀')
            if not LG.HAVE_ARROW:
                say('[알림] pyarrow 가 없어 pandas 로 읽습니다. 첫 실행이 약 12~13분 걸립니다(`pip install pyarrow` 로 약 5~6분으로 줄일 수 있음).')
        with meter.stage('원장(없으면 짓기 · 있으면 다시 씀)'):
            L = LG.build_ledger(trans_csv, accounts_csv, patterns_txt, d_max=d_max, out_dir=scratch / 'ledger',
                                threads=int(threads), engine='auto', expected_rows=SOURCE_ROWS, expected_bytes=SOURCE_BYTES,
                                reuse=True, log=say)
        r = L.meta['rows']
        say(f'원장: {r["ledger"]:,}행(원본 {r["source_total"]:,}행 중 {L.meta["d_max"]} 뒤 {r["dropped_after_cut"]:,}행은 시각 칸만 해석하고 버림 · '
            f'완전 중복 {r["duplicates_removed"]}행 지움) · 패턴 행 {L.meta["patterns"]["matched"]:,} · 엔진 {L.meta["engine"]}')
        tday = _targets_by_day(L, LG, mods['node19'], days, tlist)
        per = []
        for D in days:
            if tday[D] is not None and len(tday[D]) == 0:
                say(f'{D}: TARGETS 에 이 날 거래가 없어 건너뜀'); continue
            per.append(_build_day(L, mods, D, tday[D], meter))
        if not per:
            raise ValueError('분류할 행이 없습니다.')
        eid_all = np.concatenate([p['info']['eid'] for p in per])
        P = {}
        with meter.stage('LightGBM'):
            P['lgb'] = {42: lgbm.proba(np.concatenate([p['X55'] for p in per]))}
        for key in ('salt_c', 'tc_b'):                                          # 모든 날을 날 순서로 한 번에(묶음 4개씩 배치가 원래와 같게)
            with meter.stage(MODEL_LABEL[key]):
                e_c, Pd = SM.predict_salt(M, key, [p['salt'] for p in per], dev=dev)
                o = SM.align(e_c, eid_all); P[key] = {s: x[o] for s, x in Pd.items()}
        with meter.stage('TGT B0'):
            e_t, Pd = TM.predict_tgt(M, [p['tgt'] for p in per], dev=dev)
            o = SM.align(e_t, eid_all); P['tgt_b0'] = {s: x[o] for s, x in Pd.items()}
        with meter.stage('섞기 · csv 쓰기'):
            mean = {k: seed_mean(P[k]) for k in MODEL_KEYS}
            final = final_probs({COMP_NAME[k]: mean[k] for k in MODEL_KEYS})
            table, info = _make_table(L, per, final, mean)
            files = []
            for D, g in table.groupby('date', sort=True):
                f = out_dir / f'분류결과_{D}.csv'; g.to_csv(f, index=False, encoding='utf-8-sig'); files.append(str(f))
    finally:
        peak = meter.close()
    date = table['date'].to_numpy().astype(str); y = info['y']
    stages = pd.DataFrame(meter.rows)
    R = dict(표=table, days=[p['day'] for p in per], eid=eid_all, date=date, y=y, P=P, mean=mean, final=final, stages=stages,
             peak=peak, device=dev, files=files, ledger=str(L.path), sec=round(time.time() - T0, 1), mode='patterns' if tlist is None else 'eids',
             inputs=per)                                                          # 날별 4모델 입력(55열 · SALT/TC 8배열 · TGT 15키) — 확인용
    # ── 요약 ──
    say(f'끝 · {len(table):,}행 · {R["sec"]:.0f}초 · 메모리 꼭대기 RssAnon {peak.get("RssAnon", float("nan")):.2f} GiB · '
        f'VmHWM {peak.get("VmHWM", float("nan")):.2f} GiB')
    print('\n저장한 파일:'); [print('  ', f) for f in files]
    print('\n예측 유형 수(날별):')
    print(pd.crosstab(table['date'], table['예측']).reindex(columns=CLASS_NAMES8, fill_value=0).to_string())
    if tlist is not None:
        print('\n[알림] edge_id 목록 모드: 값은 유효하지만 SALT-GNN C · TC B 결과가 같은 묶음(128행)에 든 다른 행에 따라 달라져 '
              '원래 val 결과와 그대로 비교할 수 없습니다.')
    if (y >= 0).any():
        sc = _scores(mean, final, y, date); R['scores'] = sc
        print('\n점수(macro 평균 · 8유형 · 정답 = Patterns.txt):'); print(sc.to_string(index=False))
        val_all = all(str(d) in R['days'] for d in pd.date_range(VAL_FIRST, VAL_LAST).strftime('%Y-%m-%d'))
        if val_all and tlist is None:
            b = sc.iloc[-1]
            print(f'\nval 17일 전부를 돌렸습니다 → 최종 뒤 9일 정밀도 {b["val 뒤9일 정밀도"]:.4f} · 재현율 {b["val 뒤9일 재현율"]:.4f} '
                  f'(원래 {REF_BACK9["precision"]:.4f} · {REF_BACK9["recall"]:.4f}). 0.001 안이면 원래와 같은 결과입니다.')
    print('\n단계별 시간 · 메모리:'); print(stages.to_string(index=False))
    rec = dict(만든때=time.strftime('%F %T'), 날=R['days'], 대상=R['mode'], 행=int(len(table)), 장치=dev, 스레드=int(threads), 초=R['sec'],
               꼭대기=peak, 파일=files, 원장=str(L.path), 원장_meta_rows=L.meta['rows'], 단계=meter.rows,
               점수=(R['scores'].to_dict('records') if 'scores' in R else None),
               버전=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__))
    (scratch / f'실행기록_{time.strftime("%Y%m%d_%H%M%S")}.json').write_text(json.dumps(rec, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
    return R

In [ ]:
# 실행 — 위 설정으로 ① 원장을 짓고(처음 한 번 · 다음부터는 다시 씀) ② 날마다 4모델 입력을 만들고 ③ 최종 후보로 분류해 csv 로 씁니다
결과 = 실행(TRANS_CSV, ACCOUNTS_CSV, PATTERNS_TXT, TARGET_DAYS, TARGETS, SCRATCH, OUT_DIR, DEVICE, THREADS)
결과['표'].head(20)

### 참고

**숨긴 셀**
- 상수.
- 학습된 LightGBM 나무와 GNN · TGT 가중치. 묶음 `aml8_모델.pt` 에서 옮겨 압축했습니다.
- 코드 ①~⑦: 원장 · 거래 17열 · 계좌 19열 · SALT 묶음 · TGT 시퀀스 · SALT/TC 모델(순수 torch) · TGT 모델.
- 코드 ⑧ 도구 · ⑨ 실행 순서.
- 코드마다 옮겨 온 원래 파일과 줄 번호(file:line)를 적었습니다. 경로는 팀 작업 공간의 `Final_preprocess/` 부터입니다.
- 코드 주석에 나오는 `test_*.py` · `t*_*.py` · `LEDGER_FORMAT.md` · `logs/…` 는 만들 때 쓴 시험 스크립트와 기록이라 이 노트북에는 들어 있지 않습니다. 시험 결과는 아래 표에 적었습니다.
- 숨긴 셀은 고치지 마세요. 실행할 때 숨긴 자료와 코드를 만들 때 잰 sha256 과 비교하고, 다르면 멈춥니다. 일부러 코드를 고친 경우에만 설정 셀에 `ALLOW_MODIFIED_CODE = True` 를 넣으면 고친 코드로 돕니다(결과가 원래와 달라질 수 있음).

**원래 결과와 같은지 확인한 것**(2026-10-17 · 팀 서버 · CPU 2스레드)

| 시험 | 결과 |
|---|---|
| 기본 실행(`2022-10-06` · 1,415행) · 첫 실행 | Jupyter 커널에서 Run All: 빈 `SCRATCH` 에서 원장을 새로 짓고 끝까지 6분(원장 316초 · 입력 28초 · 모델 14초). 원장을 다시 쓰는 다음 실행은 40초(입력 20초 · 모델 17초). 두 실행의 csv 가 바이트 단위로 같음 |
| val 3일(09-28 · 10-06 · 10-14 · 4,234행) | 모델 10개(LightGBM 1 · GNN 3 × 시드 3)의 확률과 최종이, 묶음 캐시를 읽는 원래 코드(aml8.py · 같은 날 · 같은 배치 · CPU)와 비트 단위로 같음(차 0). 원래 GPU 실행 기대값과의 최대 차는 LightGBM 0 · SALT C 9.6e-6 · TC B 1.6e-6 · TGT B0 2.6e-3 · 최종 4.1e-4. 최종 1등 유형이 다른 행 0 |
| 원장 | 원본 179,702,229행 중 2022-10-14 뒤 39,685,254행은 시각 칸만 해석하고 버림 · 완전 중복 122행 지움 · 140,016,853행. 다시 지은 원장의 배열 24개가 고치기 전 코드로 지은 원장과 바이트 단위로 같음 |
| val 17일 전부(`TARGET_DAYS = 'val'` · 24,280행 · 원장 다시 씀 · 10분) | 4모델 입력(55열 · SALT 8배열 · TGT 15키)이 묶음 캐시와 17일 모두 비트 단위로 같음. 확률은 `aml8.py predict`(CPU)와 비트 단위로 같음. 최종 뒤 9일 macro 정밀도 **0.5089** · 재현율 **0.5026** = 원래와 같음. 모델별 점수도 묶음 README 7절 CPU 표와 같음 |
| 원래 GPU 실행과의 차(17일) | TGT B0 최대 2.7e-3(1등 다른 행 시드 42 · 43 · 44 별로 12 · 22 · 11 / 24,280) · 최종 5.0e-4(1행). 묶음 aml8.py 를 CPU 로 돌린 것과 같은 크기 |
| 거부 · 멈춤 | 2022-10-15 · 2022-11-30 · `'2022-10-10~2022-10-20'` 은 원장을 짓기 전에 거부. 앞 100만 줄만 남긴 거래 파일은 크기 확인에서 멈춤. 빈 공간이 모자라면 원장을 짓기 전에 멈춤. 숨긴 자료 · 코드 셀을 바꾸면 멈춤. torch · numpy 가 없으면 무엇을 깔지 알리고 폴더를 만들기 전에 멈춤 |
| 필요 없는 라이브러리 | lightgbm · torch_geometric · DuckDB · scikit-learn 의 import 를 막고 돌려도 같은 결과. pyarrow 를 막아도 돎(원장을 다시 쓰는 실행의 csv 가 바이트 단위로 같음) |
| edge_id 목록(10-06 · 패턴 행 100 + 아닌 행 30) | 돎. 패턴 행의 55열 · LightGBM 확률은 하루 전체로 돌린 것과 같음(차 0), TGT B0 는 1.2e-7. SALT C · TC B 는 묶음이 달라져 최대 0.31 다름(설명대로) |

- 3일만 돌렸을 때 TGT B0 시드 43 의 1등 같음 비율은 0.99858(6 / 4,234행 다름)입니다. CPU fp32 와 원래 GPU fp16 의 차이이며, 기준 0.999 보다 조금 낮습니다. 묶음 aml8.py 를 CPU 로 돌려도 같은 값입니다. 17일 전부에서는 0.99909 로 기준 안입니다.

**한계**
- 이 모델은 정답 패턴 행이 주어져야 유형을 고릅니다. Patterns.txt 없이 원본만으로 세탁 거래를 찾지는 못합니다.
- 원본 Trans.csv 전체가 행 순서 그대로 있어야 합니다. 거래 이력 피처는 2022-08-01 부터 누적되고, CSV 가 시간순이 아니어서 원장을 지을 때는 어느 날을 고르든 파일을 끝까지 훑습니다.
- CPU 에서는 TGT B0 가 fp32 로 돕니다. 원래 GPU fp16 결과와 확률이 1e-3 정도 다르고, 1등 유형이 다른 행이 드물게 있습니다(위 표).
- SALT-GNN C · TransformerConv B(128행 묶음 4개씩) · TGT B0(2,048행씩)은 원래처럼 여러 날을 이어 붙여 한 번에 돌립니다. 그래서 `TARGET_DAYS` 의 날 목록이 다르면 같은 날의 확률도 아주 조금 달라집니다(시험: val 3일만 넣은 실행과 17일 전부 넣은 실행의 같은 날 4,234행에서 최대 차 SALT C 3.3e-6 · TC B 2.4e-7 · TGT B0 2.4e-7 · 최종 5.6e-7, 1등 유형이 다른 행 0).

팀 채택 전 후보(2026-10-16 묶음)입니다. 만든 날: 2026-10-17 · Claude Code.